# FICBO evaluation

Runs every paper experiment: trained FICBO models vs. baselines on tasks with biased auxiliary feedback.

Run top to bottom. Smoke sizes are marked `# TEMP smoke, real: ...`; set the real values for paper numbers.
Checkpoint paths are marked `# TEMP path`; point them to your own Hydra runs (`.hydra/config.yaml` + checkpoint).

| Section | Paper |
|---|---|
| Main results: multiple benchmarks | main figures, appendix tables and classical-GP baselines |
| Ablation | ablation table (Airfoil Small / Lift) |
| Policy-feedback agreement | agreement plot |
| UMAP of feedback-error profiles | UMAP plot |
| Continuous BO | continuous BO appendix |
| Single-benchmark check, multi-seed evaluation | not in the paper, quick checks |

Method keys in `results[bench]`, as named in the paper:

| Key | Paper name |
|---|---|
| `ficbo_mixture`, `ficbo_additive`, `ficbo_combined`, `ficbo_ensemble` | FICBO (Mixture / Additive / Combined / Ensemble) |
| `gp_icbo`, `gp_icbo_fb` | GP-prior ICBO, GP-prior ICBO + feedback feature |
| `pibo_ucb`, `pibo_ei` | πBO-UCB, πBO-EI (GP-prior ICBO surrogate) |
| `pibo_mixture_ucb`, `pibo_mixture_ei` | FICBO-UCB, FICBO-EI (semi-amortized) |
| `gp_pibo_ucb`, `gp_pibo_ei` | πBO-UCB GP, πBO-EI GP |
| `gp_ucb`, `gp_feat_ucb` | GP-UCB, GP-UCB + feedback feature |
| `mtgp_icm_ucb`, `rgpe_ucb`, `imfbo_*` | MTGP-UCB (ICM), RGPE-UCB, iMFBO variants |
| `feedback`, `random` | Feedback, Random |

The only feedback adjustor is `pi_bo`: $\alpha(x)\,\tilde u(x)^{\beta/(t+1)}$ with $\tilde u = \mathrm{softmax}(u)$ and $\beta = 2$.


In [ ]:
save_plots=True

In [ ]:
#pip install nas-bench-201
# download NAS-Bench-201-v1_0-e61699.pth from https://github.com/D-X-Y/NAS-Bench-201 into data/

# Setup


In [ ]:
import sys
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
here = Path.cwd()
parent = here.parent

grandparent = parent.parent

sys.path.append(str(here))
sys.path.append(str(parent))


In [ ]:
import hydra
from hydra import initialize, compose
import itertools
import torch
import urllib.request
import tarfile
import gpytorch
from copy import deepcopy
from torch.distributions import Normal
from botorch.models.model import Model
from gpytorch.models import ExactGP
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.kernels import ScaleKernel, RBFKernel, MaternKernel
from gpytorch.distributions import MultivariateNormal
from gpytorch.mlls import ExactMarginalLogLikelihood
from tqdm import tqdm
from dataclasses import dataclass, field
import torch.nn as nn

from omegaconf import open_dict
import warnings
import time
import copy
import pickle

from IPython.display import display

from typing import List, Tuple, Dict, Optional

from dataclasses import dataclass, field, replace
from typing import Optional, Callable, Any, Dict, List
from functools import partial
from attrdictionary import AttrDict

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from matplotlib import cm
import matplotlib.colors as mcolors

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.metrics import mean_squared_error

from scipy.stats import norm, gaussian_kde

from torch.distributions import Normal#, Categorical
import torch.nn.functional as F

from botorch.posteriors.gpytorch import GPyTorchPosterior
from botorch.acquisition.analytic import (
    LogExpectedImprovement, ExpectedImprovement,
    UpperConfidenceBound, ProbabilityOfImprovement,
)
from linear_operator.operators import DiagLinearOperator
from botorch.posteriors import Posterior
from botorch.acquisition.monte_carlo import qExpectedImprovement, qUpperConfidenceBound

from model import FICBO
from expert_models import *

from utils.misc import *
from utils import (
    compute_ll,
    compute_rmse,
    load_checkpoint,
    create_target_mask,
    select_targets_by_mask,
)
import gc

from linear_operator.utils.errors import NotPSDError
import gpytorch


from tasks import *

warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

In [ ]:

if torch.cuda.is_available():
    print("CUDA is available!")
    device="cuda"
    print("Device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is NOT available.")
    device = "cpu"


In [ ]:
batch_size = 4  #TEMP smoke, real: 100
T = 5  # TEMP smoke, real: 30
embedding_mode_all = "mix"  #"data"
shuffled_text_batch = False  #take the same data, but shuffle the initial context point

In [ ]:
def seed_everything(seed: int = 42):
    import os
    import random
    import numpy as np

    # python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # environment (affects hash randomization, etc.)
    os.environ["PYTHONHASHSEED"] = str(seed)

    import torch

    # PyTorch
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # deterministic behavior (may impact performance)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


    print(f"Seed set to {seed}")


seed_everything(0)

In [ ]:
def _batch_to_device(batch, device):
    for k, v in batch.items():
        if torch.is_tensor(v):
            batch[k] = v.to(device)
    return batch

In [ ]:
def load_config_and_model(path, config_name="config.yaml", file_name="aae.pth"):
    """
    Loads configuration and model from a specified path. Instantiates the model components
    (embedder, encoder, and head) based on the configuration, and loads the model's checkpoint.
    """
    config_path = path + ".hydra/"
    print(config_path)
    with initialize(version_base=None, config_path=config_path):
        cfg = compose(config_name=config_name)

        embedder = hydra.utils.instantiate(cfg.embedder)
        encoder = hydra.utils.instantiate(cfg.encoder)
        head = hydra.utils.instantiate(cfg.head)
        #model = BaseTransformer(embedder, encoder, head)
        model = FICBO(embedder, encoder, head)

        # file_path = os.path.join(path, 'model', file_name)
        # model.load_state_dict(torch.load(file_path, map_location=torch.get_default_device(), weights_only=True))

        ckpt = torch.load(os.path.join(path, file_name), map_location=torch.get_default_device(), weights_only=False)

        try:
            model.load_state_dict(ckpt["model"])
            print(f"Loaded checkpoint using ckpt['model'] from {file_name}")
        except KeyError:
            model.load_state_dict(ckpt)
            print(f"Loaded raw state dict from {file_name}")

    return cfg, model

In [ ]:
@dataclass
class MethodConfig:
    """
    Bundles all per-method arguments for evaluate_policy_vs_bo.
 
    Parameters
    ----------
    name : str
        Display name used as key in the returned data dict.
    model : nn.Module
        The policy / surrogate model.
    cfg : DictConfig
        Hydra config for this model.
    acquisition : str
        One of "aae", "ucb", "ei", "random", "feedback"
    attend_to : str
    n_theta : int | None
        Falls back to cfg.task.n_target_theta when None.
    concat_feedback : bool
        Prepend expert feedback to x before passing to model.
    experiment_dim_x_delta : int
        Temporary offset applied to experiment.dim_x around the eval call.
        Set to +cfg.task.dim_feedback for concat_feedback methods.
    use_gp : bool
        If True, calls evaluate_gp_bo instead of evaluate_policy_vs_bo.
    gp_native : bool
        If True (independent of use_gp), calls evaluate_gp_bo_botorch instead: the
        BoTorch-native pool-argmax GP baseline (real SingleTaskGP + fit_gpytorch_mll +
        botorch acquisition-function math), scored over the full candidate pool every
        step, as opposed to evaluate_gp_bo's hand-rolled BatchedGP/gpytorch model.
    feedback_integration : str
        botorch_continuous only (see evaluate_botorch_gp_continuous). "gradient" (default)
        bakes expert_feedback_adjustor into the objective optimize_acqf actually searches,
        every L-BFGS iteration. "posthoc_topk" is the legacy behavior: optimize_acqf
        searches the raw, feedback-blind acquisition surface, and the adjustor only gets a
        post-hoc pick among the num_restarts local optima it lands on, kept runnable for
        comparison against "gradient".
    gp_model : sklearn GP | None
    expert_feedback_adjustor : Callable | None
    advanced_stats : bool
        Collect per-step prediction tensors (slow; usually only for "ficbo_combined").
    n_iter_gp : int
        Number of GP optimisation iterations (only used when use_gp=True).
    continuous : bool
        If True, gradient-refine the candidate pool (free/continuous mode, see
        utils/free_mode.py) instead of picking by argmax over the fixed pool.
    feedback_mode : str
        "frozen" (observe feedback once, hold fixed) or "live" (re-observe the
        real feedback at the current x once per refine iteration). Continuous only.
    use_multitask_gp : bool
        If True, calls evaluate_multitask_gp_bo instead of evaluate_policy_vs_bo:
        the expert feedback becomes a second GP task rather than an extra input feature or a post-hoc acquisition reweighting.
    mtgp_kind : str
        "icm" (a from-scratch batched ICM model matching botorch MultiTaskGP's
        construction, feedback observed pool-wide) or "kronecker" (the *same* ICM
        model, just restricted to block-aligned context-only data, despite the
        name it does NOT use botorch's KroneckerMultiTaskGP class; see evaluate_multitask_gp_bo's docstring for details).
    use_rgpe : bool
        If True, calls evaluate_rgpe_bo instead of evaluate_policy_vs_bo: expert
        feedback becomes RGPE's "base task" (rank-weighted GP ensemble, see
        utils/rgpe.py) instead of a second ICM task, an input feature, or an
        acquisition reweighting.
    rgpe_max_pool, rgpe_num_posterior_samples, rgpe_target_noise_var,
    rgpe_feedback_noise_var : optional
        Passed through to evaluate_rgpe_bo when set; None falls back to that function's own defaults.
    use_imfbo : bool
        If True, calls evaluate_imfbo_bo instead of evaluate_policy_vs_bo: the
        expert feedback becomes a set of *direct noisy observations of the
        objective itself*, with a learned input-dependent noise sigma_lf(x)
        deciding how far to trust it at each x (iMFBO, Fan et al. UAI 2024 --
        see utils/imfbo_gp.py). Distinct from mtgp_* (feedback as a second GP
        task coupled by a scalar task correlation and a constant per-task
        noise) and from gp_feat_* (feedback as an extra input column).
    imfbo_noise_model : str
        "parametric" (log sigma_lf(x) is a small per-environment MLP, fit by
        MAP) or "gp" (a free per-row log-noise vector under a GP prior, the
        paper's eq.-12 MAP estimate). 
    imfbo_max_pool : int | None
        Pool points carried as feedback rows each step; None falls back to
        evaluate_imfbo_bo's own default. This is the cost knob and the batched
        Cholesky is cubic in it.
    reference_optimum : torch.Tensor | None
        Shared regret baseline (e.g. the pool's true_max) so a continuous run's
        regret is comparable to its pool-based counterpart. Continuous only.
    pool_size : int | None
        Fresh candidate pool sampled each continuous BO step (default 2000 when
        None). Continuous only, see utils/free_mode.continuous_bo_step.
    n_refine_candidates : int | None
        Top-K of pool_size (by acquisition score) that get L-BFGS gradient-refined
        each step (default 100 when None). Continuous only.
    score_chunk_size : int | None
        Chunk size for the no-grad acquisition scoring pass over pool_size
        (default 200 when None -> matches n_query_init, the pool size the model
        was trained on; attend_to_queries=True means scores arent chunk-invariant).
        Continuous only.
    n_ballast_candidates : int | None
        Extra Sobol space-filling points carried through refinement UNoptimized,
        alongside the refined top-K (default 0 when None = off). They keep the query
        block from collapsing to nothing but high-acquisition points, which matters
        because candidates self-attend and the head softmaxes over the set; aim for
        n_refine_candidates + n_ballast_candidates near n_query_init. They are fully
        observed, so they can still win a step. Continuous only.
    candidate_mode : str | None
        "pool" (default when None; original behaviour: rank a fresh pool_size draw and
        refine the top-K) or "sobol": skip the pool/ranking pass and refine n_candidates
        fresh Sobol points directly, with n_background carried through unoptimized in the
        n_ballast_candidates role. Continuous only, see utils/free_mode.continuous_bo_step.
    n_candidates, n_background : int | None
        Sizes of the editable/fixed Sobol blocks when candidate_mode="sobol" (each default
        100 when None). Ignored in "pool" mode. Continuous only.
    refine_optimizer : str | None
        "lbfgs" (default when None) or "adam" for refine_pool's per-step optimizer. Adam
        runs adam_step_multiplier x n_refine_steps iterations to stay budget-matched in
        forward passes against L-BFGS's multi-evaluation line search. Continuous only.
    refine_lr : float | None
        Step size for refine_pool's optimizer, applied in normalized [-1, 1] box coordinates
        (default 0.1 when None). Continuous only.
    adam_step_multiplier : int | None
        See refine_optimizer above (default 4 when None). Continuous only.
    log_ei_mode : str | None
        "clamp" (default when None; log(EI).clamp_min(1e-30), can zero-gradient once EI
        underflows) or "botorch" (botorch's LogExpectedImprovement formula, finite gradient
        much further out). Only affects acquisition="log_ei". Continuous only.
    """
    name: str
    model: Any
    cfg: Any
    acquisition: str = "aae"
    attend_to: str = "data"
    n_theta: Optional[int] = None
    concat_feedback: bool = False
    experiment_dim_x_delta: int = 0
    use_gp: bool = False
    gp_native: bool = False
    feedback_integration: str = "gradient"
    gp_model: Any = None
    expert_feedback_adjustor: Optional[Callable] = None
    advanced_stats: bool = False
    n_iter_gp: int = 100
    continuous: bool = False
    feedback_mode: str = "frozen"
    reference_optimum: Any = None
    pool_size: Optional[int] = None
    n_refine_candidates: Optional[int] = None
    score_chunk_size: Optional[int] = None
    n_ballast_candidates: Optional[int] = None
    candidate_mode: Optional[str] = None
    n_candidates: Optional[int] = None
    n_background: Optional[int] = None
    refine_optimizer: Optional[str] = None
    refine_lr: Optional[float] = None
    adam_step_multiplier: Optional[int] = None
    log_ei_mode: Optional[str] = None
    kappa: float = 1.0
    use_multitask_gp: bool = False
    mtgp_kind: str = "icm"
    use_rgpe: bool = False
    rgpe_max_pool: Optional[int] = None
    rgpe_num_posterior_samples: Optional[int] = None
    rgpe_target_noise_var: Optional[float] = None
    rgpe_feedback_noise_var: Optional[float] = None
    use_imfbo: bool = False
    imfbo_noise_model: str = "parametric"
    imfbo_max_pool: Optional[int] = None
 
    def effective_n_theta(self) -> int:
        return self.n_theta if self.n_theta is not None else self.cfg.task.n_target_theta
 
    def vary(self, **overrides) -> "MethodConfig":
        """
        Return a shallow copy with selected fields overridden.
 
            base = MethodConfig("_", model, cfg, n_theta=3)
            ucb  = base.vary(name="ucb", acquisition="ucb")
            ei   = base.vary(name="ei",  acquisition="ei")
        """
        mc = copy.copy(self)
        for k, v in overrides.items():
            setattr(mc, k, v)
        return mc
 
 
def baseline_configs(
    template: MethodConfig,
    acquisitions: List[str] = ("random", "ucb", "ei"),
    name_prefix: str = "",
    ) -> List[MethodConfig]:
    """
    Build a list of MethodConfigs for classical BO baselines by varying only
    `acquisition` (and `name`) on a template MethodConfig you already defined.
 
    Every other field: model, cfg, n_theta, use_gp, gp_model,
    expert_feedback_adjustor, concat_feedback, advanced_stats, etc. is inherited from the template unchanged.
 
    Example
    -------
    baseline_template = MethodConfig(
        name="_",
        model=expert_model_not_using_feedback,
        cfg=cfg,
        n_theta=cfg_no_fb.task.n_target_theta,
        attend_to="data",
        use_gp=True,
        gp_model=my_gp,
        expert_feedback_adjustor=partial(pi_bo, beta=2),
    )
    method_configs = [
        MethodConfig("ficbo_combined",  model, cfg, acquisition="aae",
                     n_theta=cfg.task.n_target_theta),
        *baseline_configs(baseline_template, acquisitions=("random", "ucb", "ei", "feedback")),
    ]
    """
    return [
        template.vary(name=f"{name_prefix}{acq}", acquisition=acq)
        for acq in acquisitions
    ]


In [ ]:
def get_normalized_best_so_far(results: dict, optima: dict) -> dict:
    """
    Returns dict[method] -> tensor [n_benchmarks, T+1, batch_size], normalized to [0, 1].
    0 = no improvement over t=0, 1 = optimum found.
    """
    benchmark_names = list(results.keys())
    metrics_data_list = aggregate_results(results, metric="best_so_far")
    y_opt = torch.stack([optima[name] for name in benchmark_names], dim=0)  # [n_benchmarks, batch_size]

    normalized = {}
    for method, data in metrics_data_list.items():
        y_init = data[:, 0, :].unsqueeze(1)  # [n_benchmarks, 1, batch_size]
        y_opt_exp = y_opt.unsqueeze(1)  # [n_benchmarks, 1, batch_size]
        normalized[method] = (data - y_init) / (y_opt_exp - y_init + 1e-8)

    return normalized


def get_normalized_cum_regret(results: dict, optima: dict) -> dict:
    norm_bsf = get_normalized_best_so_far(results, optima)
    normalized = {}
    for method, data in norm_bsf.items():
        simple_regret = 1 - data[:, 1:, :]  # [n_benchmarks, T, batch_size]
        normalized[method] = simple_regret.cumsum(dim=1)
    return normalized


In [ ]:
class AirfoilTaskWrapper:
    def __init__(
        self, benchmark_name, n_target_data, n_target_theta, noise_scale=0.1, normalize_y=True,
        **kwargs, 
        ):
        task_map = {
            "airfoil_lift_drag": AirfoilLiftDragTask,
            "airfoil_nn": AirfoilNNFidelityTask,
        }
        print(f"{n_target_theta=}")
        cls = task_map[benchmark_name]
        self._task = cls(
            n_context_init=1,
            n_query_init=300,
            n_target_data=n_target_data,
            n_target_theta=n_target_theta,
            noise_scale=noise_scale,
            normalize_y=normalize_y,
            embedding_type="mix",
        )

    def sample_batch(self, batch_size, **kwargs):  
        return self._task.sample_batch(batch_size, mode="mix")

    def __getattr__(self, name): 
        if name == "_task":
            raise AttributeError(name)
        return getattr(self._task, name)


class NASBench201TaskWrapper:
    def __init__(
        self,
        benchmark_name,  # ignored, only have one task
        n_target_data,
        n_target_theta,
        dim_x,
        dim_y,
        noise_scale=0.01,
        normalize_y=True,
        pth_path="data/NAS-Bench-201-v1_0-e61699.pth",
        **kwargs,                 
        ):
        print(f"{n_target_theta=}")
        device = kwargs.get("device", None)

        self._task = NASBench201Task.from_pth(
            pth_path=pth_path,
            n_context_init=1,
            n_query_init=123,
            n_target_data=n_target_data,
            n_target_theta=n_target_theta,
            design_scale=5,
            noise_scale=noise_scale,
            normalize_y=normalize_y,
            device=device,
        )

    def sample_batch(self, batch_size, **kwargs):  
        return self._task.sample_batch(batch_size, mode="mix")

    def __getattr__(self, name):  # update_batch etc. from wrapped task
        if name == "_task":
            raise AttributeError(name)
        return getattr(self._task, name)



In [ ]:
def create_context_shuffled_batch(batch, batch_size):
    b = deepcopy(batch)

    qry_x = b.query_x[0]  # (n_query, x_dim)
    qry_y = b.query_y[0]  # (n_query, 1)
    qry_feedback = b.query_feedback[0]  # (n_query, f_dim)

    has_y_hidden_qry = hasattr(b, 'y_hidden_query') and b.y_hidden_query is not None
    y_hidden_qry = b.y_hidden_query[0] if has_y_hidden_qry else None

    n_query = qry_x.shape[0]
    assert batch_size <= n_query - 1, \
        f"batch_size ({batch_size}) cannot exceed n_query - 1 ({n_query - 1})"

    chosen_indices = torch.randperm(n_query, device=qry_x.device)[:batch_size]

    new_ctx_x, new_ctx_y, new_ctx_feedback = [], [], []
    new_qry_x, new_qry_y, new_qry_feedback = [], [], []
    new_y_hidden_qry_list = []

    for i in chosen_indices:
        mask = torch.ones(n_query, dtype=torch.bool, device=qry_x.device)
        mask[i] = False

        new_ctx_x.append(qry_x[i:i+1].unsqueeze(0))
        new_ctx_y.append(qry_y[i:i+1].unsqueeze(0))
        new_ctx_feedback.append(qry_feedback[i:i+1].unsqueeze(0))

        new_qry_x.append(qry_x[mask].unsqueeze(0))
        new_qry_y.append(qry_y[mask].unsqueeze(0))
        new_qry_feedback.append(qry_feedback[mask].unsqueeze(0))

        if has_y_hidden_qry:
            new_y_hidden_qry_list.append(y_hidden_qry[mask].unsqueeze(0))

    new_batch = deepcopy(b)

    new_batch.context_x = torch.cat(new_ctx_x, dim=0)
    new_batch.context_y = torch.cat(new_ctx_y, dim=0)
    new_batch.context_feedback = torch.cat(new_ctx_feedback, dim=0)
    new_batch.query_x = torch.cat(new_qry_x, dim=0)
    new_batch.query_y = torch.cat(new_qry_y, dim=0)
    new_batch.query_feedback = torch.cat(new_qry_feedback, dim=0)

    if has_y_hidden_qry:
        new_batch.y_hidden_query = torch.cat(new_y_hidden_qry_list, dim=0)

    for field in ['target_x', 'target_y', 'target_all', 'target_feedback',
                  'expert_x', 'expert_y', 'y_hidden_target', 'y_hidden_context']:
        val = getattr(new_batch, field, None)
        if val is not None and torch.is_tensor(val):
            setattr(new_batch, field,
                    val[0].unsqueeze(0).expand(batch_size, *val.shape[1:]).clone())

    return new_batch

In [ ]:
def concat_feedback_to_batch_x(batch):
    """
    Concatenate expert feedback directly into batch x tensors in-place.

    After this:
        context_x <- [context_x, context_feedback]
        query_x   <- [query_x,   query_feedback]
        target_x  <- [target_x,  target_feedback]

    Assumes feedback tensors are shaped (B, N_split, D_feedback).
    """

    if hasattr(batch, "context_feedback") and batch.context_feedback is not None:
        batch.context_x = torch.cat([batch.context_x, batch.context_feedback], dim=-1)

    if hasattr(batch, "query_feedback") and batch.query_feedback is not None:
        batch.query_x = torch.cat([batch.query_x, batch.query_feedback], dim=-1)

    if hasattr(batch, "target_feedback") and batch.target_feedback is not None:
        batch.target_x = torch.cat([batch.target_x, batch.target_feedback], dim=-1)
    batch.dim_x += batch.context_feedback.shape[-1]
    return batch


In [ ]:
def print_batch_stats(batch, name="batch"):
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")

    def stats(label, t):
        if t is None or t.numel() == 0:
            print(f"  {label:30s} : empty/None")
            return
        qs = torch.quantile(t.float().flatten(), torch.tensor([0.25, 0.5, 0.75, 0.95], device=t.device))

        print(f"  {label:30s} : shape={str(tuple(t.shape)):20s}  "
              f"mean={t.mean().item():7.3f}  std={t.std().item():7.3f}  "
              f"mode={t.flatten().mode().values.item():7.3f} \n"
              f"\t\t\t\t min={t.min().item():7.3f}  q25={qs[0].item():7.3f}  q50={qs[1].item():7.3f}  "
              f"q75={qs[2].item():7.3f}  q95={qs[3].item():7.3f}  max={t.max().item():7.3f}")

    stats("query_y", batch.query_y)
    stats("context_y", batch.context_y)
    stats("target_y", batch.target_y)

    print()
    query_maxes = batch.query_y.squeeze(-1).max(dim=1).values
    print(f"  {'query_y max (per batch)':30s} : "
          f"mean={query_maxes.mean().item():7.3f}  std={query_maxes.std().item():7.3f}  "
          f"min={query_maxes.min().item():7.3f}  max={query_maxes.max().item():7.3f}")
    query_means = batch.query_y.squeeze(-1).mean(dim=1)
    print(f"  {'query_y mean (per batch)':30s} : "
          f"mean={query_means.mean().item():7.3f}  std={query_means.std().item():7.3f}  "
          f"min={query_means.min().item():7.3f}  max={query_means.max().item():7.3f}")

    print()
    if getattr(batch, 'y_hidden_query', None) is not None:
        stats("y_hidden_query", batch.y_hidden_query)
        stats("y_hidden_context", batch.y_hidden_context)
        stats("y_hidden_target", batch.y_hidden_target)

    print()
    fb = getattr(batch, 'query_feedback', None)
    if fb is not None:
        stats("query_feedback", batch.query_feedback)
        stats("context_feedback", batch.context_feedback)
        stats("target_feedback", batch.target_feedback)

        # rank align, query_y vs feedback
        # [B, N, 1] or [B, N, d_fb], rank over dim=1
        y = batch.query_y.squeeze(-1)  # [B, N]
        f = fb[..., 0]  # [B, N] first feedback dim
        N = y.shape[1]

        # argsort desc -> ranks[b, i] = rank of point i (0 = best)
        y_ranks = torch.argsort(torch.argsort(y, dim=1, descending=True), dim=1)  # [B, N]
        f_ranks = torch.argsort(torch.argsort(f, dim=1, descending=True), dim=1)  # [B, N]

        # 1. rank of the true-y-best point in feedback ordering
        #    (0 = feedback agrees, N-1 = ranks it last)
        true_best_idx = y.argmax(dim=1)  # [B]
        bi = torch.arange(y.shape[0], device=y.device)
        true_best_fb_rank = f_ranks[bi, true_best_idx].float()  # [B]

        # 2. rank of the feedback-best point in true-y ordering
        fb_best_idx = f.argmax(dim=1)  # [B]
        fb_best_y_rank = y_ranks[bi, fb_best_idx].float()  # [B]

        # 3. Spearman rho per batch element, then avg
        #    rho = 1 - 6*sum(d^2) / (N*(N^2-1))
        d_sq = (y_ranks.float() - f_ranks.float()).pow(2).sum(dim=1).float()
        spearman = 1.0 - 6.0 * d_sq / (N * (N**2 - 1) + 1e-8)  # [B]

        print()
        print(f"  rank alignment (query, N={N}) {'-'*18}")

        def rstat(label, vals):
            # vals: [B] tensor, report as percentile of N (0%=best, 100%=worst)
            pct = vals / (N - 1) * 100
            print(f"  {label:40s} : "
                  f"mean={vals.mean().item():5.1f}  "
                  f"({pct.mean().item():4.1f}%)  "
                  f"std={vals.std().item():4.1f}  "
                  f"min={vals.min().item():4.0f}  max={vals.max().item():4.0f}")

        rstat("true-y-best rank in feedback  (0=perfect)", true_best_fb_rank)
        rstat("feedback-best rank in true y  (0=perfect)", fb_best_y_rank)
        print(f"  {'Spearman rho (y vs feedback)':40s} : "
              f"mean={spearman.mean().item():6.3f}  "
              f"std={spearman.std().item():6.3f}  "
              f"min={spearman.min().item():6.3f}  max={spearman.max().item():6.3f}")

    print()
    stats("target_all", batch.target_all)
    if getattr(batch, 'target_theta', None) is not None:
        stats("target_theta", batch.target_theta)
    print(f"\n  n_target_theta: {getattr(batch, 'n_target_theta', 'N/A')}")
    print(f"  expert_dim_x:   {getattr(batch, 'expert_dim_x', 'N/A')}")
    print(f"  dim_x:          {getattr(batch, 'dim_x', 'N/A')}")
    print(f"{'='*50}\n")

# Plot helpers


In [ ]:
matplotlib.rcParams.update({
        'font.family': "serif",#'times',
        'font.size': 14.0,
        'lines.linewidth': 3,
        'lines.antialiased': True,
        'axes.facecolor': 'ffffff',
        'axes.edgecolor': '777777',
        'axes.linewidth': 1,
        'axes.titlesize': 'medium',
        'axes.labelsize': 'medium',
        'axes.axisbelow': True,
        'xtick.major.size': 0,  # major tick size in points
        'xtick.minor.size': 0,  # minor tick size in points
        'xtick.major.pad': 6,  # distance to major tick label in points
        'xtick.minor.pad': 6,  # distance to the minor tick label in points
        'xtick.color': '333333',  # color of the tick labels
        'xtick.labelsize': 'medium',  # fontsize of the tick labels
        'xtick.direction': 'in',  # direction: in or out
        'ytick.major.size': 0,  # major tick size in points
        'ytick.minor.size': 0,  # minor tick size in points
        'ytick.major.pad': 6,  # distance to major tick label in points
        'ytick.minor.pad': 6,  # distance to the minor tick label in points
        'ytick.color': '333333',  # color of the tick labels
        'ytick.labelsize': 'medium',  # fontsize of the tick labels
        'ytick.direction': 'in',  # direction: in or out
        'axes.grid': False,
        'grid.alpha': 0.3,
        'grid.linewidth': 1,
        'legend.fancybox': True,
        'legend.fontsize': 'medium',
        'figure.figsize': (2.50, 2.5),
        'figure.facecolor': '1.0',
        'figure.edgecolor': '0.5',
        'hatch.linewidth': 0.1,
        'text.usetex': False,  # true
        'axes.linewidth': 3,  # thicker borders
        'axes.spines.top': False,  # remove top border
        'axes.spines.right': False,  # remove right border

    })


In [ ]:
def plot_metrics_with_confidence(
    metrics_data_list, 
    metric_names, 
    x_range=None, 
    colors=None, 
    markers=None, 
    title="", 
    xlabel="Number of Steps", 
    ylabel="RMSE", 
    figsize=(5, 4), 
    marker_frequency=5, 
    legend_loc='upper right',
    show_optimal=False,
    optimal_value=None,
    optimal_label="Optimal",
    use_markers=False,
    nbins=5,
    include_legend=True,
    top_extra=.3,
    ):
    # ensure all input data is numpy arrays
    metrics_data_list = [
        data.detach().numpy() if torch.is_tensor(data) else data 
        for data in metrics_data_list
    ]
    
    # n time steps
    T = metrics_data_list[0].shape[0]
    
    # set default x_range if not provided
    if x_range is None:
        x_range = (0, T)   # 0 if we take first context into account
        
    # create x-axis array
    x = np.arange(x_range[0], x_range[1] + 1)
    
    # set default colors if not provided
    if colors is None:
        colors = ['#8172b3', '#937860', '#4c72b0', '#dd8452', '#55a868', '#c44e52', 
                     '#8c8c8c', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
                    ]
    
    # set default markers if not provided
    if markers is None:
        markers = ['o', 's', '^', 'D', 'X', 'P',
                  '*', 'v', '<', '>', 'h', 'H', '+', 'x', 'd', '|', '_']
    
    # create figure
    fig, ax = plt.subplots(figsize=figsize, dpi=100)
    
    # helper function to calculate confidence intervals
    def get_confidence_interval(data):
        n_trials = data.shape[1]
        std = data.std(axis=1)
        ci = 1.96 * std / np.sqrt(n_trials)  # 1.96 for 95% CI under normal approx.
        
        return ci
    
    # helper function to plot with confidence bands
    def plot_with_confidence(ax, x, mean, ci, label, color, marker, i):
        ax.plot(x, mean, label=label, color=color, linewidth=4, 
                marker=marker if use_markers else None, markevery=marker_frequency, markersize=7, markeredgecolor='white', zorder=(199-i))
        ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.4 , zorder=(99-i))
    
    # plot each metric
    for i, (data, label) in enumerate(zip(metrics_data_list, metric_names)):
        color = colors[i % len(colors)]
        marker = markers[i % len(markers)]
        
        mean = data.mean(axis=1)
        ci = get_confidence_interval(data)
        
        # ensure data length matches x-axis
        if mean.shape[0] < len(x):
            padded_mean = np.full(len(x), np.nan)
            padded_mean[:mean.shape[0]] = mean
            
            padded_ci = np.full(len(x), np.nan)
            padded_ci[:ci.shape[0]] = ci
            
            plot_with_confidence(ax, x, padded_mean, padded_ci, label, color, marker=marker if use_markers else None, i=i)
        elif mean.shape[0] > len(x):
            plot_with_confidence(ax, x, mean[:len(x)], ci[:len(x)], label, color, marker=marker if use_markers else None, i=i)
        else:
            plot_with_confidence(ax, x, mean, ci, label, color, marker=marker if use_markers else None, i=i)

    # optimal line: batch mean of query-set max
    if show_optimal:
        if optimal_value is None:
            raise ValueError(
                "optimal_value must be provided when show_optimal=True. "
                "Pass the per-batch true maximum (shape: (batch_size,)) or a scalar."
            )
        opt_val = float(np.mean(optimal_value))  # scalar, avg over batch if needed
        print(f"{opt_val=}")
        ax.axhline(opt_val, color='black', linewidth=1.5,
                   linestyle='--', label=optimal_label, zorder=200)
        
        ax.set_ylim(top=opt_val+ top_extra)

    # customize the plot
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, pad=10)
    
    # add grid with lower opacity
    #ax.grid(True, linestyle='-', alpha=0.7)
    
    if include_legend:
        # add legend with a nice border
        legend = ax.legend(frameon=True, framealpha=1, edgecolor='gray', loc=legend_loc)
        legend.set_zorder(300)
        legend.get_frame().set_linewidth(0.5)
    
    # set axis limits
    #ax.set_xlim(x_range[0], x_range[1])
    ax.set_xlim(-1, x_range[1])
    
    # add minor ticks
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=nbins))
    
    # adjust layout
    plt.tight_layout()
    
    return fig, ax

In [ ]:
def plot_ranks_with_confidence(
    metrics_data_list,
    metric_names,
    x_range=None,
    colors=None,
    markers=None,
    title="",
    xlabel="Number of Steps",
    ylabel="Average Rank",
    figsize=(5, 4),
    marker_frequency=5,
    legend_loc='upper right',
    low_is_better=True,
    nbins=5,
    use_markers=False,
    rank_eps=1e-3,
    include_legend=False,
    ):
    def _rankdata_with_tol(values, lower_is_better=True, eps=1e-6):
        v = values if lower_is_better else -values
        n = len(v)
        ranks = np.zeros(n)
        order = np.argsort(v)
        sorted_v = v[order]
        i = 0
        while i < n:
            j = i + 1
            while j < n and (sorted_v[j] - sorted_v[i]) < eps:
                j += 1
            avg_rank = (i + j + 1) / 2.0
            for k in range(i, j):
                ranks[order[k]] = avg_rank
            i = j
        return ranks

    metrics_data_list = [
        data.detach().numpy() if torch.is_tensor(data) else data
        for data in metrics_data_list
    ]

    T = metrics_data_list[0].shape[0]
    n_methods = len(metrics_data_list)

    if x_range is None:
        x_range = (0, T)
    x = np.arange(x_range[0], x_range[1] + 1)

    n_trials = metrics_data_list[0].shape[1]
    T_use = len(x)

    stacked = np.full((T_use, n_trials, n_methods), np.nan)
    for m, data in enumerate(metrics_data_list):
        t_len = min(data.shape[0], T_use)
        stacked[:t_len, :, m] = data[:t_len, :]

    ranks = np.full_like(stacked, np.nan)
    for t in range(T_use):
        for j in range(n_trials):
            row = stacked[t, j, :]
            valid = ~np.isnan(row)
            if valid.sum() >= 1:
                ranks[t, j, valid] = _rankdata_with_tol(
                    row[valid], lower_is_better=low_is_better, eps=rank_eps
                )

    if colors is None:
        colors = ['#8172b3', '#937860', '#4c72b0', '#dd8452', '#55a868', '#c44e52',
                  '#8c8c8c', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
    if markers is None:
        markers = ['o', 's', '^', 'D', 'X', 'P',
                   '*', 'v', '<', '>', 'h', 'H', '+', 'x', 'd', '|', '_']

    fig, ax = plt.subplots(figsize=figsize, dpi=100)

    for m, label in enumerate(metric_names):
        color = colors[m % len(colors)]
        marker = markers[m % len(markers)]

        method_ranks = ranks[:, :, m]
        mean = np.nanmean(method_ranks, axis=1)
        std = np.nanstd(method_ranks, axis=1)
        n_valid = np.sum(~np.isnan(method_ranks), axis=1)
        ci = 1.96 * std / np.sqrt(np.maximum(n_valid, 1))

        ax.plot(x, mean, label=label, color=color, linewidth=2,
                marker=marker if use_markers else None, markevery=marker_frequency, markersize=7,
                markeredgecolor='white', zorder=100-m)
        ax.fill_between(x, mean - ci, mean + ci, color=color, alpha=0.25)

    ax.set_ylim(n_methods + 0.5, 0.5)
    ax.set_yticks(range(1, n_methods + 1))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, pad=10)

    if include_legend:
        legend = ax.legend(frameon=True, framealpha=1, edgecolor='gray', loc=legend_loc)
        legend.set_zorder(200)
        legend.get_frame().set_linewidth(0.5)

    #ax.set_xlim(x_range[0], x_range[1])
    ax.set_xlim(-1, x_range[1])
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=nbins))

    plt.tight_layout()
    return fig, ax

# Feedback adjustor (πBO)


In [ ]:
def pi_bo(
    acq_scores: torch.Tensor, expert_probs: torch.Tensor, beta: float = .5, n: int = 1, **kwargs
    ) -> torch.Tensor:
    # n is the 0-based step, so exponent beta / (t+1)
    return acq_scores * (expert_probs ** (beta / (n + 1)))


# Evaluation loops

`evaluate_policy_vs_bo` rolls out `T` steps with an amortized model, either with its policy head (`"aae"`)
or with `"ucb"` / `"ei"` on its posterior, plus `"feedback"` (greedy on feedback) and `"random"`. `"log_ei"` also works but was much worse than `"ei"`.
`evaluate_gp_bo`, `evaluate_multitask_gp_bo`, `evaluate_imfbo_bo` and `evaluate_rgpe_bo` are the classical-GP baselines.
`run_benchmark_evaluation` runs a list of `MethodConfig`s on each benchmark with shared seeds and pools.


In [ ]:
def _acq_feedback_agreement(scores: torch.Tensor, feedback: torch.Tensor) -> torch.Tensor:
    """
    Top-k overlap: what fraction of the acquisition's top-k candidates
    appear in the expert's top-k? Returns a value in [0, 1].
    k = 10 (or max(1, floor(1% of n_q)) if n_q is small).
    """
    n_q = scores.shape[-1]
    k = max(1, min(5, n_q))

    top_scores_idx = scores.topk(k, dim=-1).indices# [B, k]
    top_feedback_idx = feedback.topk(k, dim=-1).indices  # [B, k]

    # overlap of acq top-k and feedback top-k, pairwise [B, k, 1] vs [B, 1, k]
    matches = (top_scores_idx.unsqueeze(-1) == top_feedback_idx.unsqueeze(-2))  # [B, k, k]
    overlap = matches.any(dim=-1).sum(dim=-1).float()  # [B]

    return overlap / k# [B], fraction in [0, 1]


def evaluate_policy_vs_bo(
    eval_set, cfg, model, experiment, T,
    acquisition="aae", attend_to="data",
    always_T_is_1=False,
    n_theta=2,
    advanced_stats=False,
    concat_feedback=False,
    use_gp=False,
    gp_model=None,
    expert_feedback_adjustor=None,
    Turn_into_probs=True,
    mask_type="none",
    ):
    model.eval()
    batch = deepcopy(eval_set)

    batch.target_all = torch.cat([
        batch.target_all[:, :cfg.task.n_target_data + n_theta - 1],
        batch.target_all[:, -1:]
    ], dim=1)
    
    if concat_feedback:
        batch = concat_feedback_to_batch_x(batch)

    batch_size = batch.context_x.shape[0]
    
    n_tgt_actual = batch.target_y.shape[1]

    batch.target_all = torch.cat([
        batch.target_all[:, :n_tgt_actual + n_theta - 1],
        batch.target_all[:, -1:]
    ], dim=1)
    
    target_mask = create_target_mask(
        mask_type, cfg.task.embedding_type, n_tgt_actual,
        n_theta, cfg.task.n_selected_targets, cfg.task.predefined_masks,
        None, cfg.task.mask_index, attend_to,
    )

    batch.target_mask = target_mask if acquisition == "aae" else None

    selected_values = torch.zeros((T + 1, batch_size), device=batch.context_x.device)
    best_so_far = torch.zeros((T + 1, batch_size), device=batch.context_x.device)
    rmse_values = torch.zeros((T, batch_size), device=batch.context_x.device)
    regret_values = torch.zeros((T, batch_size), device=batch.context_x.device)
    regret_fraction_within_5_percent = torch.zeros((T, batch_size), device=batch.context_x.device)
    cumulative_regret_values = torch.zeros((T, batch_size), device=batch.context_x.device)

    best_y = batch.context_y.max(dim=1).values.squeeze(-1).to(batch.context_x.device)
    selected_values[0] = best_y
    best_so_far[0] = best_y
    best_y = best_y.unsqueeze(-1)
    
    acq_feedback_agreement_batch = []
    agreements_hard_batch = []
    agreements_soft_batch = []
    mean_y_diffs_batch = []
    mean_y_diffs_sampled_batch = []
    expert_best_hits_batch = []
    expert_rank_batch = []
    expert_feedback_stats = {}

    if advanced_stats:
        expert_feedback_stats["other"] = {
            "target_x": batch.target_x[:, :cfg.task.n_target_data].detach().cpu(),
            "target_y": batch.target_all[:, :cfg.task.n_target_data].detach().cpu(),
            "target_feedback": getattr(batch, "target_feedback", None).detach().cpu(),
            "thetas_true": batch.target_all[:, cfg.task.n_target_data:].detach().cpu(),
            "data_targets": {"pred_mean": [], "pred_var": []},
            "thetas": {"pred_mean": [], "pred_var": []},
            "gmm_feedback_log_likelihood": [],
            "gmm_feedback_ll_mask": [],
        }

    n_query_orig = batch.query_y.shape[1]
    true_max = torch.cat([batch.context_y, batch.query_y], dim=1).max(dim=1).values.squeeze(-1)

    gp_state = None

    for t in range(T):
        batch.t = torch.tensor([1.] if always_T_is_1 else [t / T], device=device)

        if use_gp:
            gp_botorch, gp_state = _fit_botorch_gp(
                batch.context_x, batch.context_y.squeeze(-1), prev_state=gp_state
            )
            with torch.no_grad():
                qry_post = gp_botorch.posterior(batch.query_x)
                weighted_mean_query = qry_post.mean.squeeze(-1)
                weighted_std_query = qry_post.variance.clamp_min(1e-8).sqrt().squeeze(-1)

                tgt_post = gp_botorch.posterior(batch.target_x[:, :cfg.task.n_target_data])
                weighted_mean = tgt_post.mean.squeeze(-1)

            design_out_idx = None

        else:
            outs = model.forward(batch)
            design_out = outs.design_out
            posterior = outs.posterior_out
            posterior_query = outs.posterior_out_query
            if advanced_stats:
                _feedback_vals_for_stats = batch.query_feedback[..., 0].detach()

            weighted_mean = (posterior.mixture_means *
                                posterior.mixture_weights).sum(dim=-1)
            weighted_std = torch.sqrt(
                (posterior.mixture_weights * posterior.mixture_stds ** 2).sum(dim=-1) + 1e-8
            )
            weighted_mean_query = (posterior_query.mixture_means *
                                    posterior_query.mixture_weights).sum(dim=-1)
            weighted_std_query = torch.sqrt(
                (posterior_query.mixture_weights *
                    posterior_query.mixture_stds ** 2).sum(dim=-1) + 1e-8
            )
            design_out_idx = design_out.idx

        X = batch.query_x.unsqueeze(-2)
        best_f = best_y.squeeze(-1)

        with torch.no_grad():
            if acquisition == "aae" and not use_gp:
                acquisition_scores = design_out.zt
                idx = design_out_idx

            elif acquisition == "log_ei":  # much worse than plain ei in our runs, kept for reference
                from utils.free_mode import log_expected_improvement
                acquisition_scores = log_expected_improvement(weighted_mean_query, weighted_std_query, best_f.unsqueeze(-1))
                idx = acquisition_scores.argmax(dim=1, keepdim=True)

            elif acquisition == "ei":
                from utils.free_mode import expected_improvement
                acquisition_scores = expected_improvement(weighted_mean_query, weighted_std_query, best_f.unsqueeze(-1))
                idx = acquisition_scores.argmax(dim=1, keepdim=True)
            elif acquisition == "ucb":
                from utils.free_mode import upper_confidence_bound
                kappa = getattr(cfg, "bo_kappa", 1.0)
                acquisition_scores = upper_confidence_bound(weighted_mean_query, weighted_std_query, kappa)
                idx = acquisition_scores.argmax(dim=1, keepdim=True)
            
            elif acquisition == "feedback":
                acquisition_scores = batch.query_feedback[:, :, 0]
                idx = acquisition_scores.argmax(dim=1, keepdim=True)

            elif acquisition == "random":
                acquisition_scores = torch.rand(batch_size, batch.query_x.shape[1], device=device)
                idx = acquisition_scores.argmax(dim=1, keepdim=True)

            else:
                raise NotImplementedError(f"Unknown acquisition: {acquisition}")

        #acquisition-feedback agreement
        if acquisition_scores is not None:
            scores_2d = (acquisition_scores.mean(dim=-1)
                if acquisition_scores.dim() == 3
                else acquisition_scores)
            feedback_2d = batch.query_feedback[:, :acquisition_scores.shape[1], 0]
            acq_feedback_agreement_batch.append(
                _acq_feedback_agreement(
                    scores_2d.detach(), feedback_2d.detach()
                ).cpu()
            )

        # expert feedback adjustment
        if expert_feedback_adjustor is not None and acquisition_scores is not None:
            feedback = batch.query_feedback[:, :, 0]
            if Turn_into_probs:
                feedback = torch.softmax(feedback, 1)
            if acquisition_scores.dim() == 3:
                scores_for_adjustor = acquisition_scores.mean(dim=-1)
            else:
                scores_for_adjustor = acquisition_scores
            acquisition_scores_adjusted = expert_feedback_adjustor(
                scores_for_adjustor, feedback, n=t
            )
            idx = acquisition_scores_adjusted.argmax(dim=1, keepdim=True)

        # step bookkeeping
        next_y = torch.gather(batch.query_y.squeeze(-1), 1, idx)
        selected_values[t + 1] = next_y.squeeze(-1)
        best_y = torch.max(best_y, next_y)
        best_so_far[t + 1] = best_y.squeeze(-1)

        rmse_values[t] = torch.sqrt(
            ((weighted_mean[:, :cfg.task.n_target_data] -
                batch.target_all[:, :cfg.task.n_target_data].squeeze(-1)) ** 2
                ).mean(dim=-1)
        )

        best_available_t = true_max
        regret_values[t] = best_available_t - next_y.squeeze(-1)
        regret_values_best = best_available_t - best_y.squeeze(-1)
        regret_fraction_within_5_percent[t] = (regret_values_best <= 0.05 * best_available_t).float()
        cumulative_regret_values[t] = cumulative_regret_values[t - 1] + regret_values_best

        if getattr(cfg.task, "use_expert_model", False):
            with torch.no_grad():
                expert_probs = batch.query_feedback[..., 0]
                if getattr(cfg.task, "normalize_outputs_to_max_one", False):
                    expert_probs = expert_probs / expert_probs.sum(dim=-1, keepdim=True)

                policy_actions = (
                    idx.squeeze(-1) if use_gp
                    else design_out.zt.argmax(dim=-1)
                )
                expert_actions = expert_probs.argmax(dim=-1)

                agreements_hard_batch.append((policy_actions == expert_actions).float())
                if not use_gp:
                    agreements_soft_batch.append(
                        (design_out.zt * expert_probs).sum(dim=-1)
                    )

                qy_full = batch.query_y
                model_y = qy_full[torch.arange(qy_full.size(0)), policy_actions].squeeze(-1)
                expert_y = qy_full[torch.arange(qy_full.size(0)), expert_actions].squeeze(-1)
                mean_y_diffs_batch.append(model_y - expert_y)
                mean_y_diffs_sampled_batch.append(next_y.squeeze(-1) - expert_y)

                qy = batch.query_y.squeeze(-1)
                expert_best_hits_batch.append((expert_actions == qy.argmax(dim=1)).float())
                expert_rank_batch.append(
                    ((qy > expert_y.unsqueeze(-1)).sum(dim=1) + 1).float()
                )

        batch = experiment.update_batch(batch, idx)

        if advanced_stats and not use_gp:
            other = expert_feedback_stats["other"]
            n_data = cfg.task.n_target_data
            n_total = batch.target_all.shape[1]
            pred_mean = weighted_mean[:, :n_total]
            pred_var = weighted_std[:, :n_total] ** 2
            other["data_targets"]["pred_mean"].append(pred_mean[:, :n_data].detach().cpu())
            other["data_targets"]["pred_var"].append(pred_var[:, :n_data].detach().cpu())
            other["thetas"]["pred_mean"].append(pred_mean[:, n_data:].detach().cpu())
            other["thetas"]["pred_var"].append(pred_var[:, n_data:].detach().cpu())

            feedback_vals = _feedback_vals_for_stats
            means = posterior_query.mixture_means
            stds = posterior_query.mixture_stds
            weights = posterior_query.mixture_weights
            y_exp = feedback_vals.unsqueeze(-1)
            comp_log_prob = Normal(means, stds + 1e-8).log_prob(y_exp)
            log_mixture = torch.logsumexp(comp_log_prob + torch.log(weights + 1e-8), dim=-1)

            pad_size = n_query_orig - log_mixture.shape[1]
            if pad_size > 0:
                log_mixture_padded = F.pad(log_mixture, (0, pad_size), value=0.0)
                mask = torch.cat([
                    torch.ones(log_mixture.shape, dtype=torch.bool),
                    torch.zeros((*log_mixture.shape[:-1], pad_size), dtype=torch.bool),
                ], dim=-1)
            else:
                log_mixture_padded = log_mixture
                mask = torch.ones(log_mixture.shape, dtype=torch.bool)

            other["gmm_feedback_log_likelihood"].append(log_mixture_padded.detach().cpu())
            other["gmm_feedback_ll_mask"].append(mask.cpu())

    # aggregate stats
    def step_stats(values_list):
        return [
            {"mean": v.mean().item(), "variance": v.var(unbiased=False).item(),
                "min": v.min().item(), "max": v.max().item()}
            for v in values_list
        ]

    expert_feedback_stats.update({
        "model_policy_argmax_vs_expert": [{"mean": v.mean().item()} for v in agreements_hard_batch],
        "model_policy_distribution_vs_expert": step_stats(agreements_soft_batch),
        "model_predictions_minus_expert": step_stats(mean_y_diffs_batch),
        "sampled_model_outputs_minus_expert": step_stats(mean_y_diffs_sampled_batch),
        "fraction_expert_selected_true_best": [{"mean": v.mean().item()} for v in expert_best_hits_batch],
        "rank_of_expert_choice_among_all_options": step_stats(expert_rank_batch),
        "acq_feedback_agreement": (
            torch.stack(acq_feedback_agreement_batch, dim=0)
            if acq_feedback_agreement_batch else
            torch.full((T, batch_size), float("nan"))
        ),  # [T, B]
    })

    if advanced_stats:
        other = expert_feedback_stats["other"]
        for group in ["data_targets", "thetas"]:
            for stat in ["pred_mean", "pred_var"]:
                other[group][stat] = torch.stack(other[group][stat], dim=0)
        other["gmm_feedback_log_likelihood"] = torch.stack(
            other["gmm_feedback_log_likelihood"], dim=0
        )
        other["gmm_feedback_ll_mask"] = torch.stack(
            other["gmm_feedback_ll_mask"], dim=0
        )

    return (
        selected_values.detach().cpu(),
        best_so_far.detach().cpu(),
        rmse_values.detach().cpu(),
        regret_values.detach().cpu(),
        cumulative_regret_values.detach().cpu(),
        regret_fraction_within_5_percent.detach().cpu(),
        expert_feedback_stats,
    )

In [ ]:
class BatchedGP(ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        batch_shape = torch.Size([train_x.shape[0]])
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=batch_shape)
        self.covar_module = ScaleKernel(
            MaternKernel(nu=1.5, batch_shape=batch_shape, ard_num_dims=train_x.shape[-1]),
            batch_shape=batch_shape,
        )

    def forward(self, x):
        return MultivariateNormal(
            self.mean_module(x),
            self.covar_module(x),
        )


def _normalize_x(x):
    """Normalize inputs. Falls back to identity when scale can't be estimated."""
    x_mean = x.mean(dim=1, keepdim=True)
    x_std = x.std(dim=1, keepdim=True)
    # norm only dims with spread
    # flat dim -> std=0 -> rank-1 kernel -> NotPSD
    safe = x_std > 1e-2
    x_std_safe = torch.where(safe, x_std, torch.ones_like(x_std))
    x_mean_out = torch.where(safe, x_mean, torch.zeros_like(x_mean))
    return (x - x_mean_out) / x_std_safe, x_mean_out, x_std_safe


def _normalize_y(y):
    """Normalize outputs per batch element. [B, N] -> [B, N], stats [B, 1]"""
    y_mean = y.mean(dim=1, keepdim=True)  # [B, 1]
    y_std = y.std(dim=1, keepdim=True)  # [B, 1]
    # N=1, std=0: abs(mean) so targets dont collapse
    fallback = y.abs().mean(dim=1, keepdim=True).clamp(min=1.0)
    y_std = torch.where(y_std < 1e-2, fallback, y_std)
    return (y - y_mean) / y_std, y_mean, y_std


def _fit_batched_gp(train_x, train_y, n_iter=30, lr=0.1, prev_state=None):
    device = train_x.device
    B = train_x.shape[0]

    train_x_norm, x_mean, x_std = _normalize_x(train_x)
    train_y_norm, y_mean, y_std = _normalize_y(train_y)

    likelihood = GaussianLikelihood(batch_shape=torch.Size([B])).to(device)
    model = BatchedGP(train_x_norm, train_y_norm, likelihood).to(device)

    if prev_state is not None:
        try:
            model.load_state_dict(prev_state["model"], strict=False)
            likelihood.load_state_dict(prev_state["likelihood"], strict=False)
        except Exception:
            pass

    # reject a warm-start that already contains NaNs
    if any(torch.isnan(p).any() for p in model.parameters()):
        model = BatchedGP(train_x_norm, train_y_norm, likelihood).to(device)

    model.train(); likelihood.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mll = ExactMarginalLogLikelihood(likelihood, model)

    clean = True
    for _ in range(n_iter):
        optimizer.zero_grad()
        try:
            with gpytorch.settings.cholesky_jitter(1e-2):
                loss = -mll(model(train_x_norm), train_y_norm).mean()
        except Exception:  # NotPSDError or similar
            clean = False
            break
        if not torch.isfinite(loss):
            clean = False
            break
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

    model.eval(); likelihood.eval()

    # persist only if no numerical failures
    state = (
        {"model": model.state_dict(), "likelihood": likelihood.state_dict()}
        if clean else None
    )
    norm_stats = {"x_mean": x_mean, "x_std": x_std, "y_mean": y_mean, "y_std": y_std}
    return model, likelihood, state, norm_stats


def _batched_posterior(model, likelihood, x, norm_stats):
    """
    Predict with batched GP, undoing output normalization.
    x:          [B, N, d]
    norm_stats: dict from _fit_batched_gp
    Returns mean [B, N], std [B, N] in original output scale.
    """
    x_norm = (x - norm_stats["x_mean"]) / norm_stats["x_std"]
    with torch.no_grad(), gpytorch.settings.fast_pred_var(), gpytorch.settings.cholesky_jitter(1e-3):
        pred = likelihood(model(x_norm))
        mean_norm = pred.mean  # [B, N]
        std_norm = pred.variance.clamp(min=1e-8).sqrt()

    # undo output normalization, y_mean/y_std are [B, 1], broadcast over N
    mean = mean_norm * norm_stats["y_std"] + norm_stats["y_mean"]
    std = std_norm * norm_stats["y_std"]

    # guard against any remaining NaNs
    mean = torch.nan_to_num(mean, nan=0.0)
    std = torch.nan_to_num(std, nan=1.0)
    return mean, std


def _update_batch(batch, idx):
    B = batch.query_x.shape[0]
    device = batch.query_x.device
    batch_idx = torch.arange(B, device=device)
    idx_sq = idx.squeeze(-1)

    next_x = batch.query_x[batch_idx, idx_sq].unsqueeze(1)
    next_y = batch.query_y[batch_idx, idx_sq].unsqueeze(1)
    next_f = batch.query_feedback[batch_idx, idx_sq].unsqueeze(1)

    mask = torch.ones(B, batch.query_x.shape[1], dtype=torch.bool, device=device)
    mask[batch_idx, idx_sq] = False

    batch.query_x = batch.query_x[mask].view(B, -1, batch.query_x.shape[-1])
    batch.query_y = batch.query_y[mask].view(B, -1, 1)
    batch.query_feedback = batch.query_feedback[mask].view(B, -1, 1)

    batch.context_x = torch.cat([batch.context_x, next_x], dim=1)
    batch.context_y = torch.cat([batch.context_y, next_y], dim=1)
    batch.context_feedback = torch.cat([batch.context_feedback, next_f], dim=1)

    return batch


def _init_tracking(batch, T):
    B = batch.context_x.shape[0]
    device = batch.context_x.device
    best_y = batch.context_y.max(dim=1).values.squeeze(-1)
    true_max = torch.cat([batch.context_y, batch.query_y], dim=1).max(dim=1).values.squeeze(-1)

    selected_values = torch.zeros(T + 1, B, device=device)
    best_so_far = torch.zeros(T + 1, B, device=device)
    rmse_values = torch.zeros(T, B, device=device)
    regret_values = torch.zeros(T, B, device=device)
    cumulative_regret_values = torch.zeros(T, B, device=device)
    regret_fraction_within_5_percent = torch.zeros(T, B, device=device)

    selected_values[0] = best_y
    best_so_far[0] = best_y

    return (
        best_y.unsqueeze(-1), true_max,
        selected_values, best_so_far,
        rmse_values, regret_values,
        cumulative_regret_values, regret_fraction_within_5_percent,
    )


def _acq_scores(mean, std, best_y, acquisition, kappa):
    """
    mean, std: [B, N]
    best_y:    [B, 1]
    returns:   [B, N]
    """
    if acquisition == "ei":
        z = (mean - best_y) / (std + 1e-8)
        normal = Normal(torch.zeros_like(z), torch.ones_like(z))
        return (mean - best_y) * normal.cdf(z) + std * torch.exp(normal.log_prob(z))
    elif acquisition == "log_ei":  # much worse than plain ei in our runs, kept for reference
        z = (mean - best_y) / (std + 1e-8)
        normal = Normal(torch.zeros_like(z), torch.ones_like(z))
        log_phi = normal.log_prob(z)
        log_Phi = torch.log(normal.cdf(z).clamp(min=torch.finfo(z.dtype).tiny))
        log_std = torch.log(std + 1e-8)
        return log_std + log_phi + torch.log1p(z * torch.exp(log_Phi - log_phi))
    elif acquisition == "ucb":
        return mean + kappa * std
    else:
        raise ValueError(f"Unknown acquisition: {acquisition}")


def _step_update(
    batch, idx, best_y, true_max,
    selected_values, best_so_far,
    rmse_values, regret_values,
    cumulative_regret_values, regret_fraction_within_5_percent,
    t, gp_mean_target, n_target_data,
    ):
    B = batch.context_x.shape[0]
    device = batch.context_x.device
    batch_idx = torch.arange(B, device=device)

    next_y = batch.query_y.squeeze(-1)[batch_idx, idx.squeeze(-1)].unsqueeze(-1)
    best_y = torch.max(best_y, next_y)

    selected_values[t + 1] = next_y.squeeze(-1)
    best_so_far[t + 1] = best_y.squeeze(-1)

    target_gt = batch.target_all[:, :n_target_data].squeeze(-1)  # [B, n_target]
    rmse_values[t] = torch.sqrt(((gp_mean_target - target_gt) ** 2).mean(dim=-1))

    # instant regret: regret of the point selected at this step (matches original)
    regret_values[t] = true_max - next_y.squeeze(-1)

    # cum regret: uses best-so-far regret (matches original)
    regret_values_best = true_max - best_y.squeeze(-1)
    prev_cum = cumulative_regret_values[t - 1] if t > 0 else torch.zeros_like(regret_values_best)
    cumulative_regret_values[t] = prev_cum + regret_values_best

    # fraction within 5% of true max using best-so-far (matches original)
    regret_fraction_within_5_percent[t] = (regret_values_best <= 0.05 * true_max).float()

    batch = _update_batch(batch, idx)
    return batch, best_y


# 0. vanilla GP-BO, opt feedback augmentation
#    ucb, ei, log-ei, random


def evaluate_gp_bo(
    eval_set, T, n_target_data,
    acquisition="ei",
    kappa=1.0,
    n_iter=30,
    expert_feedback_adjustor=None,
    turn_into_probs=True,
    ):
    """
    GP-BO with batched GPyTorch surrogate.
    Optionally augments acquisition with expert feedback via adjustor function,
    mirroring the interface of evaluate_policy_vs_bo.

    acquisition: one of "ucb", "ei", "log_ei", "random"
    expert_feedback_adjustor: callable(feedback, acq_scores, n=t) -> adjusted_scores
                              same signature as in the adjustor functions file
    turn_into_probs: if True, softmax the feedback before passing to adjustor
    """
    batch = deepcopy(eval_set)
    B = batch.context_x.shape[0]
    device = batch.context_x.device

    (
        best_y, true_max,
        selected_values, best_so_far,
        rmse_values, regret_values,
        cumulative_regret_values, regret_fraction_within_5_percent,
    ) = _init_tracking(batch, T)

    hf_state = None

    for t in tqdm(range(T), desc=f"GP-BO ({acquisition})"):
        ctx_x = batch.context_x  # [B, n_ctx, d]
        ctx_y = batch.context_y.squeeze(-1)  # [B, n_ctx]
        qry_x = batch.query_x  # [B, n_q, d]
        tgt_x = batch.target_x[:, :n_target_data]  # [B, n_target, d]

        model, lik, hf_state, norm_stats = _fit_batched_gp(
            ctx_x, ctx_y, n_iter=n_iter, prev_state=hf_state
        )

        mean, std = _batched_posterior(model, lik, qry_x, norm_stats)  # [B, n_q]
        mean_t, _ = _batched_posterior(model, lik, tgt_x, norm_stats)  # [B, n_target]

        # compute acquisition scores [B, n_q]
        if acquisition == "ucb":
            scores = mean + kappa * std
        elif acquisition == "ei":
            z = (mean - best_y) / (std + 1e-8)
            normal = Normal(torch.zeros_like(z), torch.ones_like(z))
            scores = (mean - best_y) * normal.cdf(z) + std * torch.exp(normal.log_prob(z))
        elif acquisition == "log_ei":  # much worse than plain ei in our runs, kept for reference
            z = (mean - best_y) / (std + 1e-8)
            normal = Normal(torch.zeros_like(z), torch.ones_like(z))
            log_phi = normal.log_prob(z)
            log_Phi = torch.log(normal.cdf(z).clamp(min=torch.finfo(z.dtype).tiny))
            log_std = torch.log(std + 1e-8)
            scores = log_std + log_phi + torch.log1p(z * torch.exp(log_Phi - log_phi))
            scores = scores.clamp(min=torch.finfo(z.dtype).min)
        elif acquisition == "random":
            scores = torch.rand_like(mean)
        else:
            raise ValueError(f"Unknown acquisition: {acquisition}")

        # augment w/ expert feedback
        if expert_feedback_adjustor is not None:
            feedback = batch.query_feedback[:, :, 0]  # [B, n_q]
            if turn_into_probs:
                feedback = torch.softmax(feedback, dim=1)
            #scores = torch.softmax(scores, dim=1)

            scores = expert_feedback_adjustor(scores,feedback, n=t)

        idx = scores.argmax(dim=1, keepdim=True)

        batch, best_y = _step_update(
            batch, idx, best_y, true_max,
            selected_values, best_so_far,
            rmse_values, regret_values,
            cumulative_regret_values, regret_fraction_within_5_percent,
            t, mean_t, n_target_data,
        )

    return (
        selected_values.cpu(),
        best_so_far.cpu(),
        rmse_values.cpu(),
        regret_values.cpu(),
        cumulative_regret_values.cpu(),
        regret_fraction_within_5_percent.cpu(),
        {},
    )


In [ ]:
def _summarize_mtgp_diags(diags, label):
    """
    Collapse the per-step fit diagnostics from utils.multitask_gp into one line.

    Every failure path in multitask_gp recovers silently (skipped Adam steps, a posterior that
    falls back to a constant mean/std, a warm start that never takes), so a run where the GP never
    actually trained is otherwise indistinguishable in this notebook's output from a healthy one.
    """
    if not diags:
        return f"    [{label}] no fit diagnostics"

    n = len(diags)
    trained = sum(d["trained"] for d in diags)
    skipped = sum(d["steps_skipped"] for d in diags)
    taken = sum(d["steps_taken"] for d in diags)
    fallbacks = sum(d["posterior_fallback"] for d in diags)
    max_jitter = max(d["max_jitter"] for d in diags)
    last = diags[-1]

    ls = last["lengthscale"]
    ls_str = "n/a" if ls is None else "[" + ", ".join(f"{v:.3f}" for v in ls) + "]"
    noise = last["noise"]
    noise_str = "n/a" if noise is None else "[" + ", ".join(f"{v:.4g}" for v in noise) + "]"
    corr_str = "n/a" if last["task_corr"] is None else f"{last['task_corr']:+.3f}"

    errors = [e for d in diags for e in d["errors"]]
    msg = (
        f"    [{label}] fits={trained}/{n} adam_steps={taken} skipped={skipped} "
        f"posterior_fallbacks={fallbacks} max_jitter={max_jitter:g}\n"
        f"    [{label}] final lengthscale={ls_str} noise={noise_str} task_corr={corr_str}"
    )
    if errors:
        uniq = list(dict.fromkeys(errors))
        msg += f"\n    [{label}] {len(errors)} error(s), e.g. {uniq[0][:160]}"
    return msg


def evaluate_multitask_gp_bo(
    eval_set, T, n_target_data,
    acquisition="ucb",
    kappa=1.0,
    mtgp_kind="icm",
    rank=None,
    max_pool=400,  # 64
    n_iter=50,
    warm_start=True,
    expert_feedback_adjustor=None,
    turn_into_probs=True,
    verbose_diagnostics=True,
    ):
    """
    GP-BO with a botorch multi-task surrogate over (objective, expert feedback).

    mtgp_kind: "icm"       -> the batched ICM model (utils/multitask_gp.BatchedMultiTaskGP),
                              matching botorch MultiTaskGP's construction. Tasks may live on
                              different inputs, so the feedback task is fit on the whole current
                              pool (context + query), where feedback is already known, while the
                              objective task only has the context points.
               "kronecker" -> the *same* BatchedMultiTaskGP/ICM model, NOT botorch's
                              KroneckerMultiTaskGP, despite the name, no Kronecker-structured
                              GP is used anywhere in this codebase. Only the data is block-aligned
                              (both tasks restricted to the context points), which means it never
                              sees the pool-wide feedback and is, structurally, a no-feedback
                              baseline, handling tasks observed at *different* inputs is exactly
                              what "icm" is for. Kept only for comparison; prefer "icm".
    n_iter:    Adam steps per model fit (see utils/multitask_gp._fit_batch). Both kinds are fit
               as a single batched gpytorch GP covering all B environments at once (see
               utils/multitask_gp.BatchedMultiTaskGP), the same batching pattern
               evaluate_gp_bo's BatchedGP already uses, not B separate per-environment fits.
    verbose_diagnostics: print a one-line fit-health summary at the end of the rollout. Leave this
               on: the multi-task fit recovers from every failure silently, so this line is the
               only signal that the surrogate actually trained.
    """
    from utils import multitask_gp
    from utils.free_mode import gp_scores

    batch = deepcopy(eval_set)

    (
        best_y, true_max,
        selected_values, best_so_far,
        rmse_values, regret_values,
        cumulative_regret_values, regret_fraction_within_5_percent,
    ) = _init_tracking(batch, T)

    states = None
    diags = []

    for t in tqdm(range(T), desc=f"MTGP-BO ({mtgp_kind}/{acquisition})"):
        qry_x = batch.query_x  # [B, n_q, d]
        tgt_x = batch.target_x[:, :n_target_data]  # [B, n_target, d]

        fits, states, diag = multitask_gp.fit_multitask_gps(
            batch.context_x, batch.context_y, batch.context_feedback,
            qry_x, batch.query_feedback,
            kind=mtgp_kind, rank=rank, max_pool=max_pool, n_iter=n_iter,
            prev_states=states if warm_start else None,
        )
        diags.append(diag)

        mean, std = multitask_gp.multitask_posterior(fits, qry_x)  # [B, n_q]
        mean_t, _ = multitask_gp.multitask_posterior(fits, tgt_x)  # [B, n_target]

        if acquisition == "random":
            scores = torch.rand_like(mean)
        else:
            scores = gp_scores(acquisition, mean, std, best_f=best_y, kappa=kappa)

        # augment w/ expert feedback, as in evaluate_gp_bo
        if expert_feedback_adjustor is not None:
            feedback = batch.query_feedback[:, :, 0]
            if turn_into_probs:
                feedback = torch.softmax(feedback, dim=1)
            scores = expert_feedback_adjustor(scores, feedback, n=t)

        idx = scores.argmax(dim=1, keepdim=True)

        batch, best_y = _step_update(
            batch, idx, best_y, true_max,
            selected_values, best_so_far,
            rmse_values, regret_values,
            cumulative_regret_values, regret_fraction_within_5_percent,
            t, mean_t, n_target_data,
        )

    if verbose_diagnostics:
        print(_summarize_mtgp_diags(diags, f"mtgp_{mtgp_kind}_{acquisition}"))
        print(f"    [mtgp_{mtgp_kind}_{acquisition}] trust (avg task corr) = {diags[-1]['task_corr']:+.3f}")

    return (
        selected_values.cpu(),
        best_so_far.cpu(),
        rmse_values.cpu(),
        regret_values.cpu(),
        cumulative_regret_values.cpu(),
        regret_fraction_within_5_percent.cpu(),
        {"diags": diags},
    )


In [ ]:
def _summarize_imfbo_diags(diags, label):
    """
    Collapse the per-step fit diagnostics from utils.imfbo_gp into a few lines.

    Same reason _summarize_mtgp_diags exists: every failure path in imfbo_gp recovers silently
    (skipped Adam steps, a posterior that falls back to a constant mean/std, a warm start that never
    takes), so a run where the surrogate never actually trained is otherwise indistinguishable here
    from a healthy one.

    The four iMFBO-specific numbers, and how to read them:

      lf_snr     low-fidelity signal-to-noise, a^2 * outputscale / mean(sigma_lf^2). Above ~1 the
                 expert feedback is informative as a function of the *visible* x; far below 1 means
                 sigma_lf(x) has inflated and the feedback rows have effectively dropped out of the
                 posterior, i.e. this has degraded to a plain GP on the context points. On the 3-D
                 benchmarks, where query_x hides the expert coordinate and the feedback is white
                 noise in the visible x, that degradation is the CORRECT outcome, not a bug.

      lf_share   how much of the posterior mean the feedback rows actually move, in [0, 1]. The
                 direct version of the same question, measured rather than inferred.

      gamma      NVUCB's variance-reduction factor sigma_t / sqrt(sigma_t^2 + sigma_hf^2). At ~1.00
                 the nvucb / nucb / ucb variants are NUMERICALLY THE SAME METHOD (all three collapse
                 to plain UCB as sigma_hf -> 0), so any ranking between them is meaningless. Check
                 this before drawing conclusions about the acquisition.

      ls_floor   fraction of ARD lengthscales pinned near their floor. This is the OTHER way the fit
                 can absorb unusable feedback: instead of inflating sigma_lf, shrink the lengthscale
                 until the feedback rows interpolate. The lengthscale is shared with the objective's
                 own rows, so that collapses the objective's fit too. A high lf_snr together with a
                 high ls_floor does not mean "the feedback helped", it means the surrogate is
                 interpolating noise.
    """
    if not diags:
        return f"    [{label}] no fit diagnostics"

    n = len(diags)
    trained = sum(d["trained"] for d in diags)
    skipped = sum(d["steps_skipped"] for d in diags)
    taken = sum(d["steps_taken"] for d in diags)
    fallbacks = sum(d["posterior_fallback"] for d in diags)
    max_jitter = max(d["max_jitter"] for d in diags)
    last = diags[-1]

    def _f(key, fmt="{:.4g}"):
        v = last.get(key)
        return "n/a" if v is None else fmt.format(v)

    ls = last["lengthscale"]
    ls_str = "n/a" if ls is None else "[" + ", ".join(f"{v:.3f}" for v in ls) + "]"

    msg = (
        f"    [{label}] fits={trained}/{n} adam_steps={taken} skipped={skipped} "
        f"posterior_fallbacks={fallbacks} max_jitter={max_jitter:g} "
        f"rows={last['n_hf']}hf/{last['n_lf']}lf\n"
        f"    [{label}] lengthscale={ls_str} (floor_frac={_f('lengthscale_floor_frac', '{:.2f}')}) "
        f"outputscale={_f('outputscale')} a={_f('scale', '{:+.3f}')} b={_f('offset', '{:+.3f}')}\n"
        f"    [{label}] hf_noise={_f('hf_noise')} lf_noise={_f('lf_noise_mean')} "
        f"(p05={_f('lf_noise_p05')} p95={_f('lf_noise_p95')} "
        f"floor_frac={_f('lf_noise_floor_frac', '{:.2f}')})\n"
        f"    [{label}] TRUST: lf_snr={_f('lf_snr')} lf_share={_f('lf_share', '{:.3f}')} "
        f"gamma={_f('gamma_mean', '{:.3f}')}"
    )

    errors = [e for d in diags for e in d["errors"]]
    if errors:
        uniq = list(dict.fromkeys(errors))
        msg += f"\n    [{label}] {len(errors)} error(s), e.g. {uniq[0][:160]}"
    return msg


def evaluate_imfbo_bo(
    eval_set, T, n_target_data,
    acquisition="nvucb",
    kappa=1.0,
    noise_model="parametric",
    max_pool=192,
    n_iter=50,
    warm_start=True,
    expert_feedback_adjustor=None,
    turn_into_probs=True,
    verbose_diagnostics=True,
    ):
    """
    GP-BO with the iMFBO surrogate: "Multi-fidelity Bayesian Optimization with Multiple Information
    Sources of Input-dependent Fidelity", Fan et al., UAI 2024 (PMLR 244:1271-1293).

    In the paper every information source is an unbiased noisy view of one shared latent objective,
    f_j(x) = f(x) + sigma_j(x) * S, fidelity IS the input-dependent noise scale, and all sources
    enter ONE GP through a heteroscedastic diagonal. Here the low-fidelity source is the expert
    feedback, which is given for free at every pool point, and only the high-fidelity y is ever
    queried. So the paper's joint (x, source) selection and its cost/budget machinery drop out, and
    what remains is a single GP over f that ingests the pool-wide feedback as direct noisy
    observations of f, with the learned sigma_lf(x) acting as a trust map. See utils/imfbo_gp.py for
    the full derivation and the design choices.

    This is a different transfer channel from the other feedback baselines here: mtgp_* makes the
    feedback a second GP task coupled through a scalar task correlation with a constant per-task
    noise; gp_feat_* makes it an extra input column; rgpe_* makes it a separate base task. iMFBO
    makes it the *same* task as f, affine-calibrated (fb = a*f + b, since y_hidden carries a
    different normalisation from y), and trusted as a function of x.

    acquisition: "nvucb" (paper eq. 13, UCB with the exploration term scaled by the variance
                 reduction a query would actually buy), "nucb" (eq. 17, the naive variant the paper
                 argues against), "ucb" (same surrogate, plain UCB, the control that separates the
                 surrogate's contribution from the acquisition's), or "random".
    noise_model: "parametric" (sigma_lf(x) from a small per-environment MLP) or "gp" (a free per-row
                 log-noise vector under a GP prior, the paper's eq.-12 MAP estimate).
    max_pool:    feedback rows carried per step. This is the cost knob, the batched Cholesky is
                 cubic in it, so it is deliberately below the mtgp_* default of 400.
    verbose_diagnostics: print the fit-health summary at the end of the rollout. Leave this on and
                 read it, see _summarize_imfbo_diags for why lf_snr, lf_share, gamma and ls_floor
                 decide whether a given benchmark's numbers mean anything at all.
    """
    from utils import imfbo_gp

    batch = deepcopy(eval_set)

    (
        best_y, true_max,
        selected_values, best_so_far,
        rmse_values, regret_values,
        cumulative_regret_values, regret_fraction_within_5_percent,
    ) = _init_tracking(batch, T)

    state = None
    diags = []

    for t in tqdm(range(T), desc=f"iMFBO ({noise_model}/{acquisition})"):
        qry_x = batch.query_x  # [B, n_q, d]
        tgt_x = batch.target_x[:, :n_target_data]  # [B, n_target, d]

        fits, state, diag = imfbo_gp.fit_imfbo_gp(
            batch.context_x, batch.context_y, batch.context_feedback,
            qry_x, batch.query_feedback,
            noise_model=noise_model, max_pool=max_pool, n_iter=n_iter,
            prev_state=state if warm_start else None,
        )
        diags.append(diag)

        mean, std = imfbo_gp.imfbo_posterior(fits, qry_x)  # [B, n_q]
        mean_t, _ = imfbo_gp.imfbo_posterior(fits, tgt_x)  # [B, n_target]
        sigma_hf = imfbo_gp.imfbo_query_noise(fits)  # [B, 1]

        if acquisition == "random":
            scores = torch.rand_like(mean)
        else:
            scores = imfbo_gp.imfbo_scores(acquisition, mean, std, sigma_hf=sigma_hf, kappa=kappa)

        # per step, depends on this step's pool
        diag["gamma_mean"] = imfbo_gp.imfbo_gamma(std, sigma_hf).mean().item()
        diag["lf_share"] = imfbo_gp.imfbo_lf_share(fits, qry_x).mean().item()

        # augment w/ expert feedback, as in evaluate_gp_bo
        if expert_feedback_adjustor is not None:
            feedback = batch.query_feedback[:, :, 0]
            if turn_into_probs:
                feedback = torch.softmax(feedback, dim=1)
            scores = expert_feedback_adjustor(scores, feedback, n=t)

        idx = scores.argmax(dim=1, keepdim=True)

        batch, best_y = _step_update(
            batch, idx, best_y, true_max,
            selected_values, best_so_far,
            rmse_values, regret_values,
            cumulative_regret_values, regret_fraction_within_5_percent,
            t, mean_t, n_target_data,
        )

    if verbose_diagnostics:
        print(_summarize_imfbo_diags(diags, f"imfbo_{noise_model}_{acquisition}"))

    return (
        selected_values.cpu(),
        best_so_far.cpu(),
        rmse_values.cpu(),
        regret_values.cpu(),
        cumulative_regret_values.cpu(),
        regret_fraction_within_5_percent.cpu(),
        {"diags": diags},
    )


In [ ]:
def _summarize_rgpe_diags(diags, label):
    """
    Collapse the per-step rank-weight diagnostics from evaluate_rgpe_bo into one line.

    RGPE has no error path for a degenerate weight split, it silently reduces to
    whichever task wins the LOOCV/ranking-loss vote every step, so this is the only
    signal that the feedback task ever actually contributed.
    """
    if not diags:
        return f"    [{label}] no fit diagnostics"

    n = len(diags)
    feedback_weight = sum(d["feedback_weight"] for d in diags) / n
    target_weight = sum(d["target_weight"] for d in diags) / n
    errors = [e for d in diags for e in d["errors"]]

    msg = (
        f"    [{label}] mean rank weight: feedback={feedback_weight:.3f} target={target_weight:.3f} "
        f"(averaged over {n} steps x {diags[0]['B']} environments)"
    )
    if errors:
        uniq = list(dict.fromkeys(errors))
        msg += f"\n    [{label}] {len(errors)} error(s), e.g. {uniq[0][:160]}"
    return msg


def _slice_state_dict(state_dict, b, B):
    # state_dict of batched model -> env b's unbatched one
    # lets get_fitted_model skip refit
    # non-batched buffers pass through
    return {
        name: (t[b] if t.dim() > 0 and t.shape[0] == B else t)
        for name, t in state_dict.items()
        }


def evaluate_rgpe_bo(
    eval_set, T, n_target_data,
    acquisition="ucb",
    kappa=1.0,
    max_pool=400,
    num_posterior_samples=256,
    target_noise_var=1e-4,
    feedback_noise_var=1e-4,
    expert_feedback_adjustor=None,
    turn_into_probs=True,
    verbose_diagnostics=True,
    ):
    """
    GP-BO with an RGPE (Rank-Weighted GP Ensemble) surrogate: a target-task GP fit on
    the real (sparse) objective, ensembled with a base-task GP fit on the (dense,
    pool-wide) expert feedback, feedback plays the role of RGPE's "related prior
    task" (Feurer, Letham, Bakshy, ICML AutoML Workshop 2018).

    Ported from BoTorch's own RGPE tutorial (utils/rgpe.py: get_fitted_model,
    roll_col, compute_ranking_loss, get_target_model_loocv_sample_preds,
    compute_rank_weights, RGPE), left unmodified. Two adaptations to this codebase:
      - The target-task and base/feedback-task SingleTaskGP fits are batched across
        all B environments in one fit_gpytorch_mll call each, SingleTaskGP already
        supports an arbitrary leading batch dimension, and every environment always
        has the same context-point count at a given step (see _update_batch), so
        nothing blocks fitting all B at once. This matches the batching every other
        baseline here uses (BatchedGP, BatchedMultiTaskGP), instead of B separate
        fits. Only the LOOCV rank-weight computation (utils/rgpe.compute_rank_weights)
        stays per-environment: it does no fitting at all, get_target_model_loocv_
        sample_preds only ever loads a state_dict, never calls fit_gpytorch_mll, so
        it's cheap relative to the two fits above and isnt worth carrying an extra
        batch-of-environments dimension through utils/rgpe.py's ported LOOCV masking /
        cartesian-product code.
      - Candidate selection is pool-based (score batch.query_x, argmax, _step_update),
        matching evaluate_multitask_gp_bo, not the tutorial's own continuous
        optimize_acqf + qLogNoisyExpectedImprovement loop.

    target_noise_var / feedback_noise_var: fixed observation-noise variance for the
    two tasks' SingleTaskGP(train_Yvar=...), matching the tutorial's own fixed
    noise_std**2 (see RGPE_TARGET_NOISE_VAR / RGPE_FEEDBACK_NOISE_VAR below for the
    values this notebook actually wires in).
    """
    from utils import rgpe
    from utils.free_mode import gp_scores

    batch = deepcopy(eval_set)
    B = batch.context_x.shape[0]
    device = batch.context_x.device
    dtype = batch.context_x.dtype

    (
        best_y, true_max,
        selected_values, best_so_far,
        rmse_values, regret_values,
        cumulative_regret_values, regret_fraction_within_5_percent,
    ) = _init_tracking(batch, T)

    diags = []

    for t in tqdm(range(T), desc=f"RGPE-BO ({acquisition})"):
        qry_x = batch.query_x  # [B, n_q, d]
        tgt_x = batch.target_x[:, :n_target_data]  # [B, n_target, d]
        n_q = qry_x.shape[1]
        n_ctx = batch.context_x.shape[1]

        mean = torch.zeros(B, n_q, device=device, dtype=dtype)
        std = torch.ones(B, n_q, device=device, dtype=dtype)
        mean_t = torch.zeros(B, tgt_x.shape[1], device=device, dtype=dtype)
        step_diag = {"feedback_weight": 0.0, "target_weight": 0.0, "B": B, "errors": []}

        if n_ctx < 2:
            # LOOCV needs >=2 target pts
            # same n_ctx for all envs, skip whole step
            # zero-info fallback, as in multitask_posterior
            step_diag["errors"].append(f"skipped step: only {n_ctx} context point(s)")
        else:
            ctx_x = batch.context_x.double()
            ctx_y = batch.context_y.squeeze(-1).double()
            ctx_fb = batch.context_feedback[:, :, 0].double()
            qry_x_d = qry_x.double()
            qry_fb = batch.query_feedback[:, :, 0].double()
            tgt_x_d = tgt_x.double()

            all_x = torch.cat([ctx_x, qry_x_d], dim=1)  # [B, n_total, d]
            all_fb = torch.cat([ctx_fb, qry_fb], dim=1)  # [B, n_total]
            lo = all_x.min(dim=1, keepdim=True).values
            hi = all_x.max(dim=1, keepdim=True).values
            span = (hi - lo).clamp_min(1e-8)

            def _norm(x):
                return ((x - lo) / span).clamp(0.0, 1.0)

            try:
                target_x_train = _norm(ctx_x)
                target_y_train = ctx_y.unsqueeze(-1)
                target_yvar = torch.full_like(target_y_train, target_noise_var)
                target_model = rgpe.get_fitted_model(target_x_train, target_y_train, target_yvar)

                # base (feedback) task: ctx + pool feedback, subsampled to max_pool
                # like fit_multitask_gps, batched over B
                n_total = all_x.shape[1]
                if n_total > max_pool:
                    keep = torch.argsort(torch.rand(B, n_total, device=device), dim=1)[:, :max_pool]
                    fb_x_raw = torch.gather(all_x, 1, keep.unsqueeze(-1).expand(-1, -1, all_x.shape[-1]))
                    fb_y_raw = torch.gather(all_fb, 1, keep)
                else:
                    fb_x_raw, fb_y_raw = all_x, all_fb
                fb_x_train = _norm(fb_x_raw)
                fb_y_train = fb_y_raw.unsqueeze(-1)
                fb_yvar = torch.full_like(fb_y_train, feedback_noise_var)
                base_model = rgpe.get_fitted_model(fb_x_train, fb_y_train, fb_yvar)

                # per-env rank weights, cheap, so python loop
                target_state = target_model.state_dict()
                base_state = base_model.state_dict()
                weights = torch.full((B, 2), 0.5, device=device, dtype=torch.double)

                for b in range(B):
                    try:
                        target_model_b = rgpe.get_fitted_model(
                            target_x_train[b], target_y_train[b], target_yvar[b],
                            state_dict=_slice_state_dict(target_state, b, B),
                            )
                        base_model_b = rgpe.get_fitted_model(
                            fb_x_train[b], fb_y_train[b], fb_yvar[b],
                            state_dict=_slice_state_dict(base_state, b, B),
                            )
                        weights[b] = rgpe.compute_rank_weights(
                            target_x_train[b], target_y_train[b], target_yvar[b],
                            [base_model_b], target_model_b, num_posterior_samples,
                            )
                    except Exception as exc:
                        # weights[b] stays 0.5/0.5
                        step_diag["errors"].append(f"env {b}: {type(exc).__name__}: {exc}")
                    step_diag["feedback_weight"] += weights[b, 0].item() / B
                    step_diag["target_weight"] += weights[b, 1].item() / B

                with torch.no_grad():
                    post_q_fb = base_model.posterior(_norm(qry_x_d))
                    post_q_tg = target_model.posterior(_norm(qry_x_d))
                    post_t_fb = base_model.posterior(_norm(tgt_x_d))
                    post_t_tg = target_model.posterior(_norm(tgt_x_d))

                    w_fb = weights[:, 0].view(B, 1)
                    w_tg = weights[:, 1].view(B, 1)

                    mean_q = w_fb * post_q_fb.mean.squeeze(-1) + w_tg * post_q_tg.mean.squeeze(-1)
                    var_q = (
                        w_fb**2 * post_q_fb.variance.clamp_min(1e-8).squeeze(-1)
                        + w_tg**2 * post_q_tg.variance.clamp_min(1e-8).squeeze(-1)
                        )
                    mean_tt = w_fb * post_t_fb.mean.squeeze(-1) + w_tg * post_t_tg.mean.squeeze(-1)

                    mean = mean_q.to(dtype)
                    std = var_q.clamp_min(1e-8).sqrt().to(dtype)
                    mean_t = mean_tt.to(dtype)
            except Exception as exc:
                step_diag["errors"].append(f"batched fit failed: {type(exc).__name__}: {exc}")

        mean = torch.nan_to_num(mean, nan=0.0)
        std = torch.nan_to_num(std, nan=1.0).clamp_min(1e-8)
        mean_t = torch.nan_to_num(mean_t, nan=0.0)
        diags.append(step_diag)

        if acquisition == "random":
            scores = torch.rand_like(mean)
        else:
            scores = gp_scores(acquisition, mean, std, best_f=best_y, kappa=kappa)

        if expert_feedback_adjustor is not None:
            feedback = batch.query_feedback[:, :, 0]
            if turn_into_probs:
                feedback = torch.softmax(feedback, dim=1)
            scores = expert_feedback_adjustor(scores, feedback, n=t)

        idx = scores.argmax(dim=1, keepdim=True)

        batch, best_y = _step_update(
            batch, idx, best_y, true_max,
            selected_values, best_so_far,
            rmse_values, regret_values,
            cumulative_regret_values, regret_fraction_within_5_percent,
            t, mean_t, n_target_data,
        )

    if verbose_diagnostics:
        print(_summarize_rgpe_diags(diags, f"rgpe_{acquisition}"))

    return (
        selected_values.cpu(),
        best_so_far.cpu(),
        rmse_values.cpu(),
        regret_values.cpu(),
        cumulative_regret_values.cpu(),
        regret_fraction_within_5_percent.cpu(),
        {"diags": diags},
    )


In [ ]:
def evaluate_multi_batch(
    n_batches: int,
    batch_size: int,
    experiment,
    T: int,
    method_configs: List[MethodConfig],
    verbose: bool = True,
    ) -> dict:
    """
    Evaluate every method over n_batches independently sampled batches.
 
    All methods share the same eval_set within each batch iteration so
    comparisons are always on identical problem instances.  Different
    batch iterations are independent, averaging out easy/hard correlation.
 
    Returns
    -------
    dict  name -> (selected_values, best_so_far, rmse_values, regret, cum_regret)
          Each tensor shape: (T, n_batches * batch_size)
 
    Same structure as the `data` dict you already build, so all existing
    plotting code works without changes.
    """
    acc = {mc.name: [] for mc in method_configs}
 
    for b in range(n_batches):
        if verbose:
            print(f"  Batch {b + 1}/{n_batches} ...", end=" ", flush=True)
        t0 = time.time()
 
        # one fresh sample shared by every method this round
        eval_set = experiment.sample_batch(batch_size)
 
        for mc in method_configs:
            batch = copy.deepcopy(eval_set)
 
            if mc.experiment_dim_x_delta:
                experiment.dim_x += mc.experiment_dim_x_delta
 
            out = evaluate_policy_vs_bo(
                batch, mc.cfg, mc.model, experiment, T,
                acquisition=mc.acquisition,
                attend_to=mc.attend_to,
                n_theta=mc.effective_n_theta(),
                concat_feedback=mc.concat_feedback,
                use_gp=mc.use_gp,
                gp_model=mc.gp_model,
                expert_feedback_adjustor=mc.expert_feedback_adjustor,
                advanced_stats=mc.advanced_stats,
            )
 
            if mc.experiment_dim_x_delta:
                experiment.dim_x -= mc.experiment_dim_x_delta
 
            acc[mc.name].append(out[:5])
 
        if verbose:
            print(f"done ({time.time() - t0:.1f}s)")
 
    # concatenate along batch dim -> (T, n_batches * batch_size)
    aggregated = {
        name: tuple(
            torch.cat([run[i] for run in runs], dim=1)
            for i in range(5)
        )
        for name, runs in acc.items()
    }
 
    if verbose:
        print(f"\nDone. Each metric tensor: (T={T}, n_samples={n_batches * batch_size})")
 
    return aggregated
 

In [ ]:
def _run_method(
    mc: MethodConfig,
    bench_sample,
    experiment,
    T: int,
    n_target_data: int,
    evaluate_policy_vs_bo: Callable,
    evaluate_gp_bo: Callable,
    ) -> tuple:

    if mc.experiment_dim_x_delta:
        experiment.dim_x += mc.experiment_dim_x_delta

    def _gp_sample():
        # feedback as extra input feature: plain GP over [x, feedback]
        sample = copy.deepcopy(bench_sample)
        return concat_feedback_to_batch_x(sample) if mc.concat_feedback else sample

    try:
        if getattr(mc, "botorch_continuous", False):
            # botorch-native continuous: SingleTaskGP + optimize_acqf, no pool
            from utils import free_mode
            out = free_mode.evaluate_botorch_gp_continuous(
                copy.deepcopy(bench_sample), experiment, T,
                acquisition=mc.acquisition,
                expert_feedback_adjustor=mc.expert_feedback_adjustor,
                feedback_integration=getattr(mc, "feedback_integration", "gradient"),
                reference_optimum=getattr(mc, "reference_optimum", None),
            )
        elif getattr(mc, "continuous", False):
            # gradient-refine pool instead of argmax, see utils/free_mode.py
            # reference_optimum: same regret baseline as pool runs
            from utils import free_mode
            # only pass if set in MethodConfig, else evaluate_continuous defaults
            pool_kwargs = {
                k: v for k, v in (
                    ("pool_size", getattr(mc, "pool_size", None)),
                    ("n_refine_candidates", getattr(mc, "n_refine_candidates", None)),
                    ("score_chunk_size", getattr(mc, "score_chunk_size", None)),
                    ("n_ballast_candidates", getattr(mc, "n_ballast_candidates", None)),
                    ("candidate_mode", getattr(mc, "candidate_mode", None)),
                    ("n_candidates", getattr(mc, "n_candidates", None)),
                    ("n_background", getattr(mc, "n_background", None)),
                    ("refine_optimizer", getattr(mc, "refine_optimizer", None)),
                    ("refine_lr", getattr(mc, "refine_lr", None)),
                    ("adam_step_multiplier", getattr(mc, "adam_step_multiplier", None)),
                    ("normalize_box", getattr(mc, "normalize_box", None)),
                    ("log_ei_mode", getattr(mc, "log_ei_mode", None)),
                    )
                if v is not None
                }
            out = free_mode.evaluate_continuous(
                copy.deepcopy(bench_sample), experiment, T,
                model=mc.model, acquisition=mc.acquisition,
                n_theta=mc.effective_n_theta(), concat_feedback=mc.concat_feedback,
                feedback_mode=getattr(mc, "feedback_mode", "frozen"),
                use_gp=mc.use_gp, n_iter_gp=mc.n_iter_gp,
                expert_feedback_adjustor=mc.expert_feedback_adjustor,
                reference_optimum=getattr(mc, "reference_optimum", None),
                **pool_kwargs,
            )
        elif getattr(mc, "use_multitask_gp", False):
            # feedback as 2nd GP task (ICM), see utils/multitask_gp.py
            out = evaluate_multitask_gp_bo(
                copy.deepcopy(bench_sample), T=T, n_target_data=n_target_data,
                acquisition=mc.acquisition, mtgp_kind=getattr(mc, "mtgp_kind", "icm"),
                kappa=mc.kappa,
                n_iter=mc.n_iter_gp,
                expert_feedback_adjustor=mc.expert_feedback_adjustor,
            )
        elif getattr(mc, "use_imfbo", False):
            # feedback as extra obs of f, learned noise sigma_lf(x), iMFBO (Fan et al. 2024)
            imfbo_kwargs = {
                k: v for k, v in (("max_pool", getattr(mc, "imfbo_max_pool", None)),) if v is not None
                }
            out = evaluate_imfbo_bo(
                copy.deepcopy(bench_sample), T=T, n_target_data=n_target_data,
                acquisition=mc.acquisition, kappa=mc.kappa,
                noise_model=getattr(mc, "imfbo_noise_model", "parametric"),
                n_iter=mc.n_iter_gp,
                expert_feedback_adjustor=mc.expert_feedback_adjustor,
                **imfbo_kwargs,
            )
        elif getattr(mc, "use_rgpe", False):
            # feedback as RGPE base task
            rgpe_kwargs = {
                k: v for k, v in (
                    ("max_pool", getattr(mc, "rgpe_max_pool", None)),
                    ("num_posterior_samples", getattr(mc, "rgpe_num_posterior_samples", None)),
                    ("target_noise_var", getattr(mc, "rgpe_target_noise_var", None)),
                    ("feedback_noise_var", getattr(mc, "rgpe_feedback_noise_var", None)),
                    )
                if v is not None
                }
            out = evaluate_rgpe_bo(
                copy.deepcopy(bench_sample), T=T, n_target_data=n_target_data,
                acquisition=mc.acquisition, kappa=mc.kappa,
                expert_feedback_adjustor=mc.expert_feedback_adjustor,
                **rgpe_kwargs,
            )
        elif getattr(mc, "gp_native", False):
            # botorch-native pool-argmax GP, see free_mode.evaluate_gp_bo_botorch
            from utils import free_mode
            out = free_mode.evaluate_gp_bo_botorch(
                _gp_sample(), T=T, n_target_data=n_target_data,
                acquisition=mc.acquisition, kappa=mc.kappa,
                expert_feedback_adjustor=mc.expert_feedback_adjustor,
            )
        elif not mc.use_gp:
            out = evaluate_policy_vs_bo(
                copy.deepcopy(bench_sample), mc.cfg, mc.model, experiment, T,
                acquisition=mc.acquisition, attend_to=mc.attend_to,
                n_theta=mc.effective_n_theta(), concat_feedback=mc.concat_feedback,
                gp_model=mc.gp_model, expert_feedback_adjustor=mc.expert_feedback_adjustor,
                advanced_stats=mc.advanced_stats,
            )
        else:
            try:
                out = evaluate_gp_bo(
                    _gp_sample(), T=T, n_target_data=n_target_data,
                    acquisition=mc.acquisition, n_iter=mc.n_iter_gp, kappa=mc.kappa,
                    expert_feedback_adjustor=mc.expert_feedback_adjustor,
                )
            except NotPSDError:
                print(f"    [GP fallback 1] {mc.name}: retrying with increased jitter")
                try:
                    with gpytorch.settings.cholesky_jitter(1e-1):
                        out = evaluate_gp_bo(
                            _gp_sample(), T=T, n_target_data=n_target_data,
                            acquisition=mc.acquisition, n_iter=mc.n_iter_gp, kappa=mc.kappa,
                            expert_feedback_adjustor=mc.expert_feedback_adjustor,
                        )
                except NotPSDError:
                    print(f"    [GP fallback 2] {mc.name}: retrying with fixed kernel (n_iter=0)")
                    try:
                        out = evaluate_gp_bo(
                            _gp_sample(), T=T, n_target_data=n_target_data,
                            acquisition=mc.acquisition, n_iter=0, kappa=mc.kappa,
                            expert_feedback_adjustor=mc.expert_feedback_adjustor,
                        )
                    except NotPSDError:
                        print(f"    [GP FAILED] {mc.name}: all fallbacks exhausted, skipping.")
                        return None

    finally:
        if mc.experiment_dim_x_delta:
            experiment.dim_x -= mc.experiment_dim_x_delta

    selected, best, rmse, regret, regret_cum, top5, _ = out
    return (selected, best, rmse, regret, top5, regret_cum)


In [ ]:
def evaluate_multi_batch_Compatible(
    n_batches: int,
    batch_size: int,
    experiment,
    T: int,
    method_configs: List[MethodConfig],
    evaluate_policy_vs_bo: Callable,
    evaluate_gp_bo: Callable,
    verbose: bool = True,
    device=None,
    ) -> Dict[str, tuple]:
    """
    Evaluate every method over n_batches independently sampled batches.

    Returns
    -------
    dict[method_name] -> (selected, best_so_far, rmse, regret, top5, regret_cum)
    Each tensor shape: (T, n_batches * batch_size)

    Same tuple structure as run_benchmark_evaluation, so aggregate_results
    and all plotting code work without changes.
    """
    acc = {mc.name: [] for mc in method_configs}

    for b in range(n_batches):
        if verbose:
            print(f"  Batch {b + 1}/{n_batches} ...", end=" ", flush=True)
        t0 = time.time()

        eval_set = experiment.sample_batch(batch_size)
        if device is not None:
            eval_set = _batch_to_device(eval_set, device)
        for mc in method_configs:
            out = _run_method(
                mc, eval_set, experiment, T,
                n_target_data=1,  # single context point for multi-batch
                evaluate_policy_vs_bo=evaluate_policy_vs_bo,
                evaluate_gp_bo=evaluate_gp_bo,
            )
            if out is not None:
                acc[mc.name].append(tuple(t.cpu() for t in out))
                #acc[mc.name].append(out)

        if verbose:
            print(f"done ({time.time() - t0:.1f}s)")

    # concatenate along batch dim -> (T, n_batches * batch_size)
    aggregated = {
        name: tuple(
            torch.cat([run[i] for run in runs], dim=1)
            for i in range(6)
        )
        for name, runs in acc.items()
        if runs
    }

    if verbose:
        print(f"\nDone. Each metric tensor: (T={T}, n_samples={n_batches * batch_size})")

    return aggregated

In [ ]:
def run_benchmark_evaluation(
    benchmarks: List[str],
    method_configs: List[MethodConfig],
    experiment,
    evaluate_policy_vs_bo: Callable,
    evaluate_gp_bo: Callable,
    BenchmarkFeedbackTask,
    cfg,
    batch_size: int,
    T: int = 30,
    # BenchmarkFeedbackTask kwargs
    expert_dim: int = 2,
    n_marginal_samples: int = 1,
    design_scale: float = 5.0,
    noise_scale: float = 0.1,
    expert_bias_model=None,
    expert_generator=None,
    n_expert_extra_min: int = 50,
    n_expert_extra_max: int = 50,
    expert_min_centers: int = 5,
    expert_max_centers: int = 5,
    expert_cluster_std_min: float = 0.5,
    expert_cluster_std_max: float = 1.5,
    uniform_fraction: float = 1,
    n_optimum_points: int = 1,
    optimum_jitter: float = 0.0,
    fake_peaks: bool = False,
    fake_peak_k: int = 3,
    fake_peak_height_mult: float = 1.5,
    fake_peak_decay_range: tuple = (0.5, 2.0),
    device=None,
    seed_fn: Optional[Callable] = None,
    verbose: int = 1,
    n_query_init: int =300,
    regret_reference: str = "pool",
    ) -> Dict[str, Dict[str, tuple]]:
    """
    Run all method_configs on all benchmarks.
 
    Returns
    -------
    results : dict[bench_name][method_name] = (
        selected_values [T, B],
        best_so_far     [T, B],
        rmse_values     [T, B],
        regret_values   [T, B],
        top_5_percent   [T, B],
        regret_cum      [T, B],
    )
 
    Example
    -------
    baseline_template = MethodConfig(
        name="_", model=model_for_baselines, cfg=cfg,
        n_theta=n_theta_baselines, attend_to="data",
        expert_feedback_adjustor=partial(pi_bo, beta=2),
    )
    method_configs = [
        MethodConfig("ficbo_combined", model, cfg,
                     acquisition="aae", n_theta=cfg.task.n_target_theta),
        MethodConfig("gp_icbo", expert_model_not_using_feedback, cfg,
                     acquisition="aae",
                     n_theta=cfg_no_fb.task.n_target_theta),
        MethodConfig("gp_icbo_fb", expert_model_using_feedback, cfg,
                     acquisition="aae", concat_feedback=True,
                     n_theta=cfg_fb.task.n_target_theta,
                     experiment_dim_x_delta=cfg.task.dim_feedback),
        *baseline_configs(baseline_template,
                          acquisitions=("random", "feedback", "ucb", "ei")),
    ]
    results = run_benchmark_evaluation(
        benchmarks=["sphere3dn", "ackley3dn"],
        method_configs=method_configs,
        experiment=experiment,
        ...
    )
    """
    results = {}
    optima = {}
    for bench in benchmarks:
        if verbose:
            print(f"\n{'='*60}\nBenchmark: {bench}\n{'='*60}")
 
        if seed_fn is not None:
            seed_fn()
 
        benchmark_kwargs = dict(
            benchmark_name=bench,
            dim_x=cfg.task.dim_x,
            dim_y=cfg.task.dim_y,
            n_context_init=1,
            n_query_init=n_query_init,
            n_target_data=cfg.task.n_target_data if bench not in ["nasbench201", "airfoil_lift_drag", "airfoil_nn",] else 1,
            n_target_theta=cfg.task.n_target_theta,
            design_scale=design_scale,
            noise_scale=noise_scale,
            expert_dim=expert_dim,
            n_marginal_samples=n_marginal_samples,
            concat_bias_generated_certainty=False,
            embedding_type="mix",
            normalize_y=True,
            expert_bias_model=expert_bias_model,
            expert_generator=expert_generator,
            n_expert_extra_min=n_expert_extra_min,
            n_expert_extra_max=n_expert_extra_max,
            expert_min_centers=expert_min_centers,
            expert_max_centers=expert_max_centers,
            expert_cluster_std_min=expert_cluster_std_min,
            expert_cluster_std_max=expert_cluster_std_max,
            uniform_fraction=uniform_fraction,
            n_optimum_points=n_optimum_points,
            optimum_jitter=optimum_jitter,
            fake_peaks=fake_peaks,
            fake_peak_k=fake_peak_k,
            fake_peak_height_mult=fake_peak_height_mult,
            fake_peak_decay_range=fake_peak_decay_range,
        )
        if device is not None:
            benchmark_kwargs["device"] = device
 
        benchmark = BenchmarkFeedbackTask(**benchmark_kwargs)
        bench_sample = benchmark.sample_batch(batch_size=batch_size, mode="mix")
        if verbose >1:
            print_batch_stats(bench_sample, name=bench)
        results[bench] = {}
        true_max = torch.cat([bench_sample.context_y, bench_sample.query_y], dim=1).max(dim=1).values.squeeze(-1).cpu()
        optima[bench] = true_max
 
        for mc in method_configs:
            t0 = time.time()
            if getattr(mc, "continuous", False):
                mc.reference_optimum = true_max
            res = _run_method(
                mc, bench_sample, benchmark, T,
                cfg.task.n_target_data,
                evaluate_policy_vs_bo, evaluate_gp_bo,
            )
            results[bench][mc.name] = res
            if verbose:
                status = "FAILED" if res is None else f"{time.time() - t0:.1f}s"
                print(f"  {mc.name}: {status}")

        # regret_reference="pool": regret vs each batch elem's pool max
        #   pool methods can't beat it
        # regret_reference="global": pool + continuous mixed
        #   ref = max(polished optimum estimate, best value any method found)
        #   recompute regret/cum_regret/top5 for all, overwrite results[bench]
        if regret_reference == "global":
            per_method = {name: res for name, res in results[bench].items() if res is not None}
            if per_method:
                from utils.free_mode import estimate_reference_optimum

                empirical_best = max(res[1][-1].max().item() for res in per_method.values())
                polished = estimate_reference_optimum(
                    benchmark, n_samples=20000, num_restarts=20, n_polish_steps=150,
                )
                reference = max(polished, empirical_best)
                reference_t = torch.tensor(reference)

                for name, (selected, best, rmse, regret, top5, regret_cum) in per_method.items():
                    regret_best = reference_t - best[1:]
                    regret_r = reference_t - selected[1:]
                    cum_r = torch.cumsum(regret_best, dim=0)
                    top5_r = (regret_best <= 0.05 * reference_t.abs().clamp_min(1e-8)).float()
                    results[bench][name] = (selected, best, rmse, regret_r, top5_r, cum_r)

                optima[bench] = torch.full_like(true_max, reference)

                if verbose:
                    print(f"\n  Reference optimum for {bench}: polished={polished:.4f} "
                          f"empirical_best={empirical_best:.4f} -> reference={reference:.4f}")
                    print(f"  Final cumulative regret ranking (lower = better; guaranteed "
                          f"non-negative by construction):")
                    finals = {
                        name: metrics[5][-1].mean().item()
                        for name, metrics in results[bench].items()
                        if metrics is not None
                    }
                    for rank, (name, val) in enumerate(
                        sorted(finals.items(), key=lambda x: x[1]), start=1
                    ):
                        print(f"    {rank}. {name:<35} {val:.4f}")
                    assert all(v >= -1e-4 for v in finals.values()), \
                        f"negative regret in {bench}: {finals}"
        elif verbose:
            print(f"\n  Final cumulative regret ranking (lower = better):")
            finals = {
                name: (
                    metrics[5][-1].mean().item()
                    if torch.is_tensor(metrics[5])
                    else float(metrics[5][-1].mean())
                )
                for name, metrics in results[bench].items()
            }
            for rank, (name, val) in enumerate(
                sorted(finals.items(), key=lambda x: x[1]), start=1
            ):
                print(f"    {rank}. {name:<35} {val:.4f}")

    return results, optima


def aggregate_results(results: dict, metric: str = "best_so_far") -> dict:
    """
    Aggregate results across benchmarks.
 
    metric: one of "selected", "best_so_far", "rmse", "regret", "top5", "regret_cum"
    Returns: dict[method] -> tensor [n_benchmarks, T, batch_size]
    """
    metric_idx = {
        "selected": 0,
        "best_so_far": 1,
        "rmse": 2,
        "regret": 3,
        "top5": 4,
        "regret_cum": 5,
    }
    idx = metric_idx[metric]
 
    all_methods = set(m for bench in results for m in results[bench])
    return {
        method: torch.stack([
            results[bench][method][idx]
            for bench in results
            if method in results[bench]
        ], dim=0)
        for method in all_methods
    }


# Load models and task


In [ ]:
# base model, also the amortized expert of the task
path = "outputs/hydra/general_gp_prior/2026-06-11/12-33-30/"  # TEMP path
expert_cfg, expert_model = load_config_and_model(path=path, file_name="model/aline_al_1to3d.pth")  # TEMP path
expert_model = expert_model.to(device)


In [ ]:
# FICBO (Combined)
path = "outputs/hydra/mixed_prior/2026-07-24/14-06-33/"  # TEMP path
cfg, model = load_config_and_model(path=path, file_name="model/aline_Human_GP_mixed.pth")  # TEMP path
model = model.to(device)


In [ ]:
# FICBO (Mixture Prior)
path = "outputs/hydra/ficbo_prior/2026-07-08/12-32-56/"  # TEMP path
cfg_reweight, model_reweight = load_config_and_model(path=path, file_name="model/aline_Human_GP_simple.pth")  # TEMP path
model_reweight = model_reweight.to(device)

# FICBO (Additive Prior)
path = "outputs/hydra/ficbo_prior/2026-06-18/16-35-44/"  # TEMP path
cfg_Amortized, model_Amortized = load_config_and_model(path=path, file_name="model/aline_Human_GP_simple.pth")  # TEMP path
model_Amortized = model_Amortized.to(device)


In [ ]:
class AveragedPolicyModel(nn.Module):
    """Averages the softmax policy distributions of two FICBO models."""

    def __init__(self, model_a: nn.Module, model_b: nn.Module) -> None:
        super().__init__()
        self.model_a = model_a
        self.model_b = model_b

    def forward(self, batch: AttrDict) -> AttrDict:
        outs_a = self.model_a.forward(batch)
        outs_b = self.model_b.forward(batch)

        zt_avg = 0.5 * (outs_a.design_out.zt + outs_b.design_out.zt)
        log_prob, idx_next = torch.max(torch.log(zt_avg.clamp_min(1e-30)), -1)
        idx_next = idx_next.unsqueeze(1)

        return AttrDict(
            posterior_out_query=outs_a.posterior_out_query,
            posterior_out=outs_a.posterior_out,
            design_out=AttrDict(
                idx=idx_next,
                log_prob=log_prob,
                zt=zt_avg,
                logits=torch.log(zt_avg.clamp_min(1e-30)),
            ),
        )


# FICBO (Ensemble)
model_combined = AveragedPolicyModel(model_reweight, model_Amortized)


In [ ]:
# GP-prior ICBO
cfg_expert_model_not_using_feedback, expert_model_not_using_feedback = load_config_and_model(
    "outputs/hydra/general_gp_prior/2026-06-11/11-26-52/",  # TEMP path
    file_name="model/aline_al_1to3d.pth",  # TEMP path
)
expert_model_not_using_feedback = expert_model_not_using_feedback.to(device)

print(f"{expert_model_not_using_feedback.embedder.embedding_type=}")


In [ ]:
# GP-prior ICBO + feedback feature
cfg_expert_model_using_feedback, expert_model_using_feedback = load_config_and_model(
    "outputs/hydra/general_gp_prior/2026-06-11/11-04-31/",  # TEMP path
    file_name="model/aline_al_1to3d.pth",  # TEMP path
)
expert_model_using_feedback = expert_model_using_feedback.to(device)


In [ ]:
folder_grp = "results"
if 1:
    os.makedirs(f"figures/{folder_grp}", exist_ok=True)


In [ ]:
do_expert_task = False  # do the experts task setup instead
model.embedder.embedding_type = embedding_mode_all
expert_model.embedder.embedding_type = embedding_mode_all
cfg.task.embedding_type = embedding_mode_all
cfg.task.mask_type = ["none"]  #none, all, split
if getattr(cfg, "expert_bias", None) is not None:
    if getattr(cfg.expert_bias, "uncertainty_noise_std", None) is not None:
        with open_dict(cfg.expert_bias):
            cfg.expert_bias.uncertainty_noise_std_range = (
                cfg.expert_bias.uncertainty_noise_std,
                cfg.expert_bias.uncertainty_noise_std,
            )
            del cfg.expert_bias.uncertainty_noise_std
    if getattr(cfg.expert_bias, "rounding_decimals", None) is not None:
        with open_dict(cfg.expert_bias):
            cfg.expert_bias.rounding_scale = cfg.expert_bias.rounding_decimals
            del cfg.expert_bias.rounding_decimals

    expert_bias_model = hydra.utils.instantiate(cfg.expert_bias, device=cfg.device)
    print(f"{expert_bias_model.device=}")

else:
    expert_bias_model = None
    print("no expert bias model")


## Optional: force noise levels or biases


In [ ]:
for _ in range(0):  # set to 1 to apply
    # here we can fix biases to check how the model behaves with them..
    print("entered")
    #expert_bias_model.bias_probs.perceived_max_bias=1
    #cfg.task.expert_extra_data_min = cfg.task.expert_extra_data_max = 30
    #break# just to set an easy stop
    #cfg.task.expert_weight_min = cfg.task.expert_weight_max = 0.8
    cfg.task.expert_noise = 0# output noise
    if getattr(cfg, "expert_bias", None) is not None:
        if cfg.expert_bias._target_ == "expert_models.bias_models.ProbExpertBiasEngine":
            expert_bias_model.bias_probs.hetero_noise= 0.#.2
            expert_bias_model.bias_probs.local_averaging= 0.#0.2
            expert_bias_model.bias_probs.add_gp_bias=1.  #.2
            expert_bias_model.bias_probs.perceived_max_bias=0#.1
            expert_bias_model.bias_probs.knowledge_center= 0.  #.2
            expert_bias_model.bias_probs.local_distortion= 1.#.3
            expert_bias_model.bias_probs.catastrophic= 1.#0.04
        else:
            print(f"No switch: {cfg.expert_bias=}")
        expert_bias_model.add_gp_bias=False  #true
        expert_bias_model.catastrophic=False
        expert_bias_model.local_distortion=False
        expert_bias_model.constant_shift_bias =True

        expert_bias_model.perceived_max_bias=False  # false        
        expert_bias_model.local_averaging=False
        expert_bias_model.knowledge_center=False
        expert_bias_model.hetero_noise=False
        
        
        expert_bias_model.noise_level = (0.0, 0.2)#(.3, .3)#.1-.7

        # strong
        expert_bias_model.distortion_range = (.7,.7)  #(.3,.7)
        expert_bias_model.distortion_centers = (5,6)  #(1,5)
        expert_bias_model.distortion_decay = (1.,1.)  #(.5,1.5)        

        # medium
        #expert_bias_model.distortion_range = (.5,.5) #(.3,.7)
        #expert_bias_model.distortion_centers = (3,4) #(1,5)
        #expert_bias_model.distortion_decay = (1.,1.) #(.5,1.5)        

        # weak
        #expert_bias_model.distortion_range = (.3,.3) #(.3,.7)
        #expert_bias_model.distortion_centers = (1,2) #(1,5)
        #expert_bias_model.distortion_decay = (1.,1.) #(.5,1.5)        

        #break
        expert_bias_model.hetero_alpha = (0.2, .2)
        expert_bias_model.hetero_n_centers = (4, 4)
        expert_bias_model.hetero_decay = (.5,.5)#(0.5, 2)
        
        #expert_bias_model.add_gp_max_std = (1., 1.)#.5-1, strong
        expert_bias_model.add_gp_max_std = (-.3, -.3)#.5-1
        expert_bias_model.add_gp_lengthscale = (.5, .5)# .5-3
        expert_bias_model.add_gp_scale = (.5, .5)  # [0.2, 2.]
        
        expert_bias_model.constant_shift_bias_range= (.3,.3)
        
        expert_bias_model.perceived_max_alpha = (.7, .7)

        expert_bias_model.knowledge_center_noise_radiation_scale= (.3, .3)#[0.1, 0.5]


        print("Adjustment completed")
    

In [ ]:
last_bool= True
if expert_bias_model is not None:
    for attr, value in expert_bias_model.__dict__.items():
        if type(value) is bool:
            last_bool = value
            print(f"{attr} = {value}")
        elif last_bool:
            print(f"{attr} = {value}")

In [ ]:
expert_model_type = "none"  # "sklearn" for the XGBoost expert, "amortized" not in paper
if expert_model_type == "amortized":
    expert_generator = partial(AmortizedExpert, amortized_model=expert_model, embedding_type=expert_model.embedder.embedding_type, 
                        expert_noise=0, normalize_outputs_to_max_one=False,
                        return_uncertainty=False, predict_epistemic_var=False,
                        adaptive=False, mode="mean",  #policy, mean
                        kappa=1.)  
elif expert_model_type == "sklearn":
    sklearn_model = build_sklearn_model(cfg.expert_model.sklearn_model)
    expert_generator = partial(
        ModelExpert,
        model = SklearnExpertModel(sklearn_model),
        fit_on = ["expert"],
        n_jobs = -1,
        adaptive = False,
        normalize_outputs_to_max_one = False,
        mode = "mean",
        #kappa    = 1.,
    )
else:
    expert_generator=None
print(f"{expert_generator=}")


In [ ]:
if not do_expert_task:
    cfg.task.n_query_init = 400
    experiment = hydra.utils.instantiate(cfg.task, expert_model=expert_model, expert_bias_model=expert_bias_model,
                                        model_expert_cfg=cfg.get("expert_model", None), device=device)
else:
    cfg = expert_cfg
    cfg.task.n_query_init = 400
    experiment = hydra.utils.instantiate(cfg.task, expert_bias_model=expert_bias_model,
                                        model_expert_cfg=cfg.get("expert_model", None), device=device)
    model=expert_model
    
print(f"{experiment.device=}")
print(cfg.task.expert_model_type)
eval_set = experiment.sample_batch(batch_size)
eval_set.keys()

In [ ]:
if shuffled_text_batch:
    eval_set = create_context_shuffled_batch(eval_set, batch_size)

# Single-benchmark check

Not in the paper. One batch from the training task, a few methods, quick plots.


In [ ]:
batch = copy.deepcopy(eval_set)
#expert_feedback_adjustor= partial(pi_bo, beta=.5)
expert_feedback_adjustor=None

selected_values_us, best_so_far_us, rmse_values_us, regret_us, cum_regret_us, top_5_percent_us, expert_feedback_stats = evaluate_policy_vs_bo(batch, cfg, model, experiment, T, 
                acquisition="aae", attend_to="data", n_theta=cfg.task.n_target_theta,
                #, always_T_is_1=True# doesnt change much
                expert_feedback_adjustor=expert_feedback_adjustor,
                advanced_stats=True
        )


## GP-prior ICBO


In [ ]:
batch = copy.deepcopy(eval_set)
selected_values_aline, best_so_far_aline, rmse_values_aline, regret_aline, cum_regret_aline, top_5_percent_aline, expert_feedback_stats_used = evaluate_policy_vs_bo(batch, 
                cfg, expert_model_not_using_feedback, experiment, T, 
                n_theta=cfg_expert_model_not_using_feedback.task.n_target_theta, 
                acquisition="aae", attend_to="data",
                #, always_T_is_1=True# doesnt change much
                concat_feedback=False, advanced_stats=False,
        )


## GP-prior ICBO + feedback feature


In [ ]:
batch = copy.deepcopy(eval_set)
#experiment_copy = copy.deepcopy(experiment) # breaks for xgboost
experiment_copy = (experiment)
experiment_copy.dim_x += cfg.task.dim_feedback
selected_values_aline_using_feedback, best_so_far_aline_using_feedback, rmse_values_aline_using_feedback, regret_aline_using_feedback, cum_regret_aline_using_feedback, top_5_percent_aline_using_feedback, expert_feedback_stats_aline_using_feedback = evaluate_policy_vs_bo(batch, 
                cfg, expert_model_using_feedback, experiment_copy, T, 
                    n_theta=cfg_expert_model_using_feedback.task.n_target_theta, 
                acquisition="aae", attend_to="data",
                #, always_T_is_1=True# doesnt change much
                concat_feedback=True, advanced_stats=False,
        )
experiment_copy.dim_x -= cfg.task.dim_feedback


## UCB / EI with πBO


In [ ]:

baselines_with_expert_data= False

use_real_gp= False
kernel = C(1.0) * RBF(length_scale=1.0)
test_gp_model = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)

expert_feedback_adjustor= partial(pi_bo, beta=2.0)


t = time.time()
if baselines_with_expert_data:
    model_for_baselines = expert_model_using_feedback
    n_theta_baselines = cfg_expert_model_using_feedback.task.n_target_theta
    experiment_for_baselines=experiment_copy
else:
    model_for_baselines = expert_model_not_using_feedback
    n_theta_baselines = cfg_expert_model_not_using_feedback.task.n_target_theta
    experiment_for_baselines= experiment

if False:# use the model trained to work using expert feedback
    model_for_baselines = model
    n_theta_baselines = cfg.task.n_target_theta
    experiment_for_baselines= experiment

batch = copy.deepcopy(eval_set)
selected_values_rnd, best_so_far_rnd, rmse_values_rnd, regret_rnd, cum_regret_rnd, top_5_percent_rnd, expert_feedback_stats_rnd = evaluate_policy_vs_bo(batch, cfg, model_for_baselines, experiment_for_baselines, T, acquisition="random", 
                                                    n_theta=n_theta_baselines, attend_to="data", 
                                                    concat_feedback=baselines_with_expert_data, advanced_stats=False,
                                                    use_gp=use_real_gp, gp_model=test_gp_model, expert_feedback_adjustor=expert_feedback_adjustor)
print("random:", time.time() - t)

t = time.time()

batch = copy.deepcopy(eval_set)
selected_values_ucb, best_so_far_ucb, rmse_values_ucb, regret_ucb, cum_regret_ucb, top_5_percent_ucb, expert_feedback_stats_ucb = evaluate_policy_vs_bo(batch, cfg, model_for_baselines, experiment_for_baselines, T, acquisition="ucb", 
                                                    attend_to="data", n_theta=n_theta_baselines, 
                                                    concat_feedback=baselines_with_expert_data, advanced_stats=False,
                                                    use_gp=use_real_gp, gp_model=test_gp_model, expert_feedback_adjustor=expert_feedback_adjustor)
print("ucb:", time.time() - t)

t = time.time()

batch = copy.deepcopy(eval_set)
selected_values_ei, best_so_far_ei, rmse_values_ei, regret_ei, cum_regret_ei, top_5_percent_ei, expert_feedback_stats_ei = evaluate_policy_vs_bo(batch, cfg, model_for_baselines, experiment_for_baselines, T, acquisition="ei", 
                                                    attend_to="data", n_theta=n_theta_baselines,
                                                    concat_feedback=baselines_with_expert_data, advanced_stats=False,
                                                    use_gp=use_real_gp, gp_model=test_gp_model, expert_feedback_adjustor=expert_feedback_adjustor)
print("ei:", time.time() - t)


## Plots


### Expert stats

Needs `advanced_stats=True` in the FICBO run above. Set `additional_vis=True` to plot.


In [ ]:
def recenter_feedback_logits(stats_dict):
    """
    Overwrite expert feedback in expert_feedback_stats with recentered inverse softmax logits.
    Assumes feedback_data shape [B, n_data, 1] and target_y shape [B, n_data, 1].
    """
    other = stats_dict["other"]

    if "target_feedback" not in other or other["target_feedback"] is None:
        return stats_dict  # nothing to do

    feedback = other["target_feedback"]  # [B, n_data, 1]
    true_y = other["target_y"]  # [B, n_data, 1]
    B, n_data, _ = feedback.shape

    # iterate over batch
    for b in range(B):
        fb = feedback[b].squeeze(-1)  # [n_data]
        y_true = true_y[b].squeeze(-1)  # [n_data]

        # inverse softmax and recenter
        fb_logits = torch.log(torch.clamp(fb, min=1e-12))  # inverse softmax
        fb_logits = fb_logits - fb_logits.mean()  # center around zero
        fb_logits = fb_logits + y_true.mean()  # shift to true mean

        feedback[b, :, 0] = fb_logits

    other["target_feedback"] = feedback
    stats_dict["other"] = other
    return stats_dict


if cfg.task.expert_mode == "policy":# undo the softmax
    expert_feedback_stats_target = recenter_feedback_logits(expert_feedback_stats)  #expert_feedback_stats_ei, expert_feedback_stats
else:
    expert_feedback_stats_target =expert_feedback_stats

additional_vis=False


In [ ]:
if additional_vis:
    data_pred = expert_feedback_stats["other"]["data_targets"]["pred_mean"]  # [T, B, n_data]
    theta_pred = expert_feedback_stats["other"]["thetas"]["pred_mean"]  # [T, B, n_theta]
    
    data_true = expert_feedback_stats["other"]["target_y"]  # [B, n_data, 1]
    data_true = data_true.squeeze(-1)  # [B, n_data]
    
    theta_true = expert_feedback_stats["other"]["thetas_true"].squeeze(-1)  # [B, n_theta]
    
    print(f"{theta_true.shape=}{data_true.shape=}{theta_pred.shape=}{data_pred.shape=}")
    
    T, B, n_data = data_pred.shape
    n_theta = theta_pred.shape[-1]
    
    timesteps = np.arange(T)
    
    data_diff_signed = torch.zeros(T)
    theta_diff_signed = torch.zeros(T)
    
    data_diff_rmse = torch.zeros(T)
    theta_diff_rmse = torch.zeros(T)
    
    for t in range(T):
        data_diff_signed[t] = (data_pred[t] - data_true).mean()
        theta_diff_signed[t] = (theta_pred[t] - theta_true).mean()
    
        data_diff_rmse[t] = torch.sqrt(((data_pred[t] - data_true) ** 2).mean())
        theta_diff_rmse[t] = torch.sqrt(((theta_pred[t] - theta_true) ** 2).mean())
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, data_diff_signed.numpy())
    plt.title("Signed difference: Predicted data targets - True data targets")
    plt.xlabel("Timestep")
    plt.ylabel("Difference")
    plt.show()
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, theta_diff_signed.numpy())
    plt.title("Signed difference: Predicted thetas - True thetas")
    plt.xlabel("Timestep")
    plt.ylabel("Difference")
    plt.show()
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, data_diff_rmse.numpy(), color='orange')
    plt.title("RMSE: Predicted data targets vs True data targets")
    plt.xlabel("Timestep")
    plt.ylabel("RMSE")
    plt.show()
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, theta_diff_rmse.numpy(), color='orange')
    plt.title("RMSE: Predicted thetas vs True thetas")
    plt.xlabel("Timestep")
    plt.ylabel("RMSE")
    plt.show()
    
    data_pred_mean = data_pred.mean(dim=(1, 2))  # avg over batch and targets
    theta_pred_mean = theta_pred.mean(dim=(1, 2))
    
    data_true_mean = data_true.mean()  # scalar
    theta_true_mean = theta_true.mean()  # scalar
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, data_pred_mean.numpy(), label="Predicted data")
    plt.axhline(data_true_mean.item(), color="r", linestyle="--", label="True data mean")
    plt.title("Average predicted data target over timesteps")
    plt.xlabel("Timestep")
    plt.ylabel("Value")
    plt.legend()
    plt.show()
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, theta_pred_mean.numpy(), label="Predicted theta")
    plt.axhline(theta_true_mean.item(), color="r", linestyle="--", label="True theta mean")
    plt.title("Average predicted theta over timesteps")
    plt.xlabel("Timestep")
    plt.ylabel("Value")
    plt.legend()
    plt.show()


In [ ]:

def plot_expert_metric(metric_stats, title, show_minmax=True, figsize=(10,4)):
    """
    Plots a metric per step. 
    - For metrics with 'variance', plots mean +- std and optionally min/max.
    - For metrics without 'variance' (binary/fraction), only plots the mean line.
    """
    if title == "other":
        return
    steps = list(range(1, len(metric_stats)+1))
    mean_vals = [s["mean"] for s in metric_stats]

    plt.figure(figsize=figsize)
    plt.plot(steps, mean_vals, marker='o', label='Mean', color='C0')

    # if variance exists, plot shaded mean -+ std
    if "variance" in metric_stats[0]:
        std_vals = [s["variance"]**0.5 for s in metric_stats]
        plt.fill_between(steps, 
                         [m - s for m,s in zip(mean_vals, std_vals)],
                         [m + s for m,s in zip(mean_vals, std_vals)],
                         color='C0', alpha=0.2, label='Mean +- Std')

        # optionally plot min/max as thin lines
        if show_minmax:
            min_vals = [s["min"] for s in metric_stats]
            max_vals = [s["max"] for s in metric_stats]
            plt.plot(steps, min_vals, linestyle='--', color='C1', alpha=0.6, label='Min')
            plt.plot(steps, max_vals, linestyle='--', color='C2', alpha=0.6, label='Max')

    plt.title(title)
    plt.xlabel('Step')
    plt.ylabel(title)
    #plt.grid(alpha=0.3)
    plt.legend()
    plt.show()


if additional_vis:
    for key, stats in expert_feedback_stats_target.items():
        print(f"{key=}")
        plot_expert_metric(stats, key)
        

In [ ]:
if additional_vis:
    # pred y: [T, B, n_data]
    pred_y = expert_feedback_stats_target["other"]["data_targets"]["pred_mean"]
    
    # expert feedback y: [B, n_data, 1]
    feedback_y = expert_feedback_stats_target["other"]["target_feedback"]  
    feedback_y = feedback_y.squeeze(-1)  # [B, n_data]
    
    T, B, n_data = pred_y.shape
    timesteps = np.arange(T)
    
    diff_feedback = torch.zeros(T)
    for t in range(T):
        diff_feedback[t] = (pred_y[t] - feedback_y).mean()
    
    diff_feedback_rmse = torch.zeros(T)
    for t in range(T):
        diff_feedback_rmse[t] = torch.sqrt(((pred_y[t] - feedback_y) ** 2).mean())
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, diff_feedback.numpy())
    plt.title("Signed difference: Predicted y - Expert feedback y")
    plt.xlabel("Timestep")
    plt.ylabel("Difference")
    plt.show()
    
    plt.figure(figsize=(10,6))
    plt.plot(timesteps, diff_feedback_rmse.numpy(), color='orange')
    plt.title("RMSE: Predicted y vs Expert feedback y")
    plt.xlabel("Timestep")
    plt.ylabel("RMSE")
    plt.show()


In [ ]:
def plot_gmm_feedback_log_likelihood(ll, mask, single_run_idx=0, figsize_wide=(12, 4), figsize_dist=(18, 3)):
    """
    ll:   (T, batch, n_query) tensor of GMM log-likelihoods
    mask: (T, batch, n_query) bool tensor, True = real point

    Produces 3 plots:
      1. Mean log-likelihood -+ std over time (across batch)
      2. Single run: avg log-likelihood over valid points per step
      3. Single run: per-step distribution as overlaid histograms
    """
    T, B, Q = ll.shape
    steps = list(range(1, T + 1))

    # mean over valid queries per (t, b), then stats over batch
    ll_masked = ll.float()
    ll_masked[~mask] = float('nan')
    per_item_mean = torch.nanmean(ll_masked, dim=-1)  # (T, B)

    mean_over_batch = per_item_mean.nanmean(dim=-1).numpy()  # (T,)
    std_over_batch = per_item_mean.std(dim=-1).numpy()  # (T,)

    # mean +- std over time
    fig, ax = plt.subplots(figsize=figsize_wide)
    ax.plot(steps, mean_over_batch, marker='o', color='C0', label='Mean (over batch)')
    ax.fill_between(steps,
                    mean_over_batch - std_over_batch,
                    mean_over_batch + std_over_batch,
                    color='C0', alpha=0.2, label='+- Std')
    ax.set_title('GMM log-likelihood of expert feedback, mean over batch')
    ax.set_xlabel('Step'); ax.set_ylabel('Log-likelihood')
    #ax.grid(alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

    # one run, avg over valid queries per step
    run_ll = ll[:, single_run_idx, :]  # (T, Q)
    run_mask = mask[:, single_run_idx, :]
    run_ll_masked = run_ll.float()
    run_ll_masked[~run_mask] = float('nan')
    run_mean = torch.nanmean(run_ll_masked, dim=-1).numpy()  # (T,)

    fig, ax = plt.subplots(figsize=figsize_wide)
    ax.plot(steps, run_mean, marker='o', color='C1')
    ax.set_title(f'Single run (idx={single_run_idx}), avg log-likelihood per step')
    ax.set_xlabel('Step'); ax.set_ylabel('Avg log-likelihood')
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # one run, distribution per step
    fig, axes = plt.subplots(1, T, figsize=figsize_dist, sharey=True)
    if T == 1: axes = [axes]
    cmap = plt.cm.viridis

    # global x range across all steps
    all_valid = run_ll[run_mask].numpy()
    x_min, x_max = all_valid.min(), all_valid.max()
    x_pad = (x_max - x_min) * 0.05
    bins = np.linspace(x_min, x_max, 25)  # shared bins too, for fair density comparison

    for t, ax in enumerate(axes):
        vals = run_ll[t][run_mask[t]].numpy()
        color = cmap(t / max(T - 1, 1))
        ax.hist(vals, bins=bins, color=color, alpha=0.8, orientation='vertical', density=True)
        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_title(f't={t+1}', fontsize=9)
        ax.set_xlabel('log-lik', fontsize=8)
        if t == 0: ax.set_ylabel('density', fontsize=8)
        ax.grid(alpha=0.2)
    fig.suptitle(f'Distribution of GMM log-likelihood per step (run {single_run_idx})')
    plt.tight_layout(); plt.show()


if additional_vis:
    ll = expert_feedback_stats_target["other"]["gmm_feedback_log_likelihood"]  # (T, B, Q)
    mask = expert_feedback_stats_target["other"]["gmm_feedback_ll_mask"]  # (T, B, Q)
    
    plot_gmm_feedback_log_likelihood(ll, mask, single_run_idx=0)


In [ ]:
def plot_targets_and_thetas(stats_dict, idx=0, plot_lines=False):
    """
    Plot data targets per timestep and thetas over time for a single batch sample.
    The x-axis uses the first x-coordinate of each target point, sorted ascending.
    Args:
        stats_dict: expert_feedback_stats
        idx: batch index to plot
        plot_lines: bool, if True plot true y and expert feedback as lines instead of dots
    """
    other = stats_dict["other"]
    T = other["data_targets"]["pred_mean"].shape[0]  # [T, batch, n_data]

    pred_mean_data = other["data_targets"]["pred_mean"][:, idx, :]  # [T, n_data]
    pred_var_data = other["data_targets"]["pred_var"][:, idx, :]  # [T, n_data]

    x_data_raw = other["target_x"][idx, :, 0]  # [n_data]
    sort_idx = torch.argsort(x_data_raw)
    x_data = x_data_raw[sort_idx].numpy()

    true_data = other["target_y"][idx, :, 0][sort_idx]  # [n_data]
    pred_mean_data = pred_mean_data[:, sort_idx]  # [T, n_data]
    pred_var_data = pred_var_data[:, sort_idx]  # [T, n_data]

    feedback_data = None
    if "target_feedback" in other and other["target_feedback"] is not None:
        feedback_data = other["target_feedback"][idx, :, 0][sort_idx]  # [n_data]

    for t in range(T):
        plt.figure(figsize=(15, 6))

        if plot_lines:
            plt.plot(x_data, true_data.numpy(), 'k-', label="True y")
            if feedback_data is not None:
                plt.plot(x_data, feedback_data.numpy(), 'r--', label="Expert feedback")
        else:
            plt.plot(x_data, true_data.numpy(), 'k.', label="True y")
            if feedback_data is not None:
                plt.plot(x_data, feedback_data.numpy(), 'r.', label="Expert feedback")

        plt.plot(x_data, pred_mean_data[t].numpy(), 'b-', label="Predicted mean")
        plt.fill_between(
            x_data,
            (pred_mean_data[t] - torch.sqrt(pred_var_data[t])).numpy(),
            (pred_mean_data[t] + torch.sqrt(pred_var_data[t])).numpy(),
            color='b', alpha=0.2, label="Pred +- std"
        )

        plt.title(f"Data targets at timestep {t} (sample {idx})")
        plt.xlabel("x coordinate")
        plt.ylabel("y value")
        plt.legend()
        plt.show()

    pred_mean_theta = other["thetas"]["pred_mean"][:, idx, :]  # [T, n_theta]
    pred_var_theta = other["thetas"]["pred_var"][:, idx, :]  # [T, n_theta]
    true_theta = other["thetas_true"][idx, :, 0]  # [n_theta]

    n_theta = pred_mean_theta.shape[1]
    timesteps = np.arange(T)

    for i in range(n_theta):
        plt.figure(figsize=(10, 5))
        plt.plot(timesteps, [true_theta[i]] * T, 'k--', label="True theta")
        plt.plot(timesteps, pred_mean_theta[:, i].numpy(), 'b-', label="Predicted mean")
        plt.fill_between(
            timesteps,
            (pred_mean_theta[:, i] - torch.sqrt(pred_var_theta[:, i])).numpy(),
            (pred_mean_theta[:, i] + torch.sqrt(pred_var_theta[:, i])).numpy(),
            color='b', alpha=0.2, label="Pred +- std"
        )
        plt.title(f"Theta {i} over time (sample {idx})")
        plt.xlabel("Timestep")
        plt.ylabel("Theta value")
        plt.legend()
        plt.show()

In [ ]:
if additional_vis:
    plot_targets_and_thetas(expert_feedback_stats_target, idx=1, plot_lines=True)


### Results

One batch, one bias setting. Use the multi-benchmark section for real numbers.


In [ ]:
data = {
    "ficbo_combined": (selected_values_us, best_so_far_us, rmse_values_us, regret_us, top_5_percent_us, cum_regret_us),
    "gp_icbo": (selected_values_aline, best_so_far_aline, rmse_values_aline, regret_aline, top_5_percent_aline, cum_regret_aline),
    "gp_icbo_fb": (selected_values_aline_using_feedback, best_so_far_aline_using_feedback, rmse_values_aline_using_feedback, regret_aline_using_feedback, top_5_percent_aline_using_feedback, cum_regret_aline_using_feedback),
    "random": (selected_values_rnd, best_so_far_rnd, rmse_values_rnd, regret_rnd, top_5_percent_rnd, cum_regret_rnd),
    "ucb": (selected_values_ucb, best_so_far_ucb, rmse_values_ucb, regret_ucb, top_5_percent_ucb, cum_regret_ucb),
    "ei": (selected_values_ei, best_so_far_ei, rmse_values_ei, regret_ei, top_5_percent_ei, cum_regret_ei),
}


In [ ]:
idx = 1
metrics_data_list = [
    data["ficbo_combined"][idx],   
    data["gp_icbo"][idx],
    data["gp_icbo_fb"][idx],
    data["random"][idx],
    data["ucb"][idx],
    data["ei"][idx],
]

metric_names = ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature","Random", "UCB", "EI"]


fig, ax = plot_metrics_with_confidence(
    metrics_data_list,
    metric_names,
    title="Best-so-far across methods",
    ylabel="Best so far",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc= "lower right",
    nbins=5,
)

plt.show()


Best-so-far can match while the selected y differs: a method with lower selected y explores more.


In [ ]:
idx = 2
metrics_data_list = [
    data["ficbo_combined"][idx],
    data["gp_icbo"][idx],
    data["gp_icbo_fb"][idx],
    data["random"][idx],
    data["ucb"][idx],
    data["ei"][idx],
]

metric_names = ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature","Random", "UCB", "EI"]

fig, ax = plot_metrics_with_confidence(
    metrics_data_list,
    metric_names,
    title="MSE across methods",
    ylabel="MSE of selected point",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc= "upper right",

)

plt.show()


In [ ]:
idx = 3
metrics_data_list = [
    data["ficbo_combined"][idx],
    data["gp_icbo"][idx],
    data["gp_icbo_fb"][idx],
    data["random"][idx],
    data["ucb"][idx],
    data["ei"][idx],
]

metric_names = ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature","Random", "UCB", "EI"]

fig, ax = plot_metrics_with_confidence(
    metrics_data_list,
    metric_names,
    title="Instantaneous Regret across methods",
    ylabel="Instantaneous Regret  of selected point",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc= "upper right",

)

plt.show()


In [ ]:
idx = 4
metrics_data_list = [
    data["ficbo_combined"][idx],
    data["gp_icbo"][idx],
    data["gp_icbo_fb"][idx],
    data["random"][idx],
    data["ucb"][idx],
    data["ei"][idx],
]

metric_names = ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature","Random", "UCB", "EI"]

fig, ax = plot_metrics_with_confidence(
    metrics_data_list,
    metric_names,
    title="Within 5% across methods",
    ylabel="Within 5%  of selected point",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc= "upper right",

)

plt.show()


In [ ]:
idx = 5
until=-1  # drop last run, sanity check
metrics_data_list = [
    data["ficbo_combined"][idx][:,:until],
    data["gp_icbo"][idx][:,:until],
    data["gp_icbo_fb"][idx][:,:until],
    data["random"][idx][:,:until],
    data["ucb"][idx][:,:until],
    data["ei"][idx][:,:until],
]

metric_names = ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature","Random", "UCB", "EI"]

fig, ax = plot_metrics_with_confidence(
    metrics_data_list,
    metric_names,
    title="Cumulative Regret across methods",
    ylabel="Cumulative Regret of selected point",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc= "upper left",

)

plt.show()


In [ ]:
idx = 5
until=-1  # drop last run, sanity check
metrics_data_list = [
    data["ficbo_combined"][idx][:,:until],
    data["gp_icbo"][idx][:,:until],
    data["gp_icbo_fb"][idx][:,:until],
    data["random"][idx][:,:until],
    data["ucb"][idx][:,:until],
    data["ei"][idx][:,:until],
]

metric_names = ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature","Random", "UCB", "EI"]

fig, ax = plot_ranks_with_confidence(
    metrics_data_list,
    metric_names,
    title="Cumulative Regret across methods",
    ylabel="Cumulative Regret of selected point",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc= "upper left",
    low_is_better=True,
    nbins=5,
)
plt.show()


# Policy-feedback agreement

Top-5 overlap between the policy and the feedback at each step, for both FICBO priors on the real-world tasks.


In [ ]:
def get_benchmark_configs(device, expert_bias_model=None):
    return {
        "Airfoil Small": dict(
            cls=AirfoilNNFidelityTask,
            model_key="3d",
            kwargs=dict(
                n_context_init=1,
                n_query_init=300,
                n_target_data=100,
                n_target_theta=3,
                use_expert_thetas=True,
                normalize_y=True,
            ),
        ),
        "Airfoil Lift": dict(
            cls=AirfoilLiftDragTask,
            model_key="3d",
            kwargs=dict(
                n_context_init=1,
                n_query_init=300,
                n_target_data=100,
                n_target_theta=3,
                use_expert_thetas=True,
                normalize_y=True,
            ),
        ),
        "ML hyper-parameters": dict(  #"ML hyper-params"
            cls=NASBench201Task,
            model_key="3d",
            kwargs=dict(
                pth_path="data/NAS-Bench-201-v1_0-e61699.pth",
                n_context_init=1,
                n_query_init=123,
                n_target_data=1,
                n_target_theta=4,
                design_scale=5,
                noise_scale=0.1,
                normalize_y=True,
                device=device,
            ),
        ),
        "Reactor": dict(
            cls=BenchmarkFeedbackTask,
            model_key="2d",
            kwargs=dict(
                benchmark_name="reactor",
                dim_x=2,
                dim_y=1,
                n_context_init=1,
                n_query_init=300,
                n_target_data=100,
                n_target_theta=2,
                design_scale=5,
                noise_scale=0.01,
                expert_dims=2,
                n_marginal_samples=10,  # TEMP smoke, real: 100
                bias_model=expert_bias_model,
                use_expert_thetas=True,
                concat_bias_generated_certainty=False,
                embedding_type="mix",
                normalize_y=True,
            ),
        ),
    }


def build_benchmarks(benchmark_configs, keys=None):
    if keys is not None:
        benchmark_configs = {k: benchmark_configs[k] for k in keys}

    benchmarks = {}
    for label, spec in benchmark_configs.items():
        print(f"  Building [{label}] ...")
        cls, kwargs = spec["cls"], spec["kwargs"].copy()
        if hasattr(cls, "from_pth") and "pth_path" in kwargs:
            benchmarks[label] = cls.from_pth(kwargs.pop("pth_path"), **kwargs)
        else:
            benchmarks[label] = cls(**kwargs)
    return benchmarks


MODEL_PATHS = {#example paths, replace with your runs
    "Mixture Prior": {
        "2d": "outputs/hydra/ficbo_prior/2026-07-08/12-32-56/",  # TEMP path
        "3d": "outputs/hydra/ficbo_prior/2026-07-08/12-44-35/",  # TEMP path
    },
    "Additive Prior": {
        "2d": "outputs/hydra/ficbo_prior/2026-06-18/16-35-44/",  # TEMP path
        "3d": "outputs/hydra/ficbo_prior/2026-06-18/16-36-45/",  # TEMP path
    },
}


def load_model_registry(model_paths, device, file_name="model/aline_Human_GP_simple.pth"):  # TEMP path
    registry = {}
    for model_name, dim_paths in model_paths.items():
        registry[model_name] = {}
        for dim_key, path in dim_paths.items():
            print(f"  Loading [{model_name} / {dim_key}] from {path}")
            cfg, model = load_config_and_model(path=path, file_name=file_name)
            registry[model_name][dim_key] = (model.to(device), cfg)
    return registry


def collect_feedback_stats(
    model_registry,  # {"2d": (model, cfg), "3d": (model, cfg)}
    benchmarks,  # {label: benchmark instance}
    benchmark_configs,  # {label: spec dict with "model_key"}
    experiment,
    batch_size,
    T,
    acquisition="aae",
    attend_to="data",
    expert_feedback_adjustor=None,
    advanced_stats=True,
    ):
    results = {}
    for label, benchmark in benchmarks.items():
        dim_key = benchmark_configs[label]["model_key"]
        model, cfg = model_registry[dim_key]

        print(f"  Evaluating [{label}]  (model_key={dim_key!r})")
        batch = copy.deepcopy(benchmark.sample_batch(batch_size=batch_size, mode="mix"))

        _, _, _, _, _, _, stats = evaluate_policy_vs_bo(
            batch, cfg, model, experiment, T,
            acquisition=acquisition,
            attend_to=attend_to,
            n_theta=cfg.task.n_target_theta,
            expert_feedback_adjustor=expert_feedback_adjustor,
            advanced_stats=advanced_stats,
        )
        results[label] = stats
    return results



In [ ]:
def plot_agreement(
    results_by_model: dict[str, dict],
    key="acq_feedback_agreement",
    figsize=(9, 5),
    savepath="figures/Agreement_plot.pdf",
    ylim=(0, 1.1),
    max_xticks=5,
    linestyles=None,
    log_scale=False,
    ):
    if linestyles is None:
        linestyles = ["-", "--", "-.", ":"]
    all_labels = []
    for stats in results_by_model.values():
        for lbl in stats:
            if lbl not in all_labels:
                all_labels.append(lbl)
    cmap = plt.cm.tab10
    colour_map = {
        lbl: cmap(i / max(len(all_labels) - 1, 1))
        for i, lbl in enumerate(all_labels)
    }
    fig, ax = plt.subplots(figsize=figsize)

    for (model_name, stats), ls in zip(results_by_model.items(), linestyles):
        for bench_label, stat in stats.items():
            sim = stat[key]
            mean_s = sim.mean(dim=1)
            std_s = sim.std(dim=1) / sim.shape[1] ** 0.5
            steps = range(sim.shape[0])
            ax.plot(steps, mean_s, color=colour_map[bench_label], linestyle=ls)
            if log_scale:
                ax.fill_between(
                    steps,
                    (mean_s - std_s).clamp(-0.01, 1),
                    (mean_s + std_s).clamp(-0.01, 1),
                    color=colour_map[bench_label], alpha=0.15,
                )
            else:
                ax.fill_between(
                    steps,
                    (mean_s - std_s).clamp(0, 1),
                    (mean_s + std_s).clamp(0, 1),
                    color=colour_map[bench_label], alpha=0.15,
                )

    if log_scale:
        ax.set_yscale("symlog", linthresh=0.01)
        ax.set_ylim(-0.01, 1.1)
    else:
        ax.set_ylim(*ylim)

    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xlabel("iterations")
    ax.set_ylabel("top 5 agreement")
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=max_xticks))

    blank = Line2D([], [], color="none", label="")
    colour_handles = [
        Line2D([0], [0], color=colour_map[lbl], linewidth=2, label=lbl)
        for lbl in all_labels
    ]
    style_handles = [
        Line2D([0], [0], color="gray", linewidth=2, linestyle=ls, label=name)
        for (name, _), ls in zip(results_by_model.items(), linestyles)
    ]
    n_rows = max(len(colour_handles), len(style_handles))
    col1 = colour_handles + [blank] * (n_rows - len(colour_handles))
    col2 = style_handles + [blank] * (n_rows - len(style_handles))
    ax.legend(
        handles=col1 + col2,
        ncols=2,
        loc="lower right",
        bbox_to_anchor=(1.0, 0.01),
        framealpha=0.9,
        columnspacing=1.5,
    )
    fig.tight_layout()
    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
        print(f"Saved -> {savepath}")
    plt.show()
    return fig, ax

In [ ]:
benchmark_configs = get_benchmark_configs(device, expert_bias_model)
benchmarks = build_benchmarks(benchmark_configs)
all_models = load_model_registry(MODEL_PATHS, device)


In [ ]:
stats_weighted = collect_feedback_stats(all_models["Mixture Prior"], benchmarks, benchmark_configs, experiment, batch_size, T)
stats_amortized = collect_feedback_stats(all_models["Additive Prior"], benchmarks, benchmark_configs, experiment, batch_size, T)


In [ ]:
plot_agreement({"Mixture Prior": stats_weighted, "Additive Prior": stats_amortized}, log_scale=False)

# Main results: multiple benchmarks

`run_benchmark_evaluation` runs every method on every benchmark with the same seed, pool and initial context.
The flags at the top select the expert regime (marginal: `N_MARGINAL_SAMPLES=2000`, XGBoost: `expert_model_type="sklearn"` and `N_MARGINAL_SAMPLES=1`, fake peaks: `FAKE_PEAKS`)
and the slow GP baselines (`INCLUDE_MTGP`, `INCLUDE_RGPE`, `INCLUDE_IMFBO`).


In [ ]:
BASELINES_WITH_EXPERT_DATA = False  # true -> baselines receive concat feedback
USE_REAL_GP = False  # true -> pass sklearn GP into amortized baselines

T = 5  # TEMP smoke, real: 30
BATCH_SIZE = 4  # TEMP smoke, real: 100
N_ITER_GP = 5  # TEMP smoke, real: 100
EXPERT_DIM = 1
N_MARGINAL_SAMPLES = 10  # TEMP smoke, real: 2000, 1 for xgboost
DESIGN_SCALE = 5
NOISE_SCALE = 0.1
n_query_init = 300
expert_bias_model_used = None  # or expert_bias_model

# multi-task GP, feedback as 2nd task (slow)
INCLUDE_MTGP = True
N_MTGP_ITER = 30  # adam steps per fit
MTGP_ACQUISITIONS = ("ucb",)
MTGP_KAPPA = 1.0  # shared by mtgp_*, gp_*, gp_feat_*

# RGPE, feedback as base task (slow, per-env GP fits + LOOCV every step)
INCLUDE_RGPE = True
RGPE_ACQUISITIONS = ("ucb",)
RGPE_KAPPA = MTGP_KAPPA
RGPE_MAX_POOL = 300
RGPE_NUM_POSTERIOR_SAMPLES = 256  # as in the RGPE tutorial
RGPE_TARGET_NOISE_VAR = NOISE_SCALE**2
RGPE_FEEDBACK_NOISE_VAR = NOISE_SCALE**2

# iMFBO (Fan et al. 2024, utils/imfbo_gp.py), feedback as noisy obs of f
INCLUDE_IMFBO = True
IMFBO_ACQUISITIONS = ("nvucb", "ucb")  # ucb = same surrogate, control for the acquisition
IMFBO_NOISE_MODELS = ("parametric", "gp")  # gp = eq. 12 free log-noise vector
IMFBO_MAX_POOL = 300  # cholesky is cubic in this
IMFBO_KAPPA = MTGP_KAPPA
N_IMFBO_ITER = 50

# decoy optima in the feedback only, never in y
FAKE_PEAKS = True
FAKE_PEAK_K = 1
FAKE_PEAK_HEIGHT_MULT = 1.1
FAKE_PEAK_DECAY_RANGE = (0.5, 0.5)

benchmarks = ["sphere3dn", "sphere2dn"] 
#or [#TEMP smoke, real: list below
    # 3-D
#    "sphere3dn", "ackley3dn", "rastrigin3dn", "rosenbrock3dn", "schwefel3dn", "levy3dn", "hartmann3dn",
    # 2-D
#    "sphere2dn", "ackley2dn", "rastrigin2dn", "rosenbrock2dn", "schwefel2dn", "levy2dn",
    # other: "branin_fb2dn", "reactor", "nasbench201", "airfoil_lift_drag", "airfoil_nn"
#]


if BASELINES_WITH_EXPERT_DATA:
    model_for_baselines = expert_model_using_feedback
    n_theta_baselines = cfg_expert_model_using_feedback.task.n_target_theta
else:
    model_for_baselines = expert_model_not_using_feedback
    n_theta_baselines = cfg_expert_model_not_using_feedback.task.n_target_theta


kernel = C(1.0) * RBF(length_scale=1.0)
sklearn_gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
_gp_model = sklearn_gp if USE_REAL_GP else None


adjustors = {
    "none": None,
    "pibo": partial(pi_bo, beta=2.),
}


def _amortized_template(adjustor):
    """GP-prior ICBO surrogate, routes through evaluate_policy_vs_bo."""
    return MethodConfig(
        name="_",
        model=model_for_baselines,
        cfg=cfg,
        attend_to="data",
        n_theta=n_theta_baselines,
        use_gp=False,
        gp_model=_gp_model,
        expert_feedback_adjustor=adjustor,
        advanced_stats=False,
        concat_feedback=BASELINES_WITH_EXPERT_DATA,
        n_iter_gp=N_ITER_GP,
    )


def _ficbo_surrogate_template(adjustor):
    """FICBO (Mixture Prior) inference head as surrogate, i.e. semi-amortized FICBO-UCB/EI."""
    return MethodConfig(
        name="_",
        model=model_reweight,
        cfg=cfg,
        attend_to="data",
        n_theta=n_theta_baselines,
        use_gp=False,
        gp_model=_gp_model,
        expert_feedback_adjustor=adjustor,
        advanced_stats=False,
        concat_feedback=BASELINES_WITH_EXPERT_DATA,
        n_iter_gp=N_ITER_GP,
    )


def _torch_gp_template(adjustor, concat_feedback=None):
    """
    Classical GP, routes through evaluate_gp_bo.
    concat_feedback=True fits the GP on [x, feedback] (gp_feat_*).
    """
    return MethodConfig(
        name="_",
        model=model_for_baselines,  # unused by evaluate_gp_bo
        cfg=cfg,
        attend_to="data",
        n_theta=n_theta_baselines,
        use_gp=True,
        gp_model=None,
        expert_feedback_adjustor=adjustor,
        advanced_stats=False,
        concat_feedback=BASELINES_WITH_EXPERT_DATA if concat_feedback is None else concat_feedback,
        n_iter_gp=N_ITER_GP,
        kappa=MTGP_KAPPA,
    )


def _multitask_gp_template(kind, adjustor=None):
    """Multi-task GP, feedback as 2nd task, routes through evaluate_multitask_gp_bo."""
    return MethodConfig(
        name="_",
        model=model_for_baselines,  # unused by evaluate_multitask_gp_bo
        cfg=cfg,
        attend_to="data",
        n_theta=n_theta_baselines,
        use_multitask_gp=True,
        mtgp_kind=kind,
        expert_feedback_adjustor=adjustor,
        advanced_stats=False,
        n_iter_gp=N_MTGP_ITER,
        kappa=MTGP_KAPPA,
    )


def _rgpe_template(adjustor=None):
    """RGPE, feedback as base task, routes through evaluate_rgpe_bo."""
    return MethodConfig(
        name="_",
        model=model_for_baselines,  # unused by evaluate_rgpe_bo
        cfg=cfg,
        attend_to="data",
        n_theta=n_theta_baselines,
        use_rgpe=True,
        rgpe_max_pool=RGPE_MAX_POOL,
        rgpe_num_posterior_samples=RGPE_NUM_POSTERIOR_SAMPLES,
        rgpe_target_noise_var=RGPE_TARGET_NOISE_VAR,
        rgpe_feedback_noise_var=RGPE_FEEDBACK_NOISE_VAR,
        expert_feedback_adjustor=adjustor,
        advanced_stats=False,
        kappa=RGPE_KAPPA,
    )


def _imfbo_template(noise_model="parametric", adjustor=None):
    """iMFBO, feedback as input-dependent-fidelity obs of f, routes through evaluate_imfbo_bo."""
    return MethodConfig(
        name="_",
        model=model_for_baselines,  # unused by evaluate_imfbo_bo
        cfg=cfg,
        attend_to="data",
        n_theta=n_theta_baselines,
        use_imfbo=True,
        imfbo_noise_model=noise_model,
        imfbo_max_pool=IMFBO_MAX_POOL,
        expert_feedback_adjustor=adjustor,
        advanced_stats=False,
        n_iter_gp=N_IMFBO_ITER,
        kappa=IMFBO_KAPPA,
    )


method_configs = [
    # FICBO
    MethodConfig("ficbo_combined", model, cfg, acquisition="aae", attend_to="data", n_theta=cfg.task.n_target_theta, advanced_stats=False,),
    MethodConfig("ficbo_mixture", model_reweight, cfg_reweight, acquisition="aae", attend_to="data", n_theta=cfg.task.n_target_theta, advanced_stats=False,),
    MethodConfig("ficbo_additive", model_Amortized, cfg_Amortized, acquisition="aae", attend_to="data", n_theta=cfg.task.n_target_theta, advanced_stats=False,),
    MethodConfig("ficbo_ensemble", model_combined, cfg_reweight, acquisition="aae", attend_to="data", n_theta=cfg.task.n_target_theta, advanced_stats=False,),

    # GP-prior ICBO, no feedback
    MethodConfig("gp_icbo", expert_model_not_using_feedback, cfg, acquisition="aae", attend_to="data", n_theta=cfg_expert_model_not_using_feedback.task.n_target_theta, concat_feedback=False, advanced_stats=False, ),

    # GP-prior ICBO + feedback feature
    MethodConfig(
        "gp_icbo_fb",
        expert_model_using_feedback, cfg,
        acquisition="aae",
        attend_to="data",
        n_theta=cfg_expert_model_using_feedback.task.n_target_theta,
        concat_feedback=True,
        experiment_dim_x_delta=cfg.task.dim_feedback,
        advanced_stats=False,
    ),

    # random, feedback
    *baseline_configs(_amortized_template(adjustors["none"]), acquisitions=("random", "feedback",)),

    *(baseline_configs(
        _multitask_gp_template("icm"), acquisitions=MTGP_ACQUISITIONS, name_prefix="mtgp_icm_",
    ) if INCLUDE_MTGP else []),

    # check the printed [imfbo_*] diagnostics before trusting a ranking: lf_snr (feedback usable?), gamma (nvucb != ucb?)
    *([
        mc
        for nm in IMFBO_NOISE_MODELS
        for mc in baseline_configs(
            _imfbo_template(nm), acquisitions=IMFBO_ACQUISITIONS,
            name_prefix="imfbo_" if nm == "parametric" else f"imfbo_{nm}_",
        )
    ] if INCLUDE_IMFBO else []),

    *(baseline_configs(
        _rgpe_template(), acquisitions=RGPE_ACQUISITIONS, name_prefix="rgpe_",
    ) if INCLUDE_RGPE else []),

    # classical-GP controls: gp_* no feedback, gp_feat_* feedback as input feature
    *baseline_configs( _torch_gp_template(adjustors["none"], concat_feedback=False), acquisitions=MTGP_ACQUISITIONS, name_prefix="gp_",),
    *baseline_configs( _torch_gp_template(adjustors["none"], concat_feedback=True), acquisitions=MTGP_ACQUISITIONS, name_prefix="gp_feat_",),

    # piBO-UCB/EI, GP-prior ICBO surrogate
    *baseline_configs(_amortized_template(adjustors["pibo"]), acquisitions=("ucb", "ei",), name_prefix="pibo_",),

    # FICBO-UCB/EI, semi-amortized
    *baseline_configs(_ficbo_surrogate_template(adjustors["pibo"]), acquisitions=("ucb", "ei", ), name_prefix="pibo_mixture_",),

    # piBO-UCB/EI GP, classical GP surrogate
    *baseline_configs(_torch_gp_template(adjustors["pibo"]), acquisitions=("ucb", "ei"), name_prefix="gp_pibo_",),
]
if True:
    results, optima = run_benchmark_evaluation(
        method_configs=method_configs,
        experiment=experiment,
        evaluate_policy_vs_bo=evaluate_policy_vs_bo,
        evaluate_gp_bo=evaluate_gp_bo,
        batch_size=BATCH_SIZE,
        T=T,
        benchmarks=benchmarks,
        BenchmarkFeedbackTask=BenchmarkFeedbackTask,  # BenchmarkFeedbackTask, AirfoilTaskWrapper, NASBench201TaskWrapper
        cfg=cfg,
        expert_dim=EXPERT_DIM,
        n_marginal_samples=N_MARGINAL_SAMPLES,
        design_scale=DESIGN_SCALE,
        noise_scale=NOISE_SCALE,
        seed_fn=seed_everything,
        verbose=1,
        expert_bias_model=expert_bias_model_used,
        expert_generator=expert_generator,
        n_expert_extra_min = 50,
        n_expert_extra_max = 50,
        expert_min_centers = 3,
        expert_max_centers = 3,
        expert_cluster_std_min = .5,
        expert_cluster_std_max = 1.5,
        uniform_fraction = 1.,
        fake_peaks=FAKE_PEAKS,
        fake_peak_k=FAKE_PEAK_K,
        fake_peak_height_mult=FAKE_PEAK_HEIGHT_MULT,
        fake_peak_decay_range=FAKE_PEAK_DECAY_RANGE,
        device=device,
        n_query_init=n_query_init,
    )
else: # one task, many batches, no benchmark loop
    results = evaluate_multi_batch_Compatible(
        n_batches=2,  # TEMP smoke, real: 10
        method_configs=method_configs,
        experiment=experiment,
        evaluate_policy_vs_bo=evaluate_policy_vs_bo,
        evaluate_gp_bo=evaluate_gp_bo,
        batch_size=BATCH_SIZE,
        T=T,
        device=device,
    )
    results = {"multi_batch": results}


In [ ]:
bench = "BenchmarksMany_marginalized_1x1.1peak_.5wide"  # figure folder name, e.g. BenchmarksMany, BenchmarksMany_XGBoost, BenchmarksMany_GP, Airflow, reactor, BraninFBpoint5
include_GP = False  # piBO GP baselines
include_MTGP = False
include_RGPE = False
include_IMFBO = False# needs INCLUDE_IMFBO in the run cell
include_amortized = True  # piBO baselines on the GP-prior ICBO surrogate
include_combined = False  # FICBO (Combined)
use_extra = False  # FICBO-UCB/EI, semi-amortized
selection_id = -1  # >= 0: plot one benchmark only
if include_amortized and include_GP:
    methods_grp_name = "GpAndNot"
elif include_amortized:
    methods_grp_name = "AmortizedOnly"
elif include_GP:
    methods_grp_name = "GpOnly"
else:
    methods_grp_name = "NoBaselines"

include_legend = False
Y_label = False
normalized_regret = True
extra_height = 3# 3 for more, else 0
print(f"{save_plots=}")


### Plots

`results[bench][method]` = (selected, best_so_far, rmse, regret, top5, regret_cum).
Colors are fixed per method key, so the same method has the same color in every figure.


In [ ]:
IMFBO_LABEL_PREFIX = {"parametric": "", "gp": "GP-"}


def imfbo_plot_methods() -> List[Tuple[str, str]]:
    # keys mirror the run cell's name_prefix
    return [
        (
            f"imfbo_{'' if nm == 'parametric' else nm + '_'}{acq}",
            f"iMFBO-{IMFBO_LABEL_PREFIX.get(nm, nm.upper() + '-')}{acq.upper()}",
            )
        for nm in IMFBO_NOISE_MODELS
        for acq in IMFBO_ACQUISITIONS
        ]


def baseline_plot_methods(
    include_MTGP: bool = True, include_RGPE: bool = True, include_IMFBO: bool = True
    ) -> List[Tuple[str, str]]:
    methods: List[Tuple[str, str]] = []
    if include_MTGP:
        methods.append(("mtgp_icm_ucb", "MTGP-UCB (ICM)"))
    if include_RGPE:
        methods.append(("rgpe_ucb", "RGPE-UCB"))
    if include_IMFBO:
        methods += imfbo_plot_methods()
    return methods


print("imfbo plot labels ->", [label for _, label in imfbo_plot_methods()])


In [ ]:
# same colors as the paper legend
KEY_COLORS = {
    "ficbo_mixture": "#dd8452",
    "ficbo_additive": "#e63946",
    "gp_icbo_fb": "#4c72b0",
    "gp_icbo": "#2196f3",
    "random": "#937860",
    "feedback": "#636e72",
    "pibo_ucb": "#55a868",
    "pibo_ei": "#26de81",
    "gp_pibo_ucb": "#9d4edd",
    "gp_pibo_ei": "#ff69b4",
}
EXTRA_COLORS = ["#f9ca24", "#8172b3", "#c77d00", "#a29bfe", "#17becf", "#bcbd22", "#7f7f7f", "#b2bec3", "#c44e52", "#8c564b"]


def key_colors(keys: List[str]) -> List[str]:
    extra = iter(EXTRA_COLORS)
    return [KEY_COLORS[k] if k in KEY_COLORS else next(extra) for k in keys]


In [ ]:
def main_series() -> List[Tuple[str, str]]:
    # (results key, label), legend order
    series = [
        ("ficbo_mixture", "FICBO (Mixture Prior)"),
        ("ficbo_additive", "FICBO (Additive Prior)"),
        ("gp_icbo_fb", "GP-prior ICBO + feedback feature"),
        ("gp_icbo", "GP-prior ICBO"),
        ("random", "Random"),
        ("feedback", "Feedback"),
        ]
    if include_amortized:
        series += [("pibo_ucb", "πBO-UCB"), ("pibo_ei", "πBO-EI")]
    series += baseline_plot_methods(include_MTGP, include_RGPE, include_IMFBO)
    if include_GP:
        series += [("gp_pibo_ucb", "πBO-UCB GP"), ("gp_pibo_ei", "πBO-EI GP")]
    if use_extra:
        series += [
            ("pibo_mixture_ucb", "πBO-UCB (Mixture-prior surrogate)"),
            ("pibo_mixture_ei", "πBO-EI (Mixture-prior surrogate)"),
            ]
    if include_combined:  # last, keeps the other colors
        series += [("ficbo_combined", "FICBO (Combined)")]
    return series


def gather_series(
    data: Dict[str, torch.Tensor], series: List[Tuple[str, str]]
    ) -> Tuple[List[torch.Tensor], List[str], List[str]]:
    # [n_bench, T, B] -> [T, n_bench * B], or one benchmark
    tensors, labels, keys = [], [], []
    for key, label in series:
        if key not in data:
            print(f"[plot] skipping {label}: '{key}' not in results")
            continue
        el = data[key]
        tensors.append(el[selection_id] if selection_id >= 0 else el.permute(1, 0, 2).reshape(el.shape[1], -1))
        labels.append(label)
        keys.append(key)
    return tensors, labels, keys


def print_final(tensors: List[torch.Tensor], labels: List[str], title: str, higher_is_better: bool):
    finals = {name: (t[-1].mean().item(), t[-1].std().item()) for name, t in zip(labels, tensors)}
    print(f"\n{title}:")
    print(f"  {'Method':<60} {'Mean':>8} {'Std':>8}")
    print("  " + "-"*78)
    ranked = sorted(finals.items(), key=lambda x: x[1][0], reverse=higher_is_better)
    for rank, (name, (mean, std)) in enumerate(ranked, 1):
        print(f"  {rank:2}. {name:<56} {mean:8.3f} {std:8.3f}")


def plot_panel(
    results: dict, optima: dict, series: List[Tuple[str, str]], panel: str, bench_name: str,
    figsize: Tuple[float, float], use_key_colors: bool = True,
    ):
    """
    Args:
        panel: "best", "best_rank", "cum_regret" or "cum_regret_rank"
        bench_name: figure folder and file prefix
        use_key_colors: False -> default palette by position
    """
    if panel in ("best", "best_rank"):
        data = (get_normalized_best_so_far(results, optima) if normalized_regret
                else aggregate_results(results, metric="best_so_far"))
    elif panel == "cum_regret":
        data = (get_normalized_cum_regret(results, optima) if normalized_regret
                else aggregate_results(results, metric="regret_cum"))
    else:
        data = aggregate_results(results, metric="regret_cum")#raw, ranks only

    tensors, labels, keys = gather_series(data, series)
    if panel in ("cum_regret", "cum_regret_rank"):
        tensors = [torch.cat([torch.zeros(1, t.shape[1]), t], dim=0) for t in tensors]  # regret 0 at t=0
    colors_used = key_colors(keys) if use_key_colors else None
    common = dict(
        title="", xlabel="iterations", figsize=figsize, marker_frequency=len(tensors),
        nbins=T//5+1, include_legend=include_legend, colors=colors_used,
        )

    if panel == "best":
        for i, t in enumerate(tensors[1:], start=1):
            assert torch.allclose(t[0], tensors[0][0]), f"{labels[i]} differs from {labels[0]} at t=0"
        print_final(tensors, labels, "Final best value found (higher = better)", higher_is_better=True)
        if normalized_regret:
            show_optimal, optimal_value = True, 1
        elif selection_id < 0 and len(optima) > 0:
            show_optimal, optimal_value = True, torch.mean(optima[list(results)[0]]).cpu().numpy()
        else:
            show_optimal, optimal_value = False, -10
        fig, ax = plot_metrics_with_confidence(
            tensors, labels, ylabel="Best Value Found" if Y_label else "", legend_loc="lower right",
            show_optimal=show_optimal, optimal_value=optimal_value, top_extra=.1, **common,
            )
        name = f"BestFound{'_legend' if include_legend else '_noLegend'}{'_showOptimal' if show_optimal else ''}"
    elif panel == "best_rank":
        fig, ax = plot_ranks_with_confidence(
            tensors, labels, ylabel="Rank" if Y_label else "", legend_loc="lower left", low_is_better=False, **common,
            )
        name = f"BestFoundRank{'_legend' if include_legend else '_noLegend'}"
    elif panel == "cum_regret":
        print_final(tensors, labels, "Final cumulative regret (lower = better)", higher_is_better=False)
        fig, ax = plot_metrics_with_confidence(
            tensors, labels, ylabel="Cumulative Regret" if Y_label else "", legend_loc="upper left", **common,
            )
        name = f"CumRegret{'_legend' if include_legend else '_noLegend'}"
    else:
        fig, ax = plot_ranks_with_confidence(
            tensors, labels, ylabel="Rank" if Y_label else "", legend_loc="lower left", low_is_better=True, **common,
            )
        name = f"CumRegretRank{'_legend' if include_legend else '_noLegend'}"

    fname = (f"figures/{folder_grp}/{bench_name}/{bench_name}_{name}_{methods_grp_name}{'_WithYLabel' if Y_label else ''}"
             f"_EXPERT_DIM{EXPERT_DIM}_BATCH_SIZE{BATCH_SIZE}_T{T}{'_NormalizedY' if normalized_regret else ''}.pdf")
    print(fname)
    if save_plots:
        os.makedirs(os.path.dirname(fname), exist_ok=True)
        fig.savefig(fname, bbox_inches="tight")
        if panel == "best":#legend as its own pdf
            fig_legend, ax_legend = plt.subplots(figsize=(3, 1))
            ax_legend.axis("off")
            handles, leg_labels = ax.get_legend_handles_labels()
            ax_legend.legend(handles, leg_labels, loc="center", frameon=True, ncol=(len(leg_labels)+1)//2)
            fig_legend.savefig(f"figures/{folder_grp}/{bench_name}/LegendOf_{bench_name}_{methods_grp_name}_EXPERT_DIM{EXPERT_DIM}.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
for panel in ("best", "best_rank", "cum_regret", "cum_regret_rank"):
    plot_panel(results, optima, main_series(), panel, bench, figsize=(8, 5 + extra_height))


## Per-benchmark table

Final best value and regret per (benchmark, method). `ALL (mean)` averages over benchmarks. Saved as .csv and .tex.


In [ ]:
# key in results[bench] -> display name used in the plots above
TABLE_METHODS = [
    # FICBO
    ("ficbo_mixture", "FICBO (Mixture Prior)"),
    ("ficbo_additive", "FICBO (Additive Prior)"),
    ("ficbo_combined", "FICBO (Combined)"),
    ("ficbo_ensemble", "FICBO (Ensemble)"),

    # GP-prior baselines
    ("gp_icbo_fb", "GP-prior ICBO + feedback feature"),
    ("gp_icbo", "GP-prior ICBO"),

    # piBO
    ("pibo_ucb", "πBO-UCB"),
    ("pibo_ei", "πBO-EI"),

    # multi-task baselines
    ("mtgp_icm_ucb", "MTGP-UCB (ICM)"),
    ("rgpe_ucb", "RGPE-UCB"),

    # iMFBO
    ("imfbo_gp_nvucb", "iMFBO-GP-NVUCB"),
    ("imfbo_gp_ucb", "iMFBO-GP-UCB"),
    ("imfbo_nvucb", "iMFBO-NVUCB"),
    ("imfbo_ucb", "iMFBO-UCB"),

    # GP versions of piBO
    ("gp_pibo_ucb", "πBO-UCB GP"),
    ("gp_pibo_ei", "πBO-EI GP"),

    # simple baselines
    ("feedback", "Feedback"),
    ("random", "Random"),

    # semi-amortized FICBO-UCB/EI
    ("pibo_mixture_ucb", "πBO-UCB (Mixture-prior surrogate)"),
    ("pibo_mixture_ei", "πBO-EI (Mixture-prior surrogate)"),
]


def _mean_sem(x: torch.Tensor) -> Tuple[float, float]:
    x = x.float().flatten()
    sem = (x.std(unbiased=True) / max(x.numel(), 1) ** 0.5).item() if x.numel() > 1 else 0.0
    return x.mean().item(), sem


def per_benchmark_table(
    results: dict, optima: dict, methods: List[Tuple[str, str]] = TABLE_METHODS
    ) -> pd.DataFrame:
    """
    One row per (benchmark, method) with final-step performance.

    Best Found          : raw best-so-far at t=T (higher = better)
    Norm. Best          : (best_T - best_0) / (optimum - best_0), 1.0 = optimum found
    Regret              : optimum - best_T, raw simple regret (lower = better)
    Cum. Regret         : sum_t (1 - normalized best-so-far_t), lower = better
    Rank (by Norm. Best): rank by Norm. Best within the benchmark, 1 = best

    Methods missing from a benchmark (or that returned None) are skipped.
    """
    rows = []
    for bench_name, per_method in results.items():
        opt = optima[bench_name].float().cpu()  # [B]
        for key, label in methods:
            res = per_method.get(key)
            if res is None:
                continue
            best = res[1].float().cpu()  # [T+1, B]
            init = best[0:1]
            norm = (best - init) / (opt.unsqueeze(0) - init + 1e-8)
            bf_m, bf_s = _mean_sem(best[-1])
            nb_m, nb_s = _mean_sem(norm[-1])
            rg_m, rg_s = _mean_sem(opt - best[-1])
            cr_m, cr_s = _mean_sem((1.0 - norm[1:]).sum(dim=0))
            rows.append({
                "Benchmark": bench_name, "Method": label,
                "Best Found": bf_m, "Best Found sem": bf_s,
                "Norm. Best": nb_m, "Norm. Best sem": nb_s,
                "Regret": rg_m, "Regret sem": rg_s,
                "Cum. Regret": cr_m, "Cum. Regret sem": cr_s,
            })

    df = pd.DataFrame(rows)
    df["Rank (by Norm. Best)"] = df.groupby("Benchmark")["Norm. Best"].rank(
        ascending=False, method="min"
    ).astype(int)

    mean_rows = df.groupby("Method", sort=False).mean(numeric_only=True).reset_index()
    mean_rows["Benchmark"] = "ALL (mean)"
    mean_rows["Rank (by Norm. Best)"] = mean_rows["Norm. Best"].rank(
        ascending=False, method="min"
    ).astype(int)
    df = pd.concat([df, mean_rows[df.columns]], ignore_index=True)

    order = list(results.keys()) + ["ALL (mean)"]
    df["Benchmark"] = pd.Categorical(df["Benchmark"], categories=order, ordered=True)
    return df.sort_values(["Benchmark", "Rank (by Norm. Best)"]).reset_index(drop=True)


def format_benchmark_table(df: pd.DataFrame, with_sem: bool = True) -> pd.DataFrame:
    """Collapse the mean/sem pairs into 'mean +- sem' strings, indexed by (Benchmark, Method)."""
    out = pd.DataFrame(index=pd.MultiIndex.from_frame(df[["Benchmark", "Method"]]))
    for col, fmt in [("Best Found", "{:.3f}"), ("Norm. Best", "{:.3f}"),
                     ("Regret", "{:.3f}"), ("Cum. Regret", "{:.2f}")]:
        if with_sem:
            out[col] = [f"{fmt.format(m)} +- {fmt.format(s)}"
                        for m, s in zip(df[col], df[f"{col} sem"])]
        else:
            out[col] = [fmt.format(m) for m in df[col]]
    out["Rank (by Norm. Best)"] = df["Rank (by Norm. Best)"].values
    return out


table_df = per_benchmark_table(results, optima)
with pd.option_context("display.max_rows", None):
    display(format_benchmark_table(table_df, with_sem=True))

table_title = f"figures/{folder_grp}/{bench}/{bench}_PerBenchmarkTable_{methods_grp_name}_EXPERT_DIM{EXPERT_DIM}_BATCH_SIZE{BATCH_SIZE}_T{T}"
print(table_title)
if save_plots:
    os.makedirs(os.path.dirname(table_title), exist_ok=True)
    table_df.to_csv(f"{table_title}.csv", index=False)
    with open(f"{table_title}.tex", "w") as f:
        f.write(format_benchmark_table(table_df, with_sem=False).to_latex(escape=False))

# Ablation

FICBO variants trained with other feedback priors, on Airfoil Small and Airfoil Lift (paper ablation table).


In [ ]:

# one entry per ablation run, replace with your runs
model_variants_2d = [
    ("outputs/hydra/ficbo_prior/2026-07-08/12-32-56/", "Mixture, no bias operators"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "Additive, no bias operators"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based feedback (standard)"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based, no catastrophic"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based, no shift"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based, no biases"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "f-based, standard biases"),  # TEMP path
]

model_variants_3d = [
    ("outputs/hydra/ficbo_prior/2026-07-08/12-32-56/", "Mixture, no bias operators"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "Additive, no bias operators"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based feedback (standard)"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based, no catastrophic"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based, no shift"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "g-based, no biases"),  # TEMP path
    ("outputs/hydra/ficbo_prior/2026-06-18/16-35-44/", "f-based, standard biases"),  # TEMP path
]

benchmarks = ["airfoil_nn", "airfoil_lift_drag"]#airfoil Small, Airfoil Lift

model_variants = model_variants_3d  # or model_variants_2d

method_configs = []
for path, name in model_variants:
    cfg_v, model_v = load_config_and_model(path=path, file_name="model/aline_Human_GP_simple.pth")  # TEMP path
    method_configs.append(
        MethodConfig(name, model_v.to(device), cfg_v, acquisition="aae", attend_to="data",
                     n_theta=cfg_v.task.n_target_theta, advanced_stats=False)
    )
    
method_configs += [
    MethodConfig("ficbo_mixture", model_reweight, cfg_reweight, acquisition="aae", attend_to="data",n_theta=cfg.task.n_target_theta, advanced_stats=False,),
    MethodConfig("ficbo_additive", model_Amortized, cfg_Amortized, acquisition="aae", attend_to="data",n_theta=cfg.task.n_target_theta, advanced_stats=False,),

    *baseline_configs(_amortized_template(None), acquisitions=("random", "feedback")),
    *baseline_configs(_amortized_template(adjustors["pibo"]), acquisitions=("ucb", "ei"), name_prefix="pibo_"),
]

results_ablation, optima_ablation = run_benchmark_evaluation(
    method_configs=method_configs,
    experiment=experiment,
    evaluate_policy_vs_bo=evaluate_policy_vs_bo,
    evaluate_gp_bo=evaluate_gp_bo,
    batch_size=BATCH_SIZE,
    T=T,
    benchmarks=benchmarks,
    BenchmarkFeedbackTask=AirfoilTaskWrapper,
    cfg=cfg,
    expert_dim=EXPERT_DIM,
    n_marginal_samples=N_MARGINAL_SAMPLES,
    design_scale=DESIGN_SCALE,
    noise_scale=NOISE_SCALE,
    seed_fn=seed_everything,
    verbose=1,
    expert_generator=expert_generator,
    n_expert_extra_min=50, n_expert_extra_max=50,
    expert_min_centers=3, expert_max_centers=3,
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
    n_query_init=n_query_init,
)


In [ ]:
ablation_series = [
    ("ficbo_mixture", "FICBO (Mixture)"),
    ("Mixture, no bias operators", "Mixture, no bias operators"),
    ("ficbo_additive", "FICBO (Additive)"),
    *[(name, name) for _, name in model_variants[1:]],
    ("pibo_ucb", "πBO-UCB"),
    ("pibo_ei", "πBO-EI"),
    ("feedback", "Feedback"),
    ("random", "Random"),
]
for panel in ("best", "best_rank", "cum_regret", "cum_regret_rank"):
    plot_panel(results_ablation, optima_ablation, ablation_series, panel, f"{bench}_ablation", figsize=(8, 5), use_key_colors=False)


# Multi-seed evaluation

Not in the paper. All methods on `N_EVAL_BATCHES` fresh batches of the training task.


In [ ]:
seed_everything()

In [ ]:
T = 5  # TEMP smoke, real: 20

In [ ]:
N_EVAL_BATCHES = 2  # TEMP smoke, real: 20
 
baselines_with_expert_data= False

use_real_gp= False
kernel = C(1.0) * RBF(length_scale=1.0)
test_gp_model = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)

expert_feedback_adjustor= partial(pi_bo, beta=2)


if baselines_with_expert_data:
    model_for_baselines = expert_model_using_feedback
    n_theta_baselines = cfg_expert_model_using_feedback.task.n_target_theta
    #experiment_for_baselines=experiment_copy
else:
    model_for_baselines = expert_model_not_using_feedback
    n_theta_baselines = cfg_expert_model_not_using_feedback.task.n_target_theta
    #experiment_for_baselines= experiment


baseline_template = MethodConfig(
    name="_",
    model=model_for_baselines,
    cfg=cfg,
    attend_to="data",
    n_theta=n_theta_baselines,
    use_gp=False,
    gp_model=test_gp_model,
    expert_feedback_adjustor=expert_feedback_adjustor,
    advanced_stats=False,
    concat_feedback=baselines_with_expert_data,
)

baseline_template2 = MethodConfig(
    name="_",
    model=model_for_baselines,
    cfg=cfg,
    attend_to="data",
    n_theta=n_theta_baselines,
    use_gp=True,
    gp_model=test_gp_model,
    expert_feedback_adjustor=expert_feedback_adjustor,
    advanced_stats=False,
    concat_feedback=baselines_with_expert_data,
)
 
 
method_configs = [
    # FICBO
    MethodConfig("ficbo_combined", model, cfg,
                 acquisition="aae", attend_to="data",
                 n_theta=cfg.task.n_target_theta),
 
    # GP-prior ICBO
    MethodConfig("gp_icbo", expert_model_not_using_feedback, cfg,
                 acquisition="aae", attend_to="data",
                 n_theta=cfg_expert_model_not_using_feedback.task.n_target_theta),
 
    # GP-prior ICBO + feedback feature
    MethodConfig("gp_icbo_fb", expert_model_using_feedback, cfg,
                 acquisition="aae", attend_to="data",
                 n_theta=cfg_expert_model_using_feedback.task.n_target_theta,
                 concat_feedback=True,
                 experiment_dim_x_delta=cfg.task.dim_feedback),
 
    # classical baselines, only acq varies
    *baseline_configs(baseline_template, acquisitions = ("random", "ucb", "ei"),),

    #*baseline_configs(baseline_template2, acquisitions = ("ucb", "ei"),name_prefix="gp_"),
]
 


In [ ]:
data_multi = evaluate_multi_batch(
    n_batches=N_EVAL_BATCHES,
    batch_size=batch_size,
    experiment=experiment,
    T=T,
    method_configs=method_configs,
)
 

In [ ]:
data_multi.keys()

In [ ]:
idx = 1  # 0=selected_values, 1=best_so_far, 2=rmse, 3=regret, 4=cum_regret
method_names = ["ficbo_combined", "gp_icbo", "gp_icbo_fb", "random", "ucb", "ei"]
 
fig, ax = plot_metrics_with_confidence(
    [data_multi[m][idx] for m in method_names],
    ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature", "Random", "UCB", "EI"],
    title=f"Best-so-far ({N_EVAL_BATCHES} batches) on GP data",
    ylabel="Best so far",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc="lower right",
    nbins=T/5,
)
plt.show()


In [ ]:
idx = 4  # 0=selected_values, 1=best_so_far, 2=rmse, 3=regret, 4=cum_regret
method_names = ["ficbo_combined", "gp_icbo", "gp_icbo_fb", "random", "ucb", "ei"]
 
fig, ax = plot_metrics_with_confidence(
    [data_multi[m][idx] for m in method_names],
    ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature", "Random", "UCB", "EI"],
    title=f"Cumulative Regret ({N_EVAL_BATCHES} batches) on GP data",
    ylabel=f"Cumulative Regret",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc="upper left",
    nbins=T/5,
)
plt.show()


In [ ]:
idx = 4  # 0=selected_values, 1=best_so_far, 2=rmse, 3=regret, 4=cum_regret
method_names = ["ficbo_combined", "gp_icbo", "gp_icbo_fb", "random", "ucb", "ei"]
 
fig, ax = plot_ranks_with_confidence(
    [data_multi[m][idx] for m in method_names],
    ["FICBO (Combined)", "GP-prior ICBO", "GP-prior ICBO + feedback feature", "Random", "UCB", "EI"],
    title="Cumulative Regret Rank across methods",
    ylabel="Cumulative Regret Rank of selected point",
    xlabel="Timestep",
    figsize=(8, 5),
    marker_frequency=7,
    legend_loc= "upper left",
    low_is_better=True,
    nbins=5,
)

plt.show()


# UMAP of feedback-error profiles

Each task is a vector of binned feedback errors $d = u - y$ (signed and absolute, points sorted by y and by feedback),
embedded with UMAP. Synthetic prior samples vs. test tasks (paper UMAP plot).


## Functions


In [ ]:
#collect_* -> (vectors, labels), concat several before umap_and_plot..
def blue_gradient(n: int) -> List[str]:
    """n blues from light to dark."""
    cmap = plt.get_cmap("Blues")
    return [mcolors.to_hex(cmap(0.3 + 0.6 * i / max(n - 1, 1))) for i in range(n)]


def _build_task_vector(y: torch.Tensor, feedback: torch.Tensor, n_bins: int) -> np.ndarray:
    y = y.detach().cpu().float().reshape(-1)
    fb = feedback.detach().cpu().float().reshape(-1)
    d = fb - y
    d_abs = d.abs()

    def bin_by(key, values):
        order = torch.argsort(key)
        v_sorted = values[order]
        bins = np.array_split(v_sorted.numpy(), n_bins)
        return np.array([b.mean() if len(b) > 0 else 0.0 for b in bins])

    # signed mean-> captures bias direction
    by_y_mean = bin_by(y, d)
    by_feedback_mean = bin_by(fb, d)

    # abs mean: captures where feedback is wrong regardless of direction
    by_y_abs = bin_by(y, d_abs)
    by_feedback_abs = bin_by(fb, d_abs)

    return np.concatenate([by_y_mean, by_feedback_mean, by_y_abs, by_feedback_abs])  # [4 * n_bins]


def collect_weight_sweep(
    task,
    weights: List[float],
    n_batches: int = 30,
    batch_size: int = 8,
    n_bins: int = 20,
    label_prefix: str = "w",
    ) -> tuple[np.ndarray, List[str]]:
    """
    Sample many batches for each weight value and build task vectors.
    Configure task / bias model however you want before calling this.
    The weight is set as expert_weight_min = expert_weight_max = w.

    Parameters
    ----------
    task       instantiated task with expert_weight_min / expert_weight_max
    weights     list of weight values, e.g. [0.1, 0.3, 0.5, 0.7, 0.9]
    n_batches   independent sample_batch() calls per weight
    batch_size  keep small - each call re-samples thetas, bias state, etc.

    Returns
    -------
    vectors     np.ndarray [total_instances, 2*n_bins]
    labels      List[str]  e.g. ["w=0.1", "w=0.1", ..., "w=0.9", ...]
    """
    all_vectors, all_labels = [], []

    for w in weights:
        task.expert_weight_min = w
        task.expert_weight_max = w

        n_collected = 0
        for i in range(n_batches):
            try:
                batch = task.sample_batch(batch_size)
            except Exception as e:
                print(f"  [warn] w={w:.1f} batch {i} failed: {e}")
                continue

            for b in range(batch.query_y.shape[0]):
                vec = _build_task_vector(batch.query_y[b], batch.query_feedback[b], n_bins)
                all_vectors.append(vec)
                all_labels.append(f"{label_prefix} w={w:.1f}")
                n_collected += 1

        print(f"  [w={w:.1f}]  {n_collected} vectors")

    return np.stack(all_vectors), all_labels


def collect_task(
    task,
    label: str,
    n_batches: int = 5,
    batch_size: int = 1,
    n_bins: int = 20,
    ) -> tuple[np.ndarray, List[str]]:
    """
    Collect vectors from a fixed task (no parameter sweep).
    Use for benchmark / real-world tasks like airfoil.

    Returns
    -------
    vectors     np.ndarray [total_instances, 2*n_bins]
    labels      List[str]  all set to label
    """
    all_vectors, all_labels = [], []

    n_collected = 0
    for i in range(n_batches):
        try:
            batch = task.sample_batch(batch_size)
        except Exception as e:
            print(f"  [warn] '{label}' batch {i} failed: {e}")
            continue

        for b in range(batch.query_y.shape[0]):
            vec = _build_task_vector(batch.query_y[b], batch.query_feedback[b], n_bins)
            all_vectors.append(vec)
            all_labels.append(label)
            n_collected += 1

    print(f"  [{label}]  {n_collected} vectors")
    return np.stack(all_vectors), all_labels


def collect_kdecoy_sweep(
    task,
    k_values: List[int],
    n_batches: int = 30,
    batch_size: int = 8,
    n_bins: int = 20,
    label_prefix: str = "k",
    ) -> tuple[np.ndarray, List[str]]:
    """
    Sweep k_decoy (min=max=k) for WeightedMultiGP_Expert_GPTask.

    k=0  feedback = weighted sum of real GPs only  -> agrees with y
    k=3  feedback pulled toward decoy GPs          -> increasingly misleading

    Returns
    -------
    vectors     np.ndarray [total_instances, 2*n_bins]
    labels      List[str]  e.g. ["k=0", "k=0", ..., "k=3", ...]
    """
    all_vectors, all_labels = [], []

    for k in k_values:
        task.k_decoy_min = k
        task.k_decoy_max = k

        n_collected = 0
        for i in range(n_batches):
            try:
                batch = task.sample_batch(batch_size)
            except Exception as e:
                print(f"  [warn] k={k} batch {i} failed: {e}")
                continue

            for b in range(batch.query_y.shape[0]):
                vec = _build_task_vector(batch.query_y[b], batch.query_feedback[b], n_bins)
                all_vectors.append(vec)
                all_labels.append(f"{label_prefix} k={k}")
                n_collected += 1

        print(f"  [k={k}]  {n_collected} vectors")

    return np.stack(all_vectors), all_labels


def umap_and_plot(
    vectors: np.ndarray,
    labels: List[str],
    label_colors: dict,# {label: hex_color}
    label_markers: dict = None,  # {label: marker}  default "o" for all
    title: str = "",
    figsize: tuple = (9, 7),
    alpha_dots: float = 0.45,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
    save_path: Optional[str] = None,
    ) -> plt.Figure:
    """
    Run UMAP on vectors then plot.

    Parameters
    ----------
    label_colors    dict mapping label -> hex color, e.g.
                    {"w=0.1": "#D6E8F7", "w=0.9": "#0A2D6E", "branin": "#E84040"}
    label_markers   dict mapping label -> marker, e.g.
                    {"w=0.1": "o", "branin": "*"}
                    defaults to "o" for any label not specified

    Clouds ("o") drawn behind, heldout markers on top.
    """
    try:
        from umap import UMAP
    except ImportError:
        raise ImportError("pip install umap-learn")

    print(f"  Running UMAP on {vectors.shape} ...")
    embedding = UMAP(
        n_neighbors = n_neighbors,
        min_dist = min_dist,
        random_state = random_state,
    ).fit_transform(vectors)

    unique_labels = list(dict.fromkeys(labels))
    label_markers = label_markers or {}
    marker_map = {lab: label_markers.get(lab, "o") for lab in unique_labels}
    color_map = label_colors

    fig, ax = plt.subplots(figsize=figsize)

    for fg in [False, True]:  # clouds first, heldout on top
        for lab in unique_labels:
            m = marker_map[lab]
            if (m == "o") == fg:
                continue
            idx = [i for i, l in enumerate(labels) if l == lab]
            pts = embedding[idx]
            ax.scatter(
                pts[:, 0], pts[:, 1],
                c = color_map[lab],
                marker = m,
                s = 16 if m == "o" else 160,
                alpha = alpha_dots if m == "o" else 0.95,
                label = lab,
                edgecolors = "none" if m == "o" else "k",
                linewidths = 0.0 if m == "o" else 0.7,
                zorder = 2 if m == "o" else 5,
            )

    ax.legend(framealpha=0.9, fontsize=9, markerscale=1.5)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved -> {save_path}")

    return fig

## Synthetic prior samples


In [ ]:
path = "outputs/hydra/ficbo_prior/2026-06-18/16-35-44/"  # TEMP path

cfg_for_vis, model_for_vis = load_config_and_model(
    path=path,
    file_name="model/aline_Human_GP_simple.pth"  # TEMP path
)
model_for_vis = model.to(device)

if getattr(cfg_for_vis, "expert_bias", None) is not None:
    if getattr(cfg_for_vis.expert_bias, "uncertainty_noise_std", None) is not None:
        with open_dict(cfg.expert_bias):
            cfg_for_vis.expert_bias.uncertainty_noise_std_range = (
                cfg_for_vis.expert_bias.uncertainty_noise_std,
                cfg_for_vis.expert_bias.uncertainty_noise_std,
            )
            del cfg_for_vis.expert_bias.uncertainty_noise_std
    if getattr(cfg_for_vis.expert_bias, "rounding_decimals", None) is not None:
        with open_dict(cfg.expert_bias):
            cfg_for_vis.expert_bias.rounding_scale = cfg_for_vis.expert_bias.rounding_decimals
            del cfg_for_vis.expert_bias.rounding_decimals

    expert_bias_model_vis = hydra.utils.instantiate(cfg_for_vis.expert_bias, device=cfg_for_vis.device)
    print(f"{expert_bias_model.device=}")

else:
    expert_bias_model = None
    print("no expert bias model")


expert_bias_model_vis = deepcopy(expert_bias_model_vis)

for _ in range(1):# fix/adjust biases
    print("entered")
    #expert_bias_model.bias_probs.perceived_max_bias=1
    #cfg.task.expert_extra_data_min = cfg.task.expert_extra_data_max = 30
    #break# just to set an easy stop
    #cfg.task.expert_weight_min = cfg.task.expert_weight_max = 0.8
    cfg.task.expert_noise = 0# output noise
    if getattr(cfg, "expert_bias", None) is not None:
        if cfg.expert_bias._target_ == "expert_models.bias_models.ProbExpertBiasEngine":
            expert_bias_model_vis.bias_probs.hetero_noise=.2
            expert_bias_model_vis.bias_probs.local_averaging=0.2
            expert_bias_model_vis.bias_probs.add_gp_bias=.2
            expert_bias_model_vis.bias_probs.perceived_max_bias=.1
            expert_bias_model_vis.bias_probs.knowledge_center= .2
            expert_bias_model_vis.bias_probs.local_distortion=.3
            expert_bias_model_vis.bias_probs.catastrophic=0.04
        else:
            print(f"No switch: {cfg.expert_bias=}")
        expert_bias_model_vis.add_gp_bias=False  #true
        expert_bias_model_vis.catastrophic=False
        expert_bias_model_vis.knowledge_center=False
        expert_bias_model_vis.perceived_max_bias=False  # false
        expert_bias_model_vis.local_distortion=False
        expert_bias_model_vis.hetero_noise=False
        expert_bias_model_vis.constant_shift_bias =False
        
        
        expert_bias_model_vis.noise_level = (0.0, 0.0)#(.3, .3)#.1-.7
        break
        expert_bias_model_vis.hetero_alpha = (0.2, .2)
        expert_bias_model_vis.hetero_n_centers = (4, 4)
        expert_bias_model_vis.hetero_decay = (.5,.5)#(0.5, 2)
        
        expert_bias_model_vis.add_gp_max_std = (.5, 1.)#.5-1
        expert_bias_model_vis.add_gp_lengthscale = (.5, 3.)# .5-3
        expert_bias_model_vis.add_gp_scale = (.5, 1)  # [0.2, 2.]
        
        expert_bias_model_vis.perceived_max_alpha = (.7, .7)

        expert_bias_model_vis.knowledge_center_noise_radiation_scale= (.3, .3)#[0.1, 0.5]

        expert_bias_model_vis.distortion_range = (.5,.5)  #(.3,.7)
        expert_bias_model_vis.distortion_centers = (2,3)  #(1,5)
        expert_bias_model_vis.distortion_decay = (1.,1.)  #(.5,1.5)        

        print("Adjustment completed")

last_bool= True
if expert_bias_model_vis is not None:
    for attr, value in expert_bias_model_vis.__dict__.items():
        if type(value) is bool:
            last_bool = value
            print(f"{attr} = {value}")
        elif last_bool:
            print(f"{attr} = {value}")

experiment_vis = hydra.utils.instantiate(cfg_for_vis.task, expert_model=expert_model, expert_bias_model=expert_bias_model_vis,
                                model_expert_cfg=cfg.get("expert_model", None), device=device)


In [ ]:
weights = [ 0.3, 0.5, 0.7, 0.9]
Additive_vectors, Additive_labels = collect_weight_sweep(experiment_vis, weights, n_batches=50, batch_size=10, label_prefix="Additive")
Additive_labels[0]

In [ ]:
expert_bias_model_vis2 = deepcopy(expert_bias_model)

for _ in range(1):# fix/adjust biases
    print("entered")
    #expert_bias_model.bias_probs.perceived_max_bias=1
    #cfg.task.expert_extra_data_min = cfg.task.expert_extra_data_max = 30
    #break# just to set an easy stop
    #cfg.task.expert_weight_min = cfg.task.expert_weight_max = 0.8
    cfg.task.expert_noise = 0# output noise
    if getattr(cfg, "expert_bias", None) is not None:
        if cfg.expert_bias._target_ == "expert_models.bias_models.ProbExpertBiasEngine":
            expert_bias_model_vis2.bias_probs.hetero_noise=.2
            expert_bias_model_vis2.bias_probs.local_averaging=0.2
            expert_bias_model_vis2.bias_probs.add_gp_bias=.2
            expert_bias_model_vis2.bias_probs.perceived_max_bias=.1
            expert_bias_model_vis2.bias_probs.knowledge_center= .2
            expert_bias_model_vis2.bias_probs.local_distortion=.3
            expert_bias_model_vis2.bias_probs.catastrophic=0.04
        else:
            print(f"No switch: {cfg.expert_bias=}")
        expert_bias_model_vis2.add_gp_bias=True  #true
        expert_bias_model_vis2.catastrophic=True
        expert_bias_model_vis2.knowledge_center=False
        expert_bias_model_vis2.perceived_max_bias=False  # false
        expert_bias_model_vis2.local_distortion=True
        expert_bias_model_vis2.hetero_noise=False
        expert_bias_model_vis2.constant_shift_bias =True
        
        
        expert_bias_model_vis2.noise_level = (0.0, 0.2)#(.3, .3)#.1-.7
        break
        expert_bias_model_vis2.hetero_alpha = (0.2, .2)
        expert_bias_model_vis2.hetero_n_centers = (4, 4)
        expert_bias_model_vis2.hetero_decay = (.5,.5)#(0.5, 2)
        
        expert_bias_model_vis2.add_gp_max_std = (.5, 1.)#.5-1
        expert_bias_model_vis2.add_gp_lengthscale = (.5, 3.)# .5-3
        expert_bias_model_vis2.add_gp_scale = (.5, 1)  # [0.2, 2.]
        
        expert_bias_model_vis2.perceived_max_alpha = (.7, .7)

        expert_bias_model_vis2.knowledge_center_noise_radiation_scale= (.3, .3)#[0.1, 0.5]

        expert_bias_model_vis2.distortion_range = (.5,.5)  #(.3,.7)
        expert_bias_model_vis2.distortion_centers = (2,3)  #(1,5)
        expert_bias_model_vis2.distortion_decay = (1.,1.)  #(.5,1.5)        
        
        expert_bias_model_vis2.constant_shift_bias=(-.3, .3)
        
        print("Adjustment completed")

last_bool= True
if expert_bias_model_vis2 is not None:
    for attr, value in expert_bias_model_vis2.__dict__.items():
        if type(value) is bool:
            last_bool = value
            print(f"{attr} = {value}")
        elif last_bool:
            print(f"{attr} = {value}")

experiment_vis2 = hydra.utils.instantiate(cfg.task, expert_model=expert_model, expert_bias_model=expert_bias_model_vis2,
                                model_expert_cfg=cfg.get("expert_model", None), device=device)


In [ ]:
weights = [ 0.3, 0.5, 0.7, 0.9]
Additive_vectors2, Additive_labels2 = collect_weight_sweep(experiment_vis2, weights, n_batches=50, batch_size=10, label_prefix="Additive Biased")
Additive_labels2[0]

In [ ]:
cfg.task.n_query_init = 400
experiment_reweighted = hydra.utils.instantiate(cfg_reweight.task, expert_model=expert_model, expert_bias_model=expert_bias_model,
                                    model_expert_cfg=cfg.get("expert_model", None), device=device)

k_values = [0, 2, 4, 6, 8]

reweight_vec, reweight_labels = collect_kdecoy_sweep(experiment_reweighted, k_values, n_batches=100, batch_size=10, label_prefix="Mixture")


In [ ]:
cfg.task.n_query_init = 400
experiment_Amortized = hydra.utils.instantiate(cfg_Amortized.task, expert_model=expert_model, expert_bias_model=expert_bias_model,
                                    model_expert_cfg=cfg.get("expert_model", None), device=device)

Amortized_vectors, Amortized_labels = collect_weight_sweep(experiment, weights, n_batches=100, batch_size=10, label_prefix="Amortized")
Amortized_labels[0]


## Test tasks


In [ ]:
n_batches=1

In [ ]:
task = AirfoilLiftDragTask(n_context_init=1, n_query_init=300, 
        # AirfoilLiftDragTask= we = good, , we are bad, AirfoilNNFidelityTask=easy, AirfoilStallMultiFidelityTask=better MF and we are good, 
    n_target_data=cfg.task.n_target_data,
    #bias_model=expert_bias_model,      #or comment out
    #use_expert_thetas = True,
    embedding_type="mix",
    n_target_theta=cfg.task.n_target_theta,
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    noise_scale=.1                      
)

airfoil_vec, airfoil_labels = collect_task(task, label="Airfoil Lift", n_batches=n_batches)

task = AirfoilNNFidelityTask(n_context_init=1, n_query_init=300, 
        # AirfoilLiftDragTask= we = good, , we are bad, AirfoilNNFidelityTask=easy, AirfoilStallMultiFidelityTask=better MF and we are good, 
    n_target_data=cfg.task.n_target_data,
    #bias_model=expert_bias_model,      #or comment out
    #use_expert_thetas = True,
    embedding_type="mix",
    n_target_theta=cfg.task.n_target_theta,
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    noise_scale=.1                      
)

airfoil_vec2, airfoil_labels2 = collect_task(task, label="Airfoil Small", n_batches=n_batches)

In [ ]:

pth_path="data/NAS-Bench-201-v1_0-e61699.pth"

NASBench201Task.from_pth(
            pth_path=pth_path,
            n_context_init=1,
            n_query_init=123,
            n_target_data=0,
            n_target_theta=3,
            design_scale=5,
            noise_scale=.1,
            normalize_y=True,
            device=device,
        )

ml_vec, ml_labels = collect_task(task, label="ML hyper-parameters", n_batches=n_batches)



In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "reactor"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
reactor_vec, reactor_labels = collect_task(task, label="reactor", n_batches=n_batches)



In [ ]:
# tree model
sklearn_model = build_sklearn_model(cfg.expert_model.sklearn_model)
expert_generator_vis = partial(
    ModelExpert,
    model = SklearnExpertModel(sklearn_model),
    fit_on = ["expert"],
    n_jobs = -1,
    adaptive = False,
    normalize_outputs_to_max_one = False,
    mode = "mean",
    #kappa    = 1.,
)

bench= "ackley3dn"  # add n for negating
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=1,  #2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    expert_generator=expert_generator_vis,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
ackley2_vec, ackley2_labels = collect_task(task, label="ackley_XGBoost", n_batches=n_batches)

In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "ackley3dn"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 1000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
ackley_vec, ackley_labels = collect_task(task, label="ackley", n_batches=n_batches)



In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "rosenbrock3dn"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
rosenbrock_vec, rosenbrock_labels = collect_task(task, label="rosenbrock", n_batches=n_batches)



In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "rastrigin3dn"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
rastrigin_vec, rastrigin_labels = collect_task(task, label="rastrigin", n_batches=n_batches)



In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "schwefel3dn"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
schwefel_vec, schwefel_labels = collect_task(task, label="schwefel", n_batches=n_batches)



In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "levy3dn"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
levy_vec, levy_labels = collect_task(task, label="levy", n_batches=n_batches)



In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "branin_fb2dn"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
branin_vec, branin_labels = collect_task(task, label="branin", n_batches=n_batches)



In [ ]:
T = 5  # TEMP smoke, real: 30
bench= "sphere3dn"  # add n for negating
expert_bias_model_used = None  #expert_bias_model
task = BenchmarkFeedbackTask(
    benchmark_name= bench,  # ackley3dn, rosenbrock3dn=not great, rastrigin3dn, levy3dn= bad?, sphere3dn = fine, 
    #griewank3dn=often shit?, schwefel3dn=terrible, we = avg, dixon_price3dn=good?,"reactor"=good, forrester_fb1dn
    # forrester1dn, branin2dn, branin_fb2dn = slightly better, gramacy2d
    dim_x=2,
    dim_y=cfg.task.dim_y,
    n_context_init=1,
    n_query_init=300,
    n_target_data=cfg.task.n_target_data,
    n_target_theta=cfg.task.n_target_theta,
    design_scale=5,
    noise_scale=0.1,

    expert_dim=1,  # which dims the expert sees
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000  not activated if something like reactor, which has built in feedback
    #use_expert_thetas = True,
    concat_bias_generated_certainty=False,
    embedding_type="mix",
    normalize_y=True,
    #_init_expert_model_from_batch=experiment.expert_generator
    #expert_bias_model=expert_bias_model_used,      #or comment out
    #expert_generator=expert_generator,
    n_expert_extra_min= 50,
    n_expert_extra_max=50,
    expert_min_centers= 5, expert_max_centers=5,  # constant anyways so choose same nr
    expert_cluster_std_min=.5, expert_cluster_std_max=1.5,
    uniform_fraction=1.,
    device=device,
)
sphere_vec, sphere_labels = collect_task(task, label="sphere", n_batches=n_batches)



## Plot


Features: binned signed and absolute $u - y$, sorted by y and by feedback. `select_features` picks a subset.


In [ ]:
def select_features(
    vectors: np.ndarray,
    mode: str = "both",
    sign: str = "both",
    top_bins: int = None,
    n_bins: int = 20,
    ) -> np.ndarray:
    signed_by_y = vectors[:, : n_bins]
    signed_by_fb = vectors[:, n_bins :2*n_bins]
    abs_by_y = vectors[:, 2*n_bins :3*n_bins]
    abs_by_fb = vectors[:, 3*n_bins : ]

    def top(x):
        return x[:, -top_bins:] if top_bins is not None else x

    if mode == "by_y":
        signed_parts = [top(signed_by_y)]
        abs_parts = [top(abs_by_y)]
    elif mode == "by_feedback":
        signed_parts = [top(signed_by_fb)]
        abs_parts = [top(abs_by_fb)]
    else:
        signed_parts = [top(signed_by_y), top(signed_by_fb)]
        abs_parts = [top(abs_by_y), top(abs_by_fb)]

    if sign == "signed":
        return np.concatenate(signed_parts, axis=1)
    elif sign == "abs":
        return np.concatenate(abs_parts, axis=1)
    else:
        return np.concatenate(signed_parts + abs_parts, axis=1)

In [ ]:
weight_colors = dict(zip(
    [f"Additive w={w:.1f}" for w in weights],
    ["#D6E8F7", "#93C4E8", "#4A90D9", "#1F5FAD", "#0A2D6E"],
))

weight2_colors = dict(zip(
    [f"Additive Biased w={w}" for w in weights],
    ["#FFCCCC", "#FF8888", "#FF4444", "#CC0000", "#7A0000"],

))
kdecoy_colors = dict(zip(
    [f"Mixture k={k}" for k in k_values],
    ["#FFCCCC", "#FF8888", "#FF4444", "#CC0000", "#7A0000"],
))
colors_green = dict(zip(
    [f"Amortized w={w}" for w in weights],
    ["#CCFFCC", "#88FF88", "#44FF44", "#00CC00", "#007A00"],
))

airfoil_colors = {
    "Airfoil Lift": "#A8E6CF",  # light green
    "Airfoil Small": "#1A7A3F",  # dark green
}

benchmark_colors = {
    "reactor": "#E74C3C",  # red
    "sphere": "#9B59B6",  # purple
    "ackley": "#1ABC9C",  # teal
    "ackley_XGBoost":"#1ABC9C",  # teal
    "rastrigin": "#E67E22",  # orange
    "rosenbrock": "#F39C12",  # amber
    "schwefel": "#2ECC71",  # green  (if not used by airfoil)
    "levy": "#E91E63",  # pink
    "branin": "#06D6A0",  # minr
    "ML hyper-parameters": "#FFD166",  # yellow
    #"#F4A261",   # peach, 
}


fig = umap_and_plot(
    select_features(
        np.concatenate([Additive_vectors, 
                        #Additive_vectors2,
                        #reweight_vec, 
                        #Amortized_vectors, 
                        ackley_vec, rosenbrock_vec, rastrigin_vec,schwefel_vec, levy_vec, sphere_vec,
                       #ackley2_vec, 
                        branin_vec, ml_vec,airfoil_vec, airfoil_vec2, reactor_vec,
                       ]),
        mode="by_feedback",  # both, by_y, by_feedback
        sign= "both",  #both, signed, abs
        top_bins=20,
    ),
    Additive_labels + 
    #Additive_labels2+
    #reweight_labels+
    #Amortized_labels+
    ackley_labels+rosenbrock_labels+rastrigin_labels+schwefel_labels +levy_labels+ sphere_labels+
                    #+ackley2_labels
                    branin_labels
                    +ml_labels+airfoil_labels+airfoil_labels2+reactor_labels,
    label_colors = {**weight_colors, **weight2_colors, **kdecoy_colors, **colors_green, **airfoil_colors, **benchmark_colors},
     label_markers = {"Airfoil Lift": "*",
                     "Airfoil Small": "*",
                      "ML hyper-parameters": "*",#"D",
                      "reactor": "*",#"D",
                      "branin": "D",#"D",
                      "sphere": "s",
                      "ackley": "s",
                      "rastrigin": "s",
                      "rosenbrock": "s",
                      "schwefel": "s",
                      "levy": "s",
                      #"ackley_XGBoost": "p",
                      
                     }, 
            n_neighbors = 5,  # default 15, higher = more global
    min_dist = .1,  # default 0.1, higher = less tight clusters
)

plt.savefig("figures/umap.pdf", bbox_inches="tight")


# Continuous BO

Pool vs. continuous candidates (paper continuous BO appendix). In continuous mode each step draws a fresh Sobol set,
refines the editable block with Adam on the acquisition (`utils/free_mode.py`), and keeps a fixed background block.
Feedback is never differentiated, only re-observed (`FEEDBACK_MODE`).


## Multiple benchmarks

For each prior: policy head, UCB and EI (both with πBO), each in pool and continuous mode, plus the GP surrogate in pool mode.
Regret is against the global optimum (`regret_reference="global"`), since continuous methods can beat the pool optimum.


In [ ]:
benchmarks_3d = ["sphere3dn"] or ["ackley3dn", "sphere3dn", "rastrigin3dn", "rosenbrock3dn", "schwefel3dn", "levy3dn", "hartmann3dn",  # TEMP smoke, real: list after "or"
                 "sphere2dn", "ackley2dn", "rastrigin2dn", "rosenbrock2dn", "schwefel2dn", "levy2dn", 
                ]

T_MULTI = 5  # TEMP smoke, real: 30
BATCH_SIZE_MULTI = 4  # TEMP smoke, real: 100
FEEDBACK_MODE = "live"  # "frozen" (observe feedback once) or "live" (re-observe every refine step)
FEEDBACK_ADJUSTOR = partial(pi_bo, beta=2)

# continuous knobs (utils/free_mode.py), None -> evaluate_continuous default
CANDIDATE_MODE = "sobol"  # "pool" (ranked pool + top-k) or "sobol" (two Sobol draws, no ranking pass)
N_CANDIDATES = 100  # sobol mode: size of the editable block (default 100 when None)
N_BACKGROUND = 200  # sobol mode: size of the fixed background block (default 100 when None)
REFINE_OPTIMIZER = "adam"  # "lbfgs" or "adam" for refine_pool's per-step optimizer
REFINE_LR = None#step size in norm [-1, 1] box coords (default 0.1 when None)
ADAM_STEP_MULTIPLIER = 1# adam steps = this x n_refine_steps = 10, for forward-pass budget parity (default 4)
NORMALIZE_BOX = None  # optimize in unit-cube coords instead of raw design_scale coords (default True)
LOG_EI_MODE = "clamp"  # "clamp" (log(EI).clamp_min) or "botorch" (LogExpectedImprovement formula)


@dataclass
class PriorSpec:
    label: str
    model: Any
    n_theta: int
    concat_feedback: bool = False
    experiment_dim_x_delta: int = 0


priors = [
    #PriorSpec("FICBO (Combined)", model, cfg.task.n_target_theta),
    PriorSpec("FICBO (Mixture Prior)", model_reweight, cfg.task.n_target_theta),  # model_reweight, model_Amortized
    #PriorSpec("FICBO (Additive Prior)", model_Amortized, cfg.task.n_target_theta),
    #PriorSpec( "GP-prior ICBO", expert_model_not_using_feedback,cfg_expert_model_not_using_feedback.task.n_target_theta,),
    #PriorSpec("GP-prior ICBO + feedback feature", expert_model_using_feedback, cfg_expert_model_using_feedback.task.n_target_theta, 
    #          concat_feedback=True, experiment_dim_x_delta=cfg.task.dim_feedback, ),
]


def prior_factorial(spec: PriorSpec) -> List[MethodConfig]:
    """Policy, UCB and EI for one prior, each in pool and continuous mode. UCB/EI always use pi_bo."""
    common = dict(
        model=spec.model, cfg=cfg, n_theta=spec.n_theta,
        concat_feedback=spec.concat_feedback,
        experiment_dim_x_delta=spec.experiment_dim_x_delta,
    )
    internal_pool = MethodConfig(f"{spec.label} (pool)", acquisition="aae", **common)
    internal_continuous = internal_pool.vary(
        name=f"{spec.label} (continuous)", continuous=True, feedback_mode=FEEDBACK_MODE,
        candidate_mode=CANDIDATE_MODE, n_candidates=N_CANDIDATES, n_background=N_BACKGROUND,
        refine_optimizer=REFINE_OPTIMIZER, refine_lr=REFINE_LR,
        adam_step_multiplier=ADAM_STEP_MULTIPLIER, normalize_box=NORMALIZE_BOX,
        log_ei_mode=LOG_EI_MODE,
    )

    ucb_pool = MethodConfig(
        f"{spec.label} + UCB (pool)", acquisition="ucb",
        expert_feedback_adjustor=FEEDBACK_ADJUSTOR, **common,
    )
    ucb_continuous = ucb_pool.vary(
        name=f"{spec.label} + UCB (continuous)", continuous=True, feedback_mode=FEEDBACK_MODE,
        candidate_mode=CANDIDATE_MODE, n_candidates=N_CANDIDATES, n_background=N_BACKGROUND,
        refine_optimizer=REFINE_OPTIMIZER, refine_lr=REFINE_LR,
        adam_step_multiplier=ADAM_STEP_MULTIPLIER, normalize_box=NORMALIZE_BOX,
        log_ei_mode=LOG_EI_MODE,
    )

    ei_pool = MethodConfig(
        f"{spec.label} + EI (pool)", acquisition="ei",
        expert_feedback_adjustor=FEEDBACK_ADJUSTOR, **common,
    )
    ei_continuous = ei_pool.vary(
        name=f"{spec.label} + EI (continuous)", continuous=True, feedback_mode=FEEDBACK_MODE,
        candidate_mode=CANDIDATE_MODE, n_candidates=N_CANDIDATES, n_background=N_BACKGROUND,
        refine_optimizer=REFINE_OPTIMIZER, refine_lr=REFINE_LR,
        adam_step_multiplier=ADAM_STEP_MULTIPLIER, normalize_box=NORMALIZE_BOX,
        log_ei_mode=LOG_EI_MODE,
    )

    return [
        internal_pool, internal_continuous,
        ucb_pool, ucb_continuous,
        ei_pool, ei_continuous,
    ]


gp_pool = MethodConfig(
    "GP Surrogate (pool)", None, cfg, acquisition="ei", use_gp=True, n_iter_gp=30,
    expert_feedback_adjustor=FEEDBACK_ADJUSTOR,
)
# botorch SingleTaskGP + fit_gpytorch_mll, pool argmax
gp_pool_botorch = gp_pool.vary(name="GP Surrogate (pool, botorch)", gp_native=True)

multi_method_configs = [mc for spec in priors for mc in prior_factorial(spec)]
multi_method_configs.append(gp_pool)
multi_method_configs.append(gp_pool_botorch)

In [ ]:
# experiment unused by continuous methods, pool methods only need some task object
results_multi, optima_multi = run_benchmark_evaluation(
    benchmarks=benchmarks_3d,
    method_configs=multi_method_configs,
    experiment=experiment,
    evaluate_policy_vs_bo=evaluate_policy_vs_bo,
    evaluate_gp_bo=evaluate_gp_bo,
    BenchmarkFeedbackTask=BenchmarkFeedbackTask,
    cfg=cfg,
    batch_size=BATCH_SIZE_MULTI,
    T=T_MULTI,
    expert_dim=1,
    n_marginal_samples=10,  # TEMP smoke, real: 2000; 2000
    design_scale=5,
    noise_scale=0.1,
    expert_bias_model=expert_bias_model,
    n_query_init=300,
    device=device,
    seed_fn=seed_everything,
    regret_reference="global",
)

In [ ]:
# run_benchmark_evaluation already recomputes regret against a non-negative reference, only check it
results_multi_fixed = results_multi
optima_multi_fixed = optima_multi

for bench in benchmarks_3d:
    for name, metrics in results_multi_fixed[bench].items():
        assert (metrics[3] >= -1e-4).all(), f"negative regret: {bench}/{name}"
        assert (metrics[5] >= -1e-4).all(), f"negative cum_regret: {bench}/{name}"
print("no negative regret anywhere, guarantee holds")


In [ ]:
norm_best_so_far = get_normalized_best_so_far(results_multi, optima_multi_fixed)

# final mean score for each method (avg over benchmarks and runs)
final_scores = {
    m: norm_best_so_far[m].mean(dim=(0, 2))[-1].item()
    for m in norm_best_so_far
}

# final std for each method
final_stds = {
    m: norm_best_so_far[m].mean(dim=0)[-1].std(dim=0).item()
    for m in norm_best_so_far
}

# sort methods by final score (highest first)
method_names_multi = sorted(
    final_scores,
    key=final_scores.get,
    reverse=True,
)

print(f"\nFinal normalized best-so-far, averaged over {len(benchmarks_3d)} benchmarks (higher = better):")
print(f"  {'Method':<60} {'Score':>8} {'Std':>8}")
print("  " + "-"*76)
for rank, name in enumerate(method_names_multi, 1):
    print(f"  {rank:2}. {name:<56} {final_scores[name]:8.4f} {final_stds[name]:8.4f}")

fig, ax = plot_metrics_with_confidence(
    [norm_best_so_far[m].mean(dim=0) for m in method_names_multi],
    method_names_multi,
    title=f"Normalized best-so-far, averaged over {len(benchmarks_3d)} benchmarks (dim_x={cfg.task.dim_x})",
    ylabel="Normalized best so far (0=start, 1=optimum)",
    xlabel="Timestep",
    figsize=(12, 10),
    marker_frequency=max(1, T_MULTI // 10),
    include_legend=False,
    nbins=max(2, T_MULTI / 5),
)
fig.tight_layout()
plt.show()

# legend as its own figure
handles, labels = ax.get_legend_handles_labels()
legend_ncol = 3
legend_rows = -(-len(labels) // legend_ncol)
fig_legend, ax_legend = plt.subplots(figsize=(14, 0.45 * legend_rows))
ax_legend.axis("off")
ax_legend.legend(handles, labels, loc="center", frameon=True, ncol=legend_ncol)
plt.show()

In [ ]:
norm_cum_regret = get_normalized_cum_regret(results_multi, optima_multi_fixed)

# final mean score for each method (lower cum regret is better)
final_scores = {
    m: norm_cum_regret[m].mean(dim=(0, 2))[-1].item()
    for m in norm_cum_regret
}

# final std for each method
final_stds = {
    m: norm_cum_regret[m].mean(dim=0)[-1].std(dim=0).item()
    for m in norm_cum_regret
}

# sort methods by final score (lowest regret first)
method_names_multi = sorted(
    final_scores,
    key=final_scores.get,
)

print(f"\nFinal normalized cumulative regret, averaged over {len(benchmarks_3d)} benchmarks (lower = better):")
print(f"  {'Method':<60} {'Score':>8} {'Std':>8}")
print("  " + "-"*76)
for rank, name in enumerate(method_names_multi, 1):
    print(f"  {rank:2}. {name:<56} {final_scores[name]:8.4f} {final_stds[name]:8.4f}")

fig, ax = plot_metrics_with_confidence(
    [norm_cum_regret[m].mean(dim=0) for m in method_names_multi],
    method_names_multi,
    title=f"Normalized cumulative regret, averaged over {len(benchmarks_3d)} benchmarks (dim_x={cfg.task.dim_x})",
    ylabel="Normalized cumulative regret",
    xlabel="Timestep",
    figsize=(12, 5),
    marker_frequency=max(1, T_MULTI // 10),
    include_legend=False,
    nbins=max(2, T_MULTI / 5),
)
fig.tight_layout()
plt.show()

# legend as its own figure, too many methods
handles, labels = ax.get_legend_handles_labels()
legend_ncol = 3
legend_rows = -(-len(labels) // legend_ncol)
fig_legend, ax_legend = plt.subplots(figsize=(14, 0.75 * legend_rows))
ax_legend.axis("off")
ax_legend.legend(handles, labels, loc="center", frameon=True, ncol=legend_ncol)
plt.show()

In [ ]:
def select_methods(names: List[str], mode: str, ucb: bool = False) -> List[str]:
    """Filter method_names_multi down to one (mode, acquisition) slice across all priors."""
    suffix = f"+ UCB ({mode})" if ucb else f"({mode})"
    return [n for n in names if n.endswith(suffix)]


for mode, ucb in [("pool", False), ("continuous", False), ("pool", True), ("continuous", True)]:
    subset = select_methods(method_names_multi, mode=mode, ucb=ucb)
    if not subset:
        continue

    subset_scores = {m: norm_best_so_far[m].mean(dim=(0, 2))[-1].item() for m in subset}
    subset_ranked = sorted(subset_scores, key=subset_scores.get, reverse=True)
    print(f"\n{'UCB' if ucb else 'internal policy'}, {mode} mode, ranking (higher = better):")
    for rank, name in enumerate(subset_ranked, 1):
        print(f"  {rank:2}. {name:<56} {subset_scores[name]:8.4f}")
    fig, ax = plot_metrics_with_confidence(
        [norm_best_so_far[m].mean(dim=0) for m in subset],
        subset,
        title=f"Priors compared: {'UCB' if ucb else 'internal policy'}, {mode} mode",
        ylabel="Normalized best so far (0=start, 1=optimum)",
        xlabel="Timestep",
        figsize=(10, 4),
        marker_frequency=max(1, T_MULTI // 10),
        legend_loc="lower right",
        nbins=max(2, T_MULTI / 5),
    )
    plt.show()

In [ ]:
#per-benchmark final-ranking table (lower cum. regret = better)
for bench in benchmarks_3d:
    print(f"\n{bench}:")
    finals = {name: metrics[5][-1].mean().item() for name, metrics in results_multi_fixed[bench].items()}
    for rank, (name, val) in enumerate(sorted(finals.items(), key=lambda x: x[1]), start=1):
        print(f"  {rank}. {name:<35} {val:.4f}")